In [ ]:
# @title 📦 SEL 1: Setup & Bootstrap Modular Vision AI Studio (CUDA 12.x + Dependensi + Modul)
# @markdown Sel ini memasang dependensi Python, mengekstrak struktur repositori modular ke `/content/vlm-caption-studio` (atau melakukan `git clone` jika `GITHUB_REPO_URL` diisi), dan mengonfigurasi binary `llama-server` & `llama-mtmd-cli` untuk Tesla T4.

import os, sys, json, shutil, subprocess
from pathlib import Path

PROJECT_DIR = Path("/content/vlm-caption-studio")
GITHUB_REPO_URL = "" # @param {type:"string"}
MOUNT_GOOGLE_DRIVE = False # @param {type:"boolean"}

if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print(f"[!] Info mount Google Drive: {e}")

import torch
if not torch.cuda.is_available():
    raise RuntimeError("GPU CUDA tidak terdeteksi! Pilih menu Runtime > Change runtime type > T4 GPU.")
print(f"[+] GPU Aktif: {torch.cuda.get_device_name(0)} | CUDA: {torch.version.cuda}")

# 1. Siapkan struktur modular di /content/vlm-caption-studio
if GITHUB_REPO_URL.strip():
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    subprocess.run(["git", "clone", GITHUB_REPO_URL.strip(), str(PROJECT_DIR)], check=True)
else:
    MODULAR_FILES = {
  "requirements.txt": "huggingface_hub>=0.25.0\nrequests>=2.31.0\npillow>=10.0.0\nfastapi>=0.110.0\nuvicorn>=0.29.0\npython-multipart>=0.0.9\ngradio>=5.0.0,<6.0.0\ntimm>=1.0.17\nsafetensors>=0.4.3\ntorchvision>=0.17.0\nnvidia-cuda-runtime-cu12\nnvidia-cublas-cu12\n",
  "README.md": "# ⚡ Vision AI Dataset Labeling, PixAI Booru Tagger, Multimodal Chat & Smart File Manager Studio (T4 CUDA Edition)\n\nSistem produksi modular untuk **Auto-Captioning Dataset LoRA/Fine-Tuning (Flux, Qwen-Image 2.1, MOSS, Chroma1-HD, Anima, Krea 2, Wan 2.1)**, **Ultra-Fast Anime Booru Tagging (`pixai-labs/pixai-tagger-v0.9`)**, **Multimodal Vision Chat**, dan **Smart Dataset/VRAM Manager** di atas GPU NVIDIA Tesla T4 (15 GiB VRAM).\n\n## Struktur Modul\n\n```text\nvlm-caption-studio/\n├── requirements.txt                  # Dependensi terverifikasi (Gradio 5, FastAPI, timm, CUDA 12 libs)\n├── bootstrap_vlm.py                  # Smart installer binary llama.cpp CUDA 12.x (llama-server & llama-mtmd-cli)\n├── run_vlm_studio.py                 # Entrypoint utama FastAPI + Gradio Tunnel\n├── config/\n│   ├── __init__.py\n│   └── vlm_catalog.py                # Katalog 7 VLM GGUF + PixAI Tagger v0.9 + Per-Model Routing Adapters + 8 Style Presets\n├── core/\n│   ├── __init__.py\n│   ├── llama_manager.py              # Manajer llama-server (-c 8192, -np 1, KV-cache q8_0, dynamic thinking, VRAM/Disk telemetry)\n│   ├── tagger_manager.py             # PyTorch + timm EVA02-Large runner untuk PixAI Tagger v0.9 (Standalone & Hybrid VLM Assistant)\n│   ├── dataset_manager.py            # Safe ZIP extraction anti-__MACOSX, verifikasi Pillow, Bulk Trigger Edit, Google Drive Sync\n│   └── text_cleaner.py               # Anti-leak <think> cleaner (closed & truncated) + Dual Word/Token estimator\n├── ui/\n│   ├── __init__.py\n│   ├── studio_routes.py              # Non-blocking FastAPI routes (asyncio.to_thread)\n│   └── templates/\n│       └── studio.html               # Single-Page Application (Pagination, Lightbox Split Editor, Persistent Vision Chat)\n└── notebooks/\n    └── VLM_Caption_Studio_Colab.ipynb # Notebook Google Colab (2 Sel: Setup & Launch)\n```\n",
  "bootstrap_vlm.py": "\"\"\"\nbootstrap_vlm.py\nSmart CUDA 12.x binary installer & verifier for llama.cpp (llama-server & llama-mtmd-cli)\nTargeted for Google Colab NVIDIA Tesla T4 (15 GiB VRAM, Compute Capability 7.5).\n\"\"\"\n\nimport os\nimport sys\nimport glob\nimport shutil\nimport tarfile\nimport zipfile\nimport subprocess\nfrom pathlib import Path\nfrom typing import Dict, Optional, Tuple\n\nimport requests\n\nDEFAULT_BIN_ROOT = Path(\"/content/llama-bin\")\n\n\ndef detect_cuda_version() -> str:\n    \"\"\"Detect active CUDA runtime version via torch or nvcc, defaulting to 12.6.\"\"\"\n    try:\n        import torch\n        if torch.cuda.is_available() and torch.version.cuda:\n            return str(torch.version.cuda)\n    except Exception:\n        pass\n    try:\n        res = subprocess.run([\"nvcc\", \"--version\"], capture_output=True, text=True, timeout=5)\n        if res.returncode == 0:\n            import re\n            m = re.search(r\"release\\s+(\\d+\\.\\d+)\", res.stdout)\n            if m:\n                return m.group(1)\n    except Exception:\n        pass\n    return \"12.6\"\n\n\ndef link_nvidia_cuda_libs(bin_dir: str) -> str:\n    \"\"\"\n    Symlink libcudart.so.12, libcublas.so.12, libcublasLt.so.12 from Python dist-packages\n    into the llama.cpp binary directory and update LD_LIBRARY_PATH.\n    \"\"\"\n    nvidia_lib_dirs = glob.glob(\"/usr/local/lib/python*/dist-packages/nvidia/*/lib\")\n    system_cuda_dirs = [\n        d for d in [\"/usr/local/cuda/lib64\", \"/usr/local/cuda-12/lib64\", \"/usr/lib/x86_64-linux-gnu\"]\n        if os.path.isdir(d)\n    ]\n    all_lib_dirs = [bin_dir] + nvidia_lib_dirs + system_cuda_dirs\n\n    for lib_dir in nvidia_lib_dirs:\n        for so_file in glob.glob(os.path.join(lib_dir, \"*.so*\")):\n            dest = os.path.join(bin_dir, os.path.basename(so_file))\n            if not os.path.exists(dest):\n                try:\n                    os.symlink(so_file, dest)\n                except Exception:\n                    pass\n\n    current_ld = os.environ.get(\"LD_LIBRARY_PATH\", \"\")\n    extra_ld = \":\".join(all_lib_dirs)\n    new_ld = f\"{extra_ld}:{current_ld}\" if current_ld else extra_ld\n    os.environ[\"LD_LIBRARY_PATH\"] = new_ld\n    return new_ld\n\n\ndef _find_binaries(bin_root: Path) -> Tuple[Optional[str], Optional[str], Optional[str]]:\n    \"\"\"Locate llama-server, llama-mtmd-cli, and llama-cli inside bin_root.\"\"\"\n    root_str = str(bin_root)\n    servers = sorted(glob.glob(f\"{root_str}/**/llama-server\", recursive=True))\n    mtmds = sorted(glob.glob(f\"{root_str}/**/llama-mtmd-cli\", recursive=True))\n    clis = sorted(glob.glob(f\"{root_str}/**/llama-cli\", recursive=True))\n    return (\n        servers[0] if servers else None,\n        mtmds[0] if mtmds else None,\n        clis[0] if clis else None,\n    )\n\n\ndef _download_and_extract_archive(url: str, dest_dir: Path) -> None:\n    \"\"\"Download a .tar.gz or .zip release asset and extract it safely into dest_dir.\"\"\"\n    dest_dir.mkdir(parents=True, exist_ok=True)\n    fname = url.split(\"/\")[-1].split(\"?\")[0]\n    archive_path = dest_dir / fname\n\n    print(f\"[+] Mengunduh paket binary CUDA: {fname} ...\", flush=True)\n    with requests.get(url, stream=True, timeout=180) as r:\n        r.raise_for_status()\n        with open(archive_path, \"wb\") as f:\n            for chunk in r.iter_content(chunk_size=65536):\n                if chunk:\n                    f.write(chunk)\n\n    if fname.endswith(\".tar.gz\") or fname.endswith(\".tgz\"):\n        with tarfile.open(archive_path, \"r:gz\") as tar:\n            tar.extractall(path=str(dest_dir))\n    elif fname.endswith(\".zip\"):\n        with zipfile.ZipFile(archive_path, \"r\") as zf:\n            zf.extractall(path=str(dest_dir))\n\n    try:\n        archive_path.unlink(missing_ok=True)\n    except Exception:\n        pass\n\n\ndef _select_aidock_cuda12_asset(assets: list, detected_cuda: str) -> Optional[dict]:\n    \"\"\"\n    Strictly select a Linux amd64/x64 CUDA 12.x archive from ai-dock/llama.cpp-cuda.\n    Never selects cuda-13 or arm64/aarch64 builds on Colab T4.\n    \"\"\"\n    candidates = []\n    for asset in assets:\n        name = asset.get(\"name\", \"\").lower()\n        if \"cuda-13\" in name or \"arm64\" in name or \"aarch64\" in name or \"darwin\" in name or \"win\" in name:\n            continue\n        if (name.endswith(\"-amd64.tar.gz\") or name.endswith(\"-x64.tar.gz\") or name.endswith(\".tar.gz\")) and \"cuda-12\" in name:\n            candidates.append(asset)\n\n    if not candidates:\n        return None\n\n    priority_tags = [f\"cuda-{detected_cuda}\", \"cuda-12.6\", \"cuda-12.8\", \"cuda-12.4\", \"cuda-12.2\", \"cuda-12\"]\n    for ptag in priority_tags:\n        for cand in candidates:\n            if ptag in cand[\"name\"].lower():\n                return cand\n    return candidates[0]\n\n\ndef _install_from_ggml_official(dest_dir: Path) -> bool:\n    \"\"\"\n    Fallback installer from official ggml-org/llama.cpp releases.\n    Downloads both llama-bXXXX-bin-ubuntu-cuda-12.x-x64 and cudart-llama-bXXXX-bin-ubuntu-cuda-12.x-x64.\n    \"\"\"\n    resp = requests.get(\"https://api.github.com/repos/ggml-org/llama.cpp/releases/latest\", timeout=25)\n    resp.raise_for_status()\n    assets = resp.json().get(\"assets\", [])\n\n    main_asset = None\n    cudart_asset = None\n    for asset in assets:\n        name = asset.get(\"name\", \"\").lower()\n        if \"cuda-13\" in name or \"arm64\" in name or \"aarch64\" in name:\n            continue\n        if \"ubuntu\" in name and \"cuda-12\" in name and (\"x64\" in name or \"amd64\" in name):\n            if name.startswith(\"cudart-\"):\n                cudart_asset = asset\n            elif name.startswith(\"llama-\"):\n                main_asset = asset\n\n    if not main_asset:\n        return False\n\n    if cudart_asset:\n        _download_and_extract_archive(cudart_asset[\"browser_download_url\"], dest_dir)\n    _download_and_extract_archive(main_asset[\"browser_download_url\"], dest_dir)\n    return True\n\n\ndef verify_binary_execution(bin_path: str) -> Tuple[bool, str]:\n    \"\"\"Run `<binary> --version` to verify shared libraries and executable permissions.\"\"\"\n    if not bin_path or not os.path.exists(bin_path):\n        return False, \"Binary path does not exist\"\n    try:\n        os.chmod(bin_path, 0o755)\n        res = subprocess.run(\n            [bin_path, \"--version\"],\n            capture_output=True,\n            text=True,\n            env=os.environ.copy(),\n            timeout=15,\n        )\n        out = (res.stdout + \"\\n\" + res.stderr).strip()\n        if res.returncode == 0 or \"version\" in out.lower() or \"llama\" in out.lower():\n            return True, out.splitlines()[0] if out else \"OK\"\n        return False, out\n    except Exception as e:\n        return False, str(e)\n\n\ndef bootstrap_llama_cpp(bin_root: Path = DEFAULT_BIN_ROOT, force_reinstall: bool = False) -> Dict[str, str]:\n    \"\"\"\n    Ensure pre-compiled CUDA 12.x llama-server and llama-mtmd-cli are installed, linked, and verified.\n    Returns dictionary with resolved paths and environment details.\n    \"\"\"\n    if force_reinstall and bin_root.exists():\n        shutil.rmtree(bin_root, ignore_errors=True)\n\n    detected_cuda = detect_cuda_version()\n    server_bin, mtmd_bin, cli_bin = _find_binaries(bin_root)\n\n    if not server_bin or (not mtmd_bin and not cli_bin):\n        print(f\"[+] Menyiapkan instalasi llama.cpp CUDA (Runtime CUDA terdeteksi: {detected_cuda})...\", flush=True)\n        installed = False\n        try:\n            resp = requests.get(\"https://api.github.com/repos/ai-dock/llama.cpp-cuda/releases/latest\", timeout=25)\n            resp.raise_for_status()\n            assets = resp.json().get(\"assets\", [])\n            selected = _select_aidock_cuda12_asset(assets, detected_cuda)\n            if selected:\n                _download_and_extract_archive(selected[\"browser_download_url\"], bin_root)\n                installed = True\n        except Exception as e:\n            print(f\"[!] Peringatan unduhan ai-dock ({e}), beralih ke rilis resmi ggml-org/llama.cpp...\", flush=True)\n\n        if not installed:\n            if not _install_from_ggml_official(bin_root):\n                raise RuntimeError(\n                    \"Gagal menemukan paket biner Linux CUDA 12.x baik di ai-dock/llama.cpp-cuda maupun ggml-org/llama.cpp.\"\n                )\n\n        server_bin, mtmd_bin, cli_bin = _find_binaries(bin_root)\n\n    if not server_bin:\n        raise RuntimeError(f\"Tidak menemukan binary 'llama-server' di dalam {bin_root}.\")\n\n    # Fallback if an older archive only has llama-cli or only llama-mtmd-cli\n    effective_mtmd = mtmd_bin or cli_bin\n    effective_cli = cli_bin or mtmd_bin\n\n    bin_dir = os.path.dirname(server_bin)\n    for b in filter(None, [server_bin, effective_mtmd, effective_cli]):\n        try:\n            os.chmod(b, 0o755)\n        except Exception:\n            pass\n\n    ld_path = link_nvidia_cuda_libs(bin_dir)\n\n    ok_srv, ver_srv = verify_binary_execution(server_bin)\n    if not ok_srv:\n        raise RuntimeError(f\"Verifikasi llama-server gagal ({server_bin}): {ver_srv}\")\n\n    if effective_mtmd:\n        verify_binary_execution(effective_mtmd)\n\n    os.environ[\"VLM_LLAMA_SERVER\"] = server_bin\n    os.environ[\"VLM_LLAMA_MTMD_CLI\"] = effective_mtmd or \"\"\n    os.environ[\"VLM_LLAMA_CLI\"] = effective_cli or \"\"\n\n    print(f\"[✓] llama-server siap   : {server_bin} ({ver_srv})\", flush=True)\n    print(f\"[✓] llama-mtmd-cli siap : {effective_mtmd}\", flush=True)\n\n    return {\n        \"llama_server\": server_bin,\n        \"llama_mtmd_cli\": effective_mtmd or \"\",\n        \"llama_cli\": effective_cli or \"\",\n        \"bin_dir\": bin_dir,\n        \"cuda_version\": detected_cuda,\n        \"ld_library_path\": ld_path,\n    }\n\n\nif __name__ == \"__main__\":\n    info = bootstrap_llama_cpp()\n    print(info)\n",
  "run_vlm_studio.py": "\"\"\"\nrun_vlm_studio.py\nMain entrypoint for Vision AI Dataset Labeling, Booru Tagging, Multimodal Chat & Smart File Manager Studio.\nKey Architectural Guarantees:\n1. Bootstraps llama.cpp CUDA 12.x (`llama-server` & `llama-mtmd-cli`) if not already configured.\n2. Pre-registers all `/studio_ui` and `/studio_api/*` FastAPI routes so no initial request hits a 404 race condition.\n3. Launches Gradio tunnel (`gradio.live`) embedding `/studio_ui` full-screen.\n\"\"\"\n\nimport os\nimport sys\nimport time\nimport warnings\nfrom pathlib import Path\n\ntry:\n    sys.stdout.reconfigure(line_buffering=True)\n    sys.stderr.reconfigure(line_buffering=True)\nexcept Exception:\n    pass\nwarnings.filterwarnings(\"ignore\")\n\n# Ensure project root is on sys.path\nPROJECT_ROOT = Path(__file__).resolve().parent\nif str(PROJECT_ROOT) not in sys.path:\n    sys.path.insert(0, str(PROJECT_ROOT))\n\nimport gradio as gr\nfrom bootstrap_vlm import bootstrap_llama_cpp\nfrom config.vlm_catalog import DEFAULT_MODEL_KEY, MODEL_REGISTRY\nfrom core.llama_manager import ensure_llama_server\nfrom core.tagger_manager import ensure_pixai_tagger\nfrom ui.studio_routes import register_studio_routes\n\n\ndef create_gradio_shell() -> gr.Blocks:\n    \"\"\"Create the full-viewport Gradio Blocks container hosting `/studio_ui`.\"\"\"\n    custom_css = (\n        \"body, html, .gradio-container { margin:0 !important; padding:0 !important; \"\n        \"max-width:100vw !important; overflow:hidden !important; background:#090b10 !important; } \"\n        \"footer { display:none !important; }\"\n    )\n    with gr.Blocks(\n        title=\"Vision AI Labeling, PixAI Booru & Chat Studio\",\n        css=custom_css,\n        analytics_enabled=False,\n    ) as demo:\n        gr.HTML(\n            '<iframe src=\"/studio_ui\" '\n            'style=\"position:fixed; inset:0; width:100vw; height:100vh; border:none; '\n            'margin:0; padding:0; z-index:999999; background:#090b10;\"></iframe>'\n        )\n    return demo\n\n\ndef main() -> None:\n    # 1. Ensure llama.cpp CUDA binaries are installed & linked\n    if not os.environ.get(\"VLM_LLAMA_SERVER\") or not os.path.exists(os.environ.get(\"VLM_LLAMA_SERVER\", \"\")):\n        bootstrap_llama_cpp()\n\n    default_model = os.environ.get(\"VLM_DEFAULT_MODEL\", DEFAULT_MODEL_KEY)\n    if default_model not in MODEL_REGISTRY:\n        default_model = DEFAULT_MODEL_KEY\n\n    prewarm = os.environ.get(\"VLM_PREWARM_MODEL\", \"1\") == \"1\"\n    if prewarm:\n        print(f\"[+] Pre-warming model default: {default_model} ...\", flush=True)\n        try:\n            spec = MODEL_REGISTRY[default_model]\n            if spec.get(\"engine\") == \"pixai_tagger\":\n                ensure_pixai_tagger()\n            else:\n                ensure_llama_server(default_model)\n        except Exception as e:\n            print(f\"[!] Peringatan pre-warm model ({e}); model dapat dimuat langsung dari tombol UI.\", flush=True)\n\n    demo = create_gradio_shell()\n    ui_port = int(os.environ.get(\"VLM_UI_PORT\", \"7865\"))\n\n    # Launch Gradio with prevent_thread_lock=True and immediately attach routes to the underlying FastAPI app\n    app, local_url, share_url = demo.launch(\n        server_name=\"0.0.0.0\",\n        server_port=ui_port,\n        share=True,\n        show_error=True,\n        prevent_thread_lock=True,\n    )\n    register_studio_routes(app)\n\n    print(f\"VLM_LOCAL_URL={local_url}\", flush=True)\n    print(f\"VLM_SHARE_URL={share_url}\", flush=True)\n\n    while True:\n        time.sleep(3600)\n\n\nif __name__ == \"__main__\":\n    main()\n",
  "config/__init__.py": "\"\"\"Configuration package for Vision AI Studio.\"\"\"\nfrom .vlm_catalog import (\n    MODEL_REGISTRY,\n    STYLE_PRESETS,\n    DEFAULT_MODEL_KEY,\n    PIXAI_TAGGER_KEY,\n    get_model_spec,\n)\n\n__all__ = [\n    \"MODEL_REGISTRY\",\n    \"STYLE_PRESETS\",\n    \"DEFAULT_MODEL_KEY\",\n    \"PIXAI_TAGGER_KEY\",\n    \"get_model_spec\",\n]\n",
  "config/vlm_catalog.py": "\"\"\"\nconfig/vlm_catalog.py\nVerified catalog of 7 GGUF Vision-Language Models + PixAI Tagger v0.9 (PyTorch + timm EVA02-Large),\ncomplete with per-model prompt routing adapters, sampling defaults, and 8 caption style presets.\n\"\"\"\n\nfrom typing import Any, Dict\n\nDEFAULT_MODEL_KEY = \"Qwen3.5-9B [Uncensored Aggressive - HauhauCS]\"\nPIXAI_TAGGER_KEY = \"PixAI Tagger v0.9 [Fast Anime & Character/IP Booru Tagger]\"\n\nMODEL_REGISTRY: Dict[str, Dict[str, Any]] = {\n    \"Qwen3.5-9B [Uncensored Aggressive - HauhauCS]\": {\n        \"engine\": \"llama_cpp\",\n        \"repo\": \"HauhauCS/Qwen3.5-9B-Uncensored-HauhauCS-Aggressive\",\n        \"file\": \"Qwen3.5-9B-Uncensored-HauhauCS-Aggressive-Q4_K_M.gguf\",\n        \"file_pattern\": \"*Q4_K_M*.gguf\",\n        \"mmproj_repo\": \"HauhauCS/Qwen3.5-9B-Uncensored-HauhauCS-Aggressive\",\n        \"mmproj_file\": \"mmproj-Qwen3.5-9B-Uncensored-HauhauCS-Aggressive-BF16.gguf\",\n        \"mmproj_pattern\": \"*mmproj*.gguf\",\n        \"family\": \"qwen35\",\n        \"server_flags\": [\"--jinja\"],\n        \"caption_sampling\": {\n            \"temperature\": 0.7,\n            \"top_p\": 0.8,\n            \"top_k\": 20,\n            \"min_p\": 0.0,\n            \"presence_penalty\": 1.5,\n            \"repeat_penalty\": 1.0,\n            \"enable_thinking\": False,\n            \"reasoning_budget\": 0,\n        },\n        \"prompt_routing\": \"system_and_user\",\n        \"approx_size_gb\": 6.2,\n        \"desc\": \"Generasi terbaru Qwen 3.5 (9B) Aggressive Uncensored (0/465 refusals). Pemahaman ruang, anatomi & detail visual sangat tinggi.\",\n    },\n    \"Qwen3-VL-8B [Uncensored Abliterated - wangkanai]\": {\n        \"engine\": \"llama_cpp\",\n        \"repo\": \"wangkanai/qwen3-vl-8b-instruct\",\n        \"file\": \"qwen3-vl-8b-instruct-abliterated-q4-k-m.gguf\",\n        \"file_pattern\": \"*q4*k*m*.gguf\",\n        \"mmproj_repo\": \"Qwen/Qwen3-VL-8B-Instruct-GGUF\",\n        \"mmproj_file\": \"mmproj-Qwen3VL-8B-Instruct-F16.gguf\",\n        \"mmproj_pattern\": \"*mmproj*F16*.gguf\",\n        \"family\": \"qwen3vl\",\n        \"server_flags\": [\"--jinja\"],\n        \"caption_sampling\": {\n            \"temperature\": 0.2,\n            \"top_p\": 0.8,\n            \"top_k\": 20,\n            \"repeat_penalty\": 1.05,\n            \"enable_thinking\": False,\n        },\n        \"prompt_routing\": \"system_and_user\",\n        \"approx_size_gb\": 5.8,\n        \"desc\": \"Uncensored/Abliterated favorit komunitas. Sangat patuh batas token, instruksi ketat, & akurat tanpa sensor.\",\n    },\n    \"JoyCaption Beta One [Llama-3.1-8B Uncensored - fancyfeast]\": {\n        \"engine\": \"llama_cpp\",\n        \"repo\": \"concedo/llama-joycaption-beta-one-hf-llava-mmproj-gguf\",\n        \"file\": \"Llama-Joycaption-Beta-One-Hf-Llava-Q4_K.gguf\",\n        \"file_pattern\": \"*Q4_K*.gguf\",\n        \"mmproj_repo\": \"concedo/llama-joycaption-beta-one-hf-llava-mmproj-gguf\",\n        \"mmproj_file\": \"llama-joycaption-beta-one-llava-mmproj-model-f16.gguf\",\n        \"mmproj_pattern\": \"*mmproj*.gguf\",\n        \"family\": \"joycaption\",\n        \"server_flags\": [\"--jinja\"],\n        \"caption_sampling\": {\n            \"temperature\": 0.5,\n            \"top_p\": 0.9,\n            \"top_k\": 40,\n            \"repeat_penalty\": 1.05,\n            \"enable_thinking\": False,\n        },\n        \"prompt_routing\": \"joycaption_mode\",\n        \"joycaption_system_prompt\": \"You are a helpful image captioner.\",\n        \"approx_size_gb\": 5.8,\n        \"desc\": \"Raja spesialis captioning dataset LoRA/Diffusion (Flux, MOSS, Qwen, Chroma1-HD, Krea). 100% Uncensored dari lahir.\",\n    },\n    \"Gemma-3-12B-IT [Uncensored Abliterated - mlabonne]\": {\n        \"engine\": \"llama_cpp\",\n        \"repo\": \"bartowski/mlabonne_gemma-3-12b-it-abliterated-GGUF\",\n        \"file\": \"mlabonne_gemma-3-12b-it-abliterated-Q4_K_M.gguf\",\n        \"file_pattern\": \"*Q4_K_M*.gguf\",\n        \"mmproj_repo\": \"bartowski/mlabonne_gemma-3-12b-it-abliterated-GGUF\",\n        \"mmproj_file\": \"mmproj-mlabonne_gemma-3-12b-it-abliterated-f16.gguf\",\n        \"mmproj_pattern\": \"*mmproj*f16*.gguf\",\n        \"family\": \"gemma3\",\n        \"server_flags\": [\"--jinja\"],\n        \"caption_sampling\": {\n            \"temperature\": 0.3,\n            \"top_p\": 0.95,\n            \"top_k\": 64,\n            \"repeat_penalty\": 1.0,\n            \"enable_thinking\": False,\n        },\n        \"prompt_routing\": \"user_merged\",\n        \"approx_size_gb\": 8.1,\n        \"desc\": \"Model 12B Uncensored dari mlabonne (Bartowski GGUF + MMProj). Gaya bahasa deskriptif paling luwes, natural, dan sinematik.\",\n    },\n    \"Qwen3-VL-8B [Official - Qwen]\": {\n        \"engine\": \"llama_cpp\",\n        \"repo\": \"Qwen/Qwen3-VL-8B-Instruct-GGUF\",\n        \"file\": \"Qwen3VL-8B-Instruct-Q4_K_M.gguf\",\n        \"file_pattern\": \"*Q4_K_M*.gguf\",\n        \"mmproj_repo\": \"Qwen/Qwen3-VL-8B-Instruct-GGUF\",\n        \"mmproj_file\": \"mmproj-Qwen3VL-8B-Instruct-F16.gguf\",\n        \"mmproj_pattern\": \"*mmproj*F16*.gguf\",\n        \"family\": \"qwen3vl\",\n        \"server_flags\": [\"--jinja\"],\n        \"caption_sampling\": {\n            \"temperature\": 0.2,\n            \"top_p\": 0.8,\n            \"top_k\": 20,\n            \"repeat_penalty\": 1.05,\n            \"enable_thinking\": False,\n        },\n        \"prompt_routing\": \"system_and_user\",\n        \"approx_size_gb\": 5.8,\n        \"desc\": \"Versi resmi Qwen3-VL-8B Instruct dari tim Qwen. Sangat akurat untuk OCR, analisis tata letak, dan deskripsi objektif.\",\n    },\n    \"Qwen3.5-9B [Official - Unsloth]\": {\n        \"engine\": \"llama_cpp\",\n        \"repo\": \"unsloth/Qwen3.5-9B-GGUF\",\n        \"file\": \"Qwen3.5-9B-Q4_K_M.gguf\",\n        \"file_pattern\": \"*Q4_K_M*.gguf\",\n        \"mmproj_repo\": \"unsloth/Qwen3.5-9B-GGUF\",\n        \"mmproj_file\": \"mmproj-F16.gguf\",\n        \"mmproj_pattern\": \"*mmproj*F16*.gguf\",\n        \"family\": \"qwen35\",\n        \"server_flags\": [\"--jinja\"],\n        \"caption_sampling\": {\n            \"temperature\": 0.7,\n            \"top_p\": 0.8,\n            \"top_k\": 20,\n            \"min_p\": 0.0,\n            \"presence_penalty\": 1.5,\n            \"repeat_penalty\": 1.0,\n            \"enable_thinking\": False,\n            \"reasoning_budget\": 0,\n        },\n        \"prompt_routing\": \"system_and_user\",\n        \"approx_size_gb\": 6.2,\n        \"desc\": \"Versi resmi Qwen3.5-9B Multimodal dari Unsloth. Mendukung mode Thinking untuk analisis visual mendalam.\",\n    },\n    \"Gemma-3-12B-IT [Official - Unsloth]\": {\n        \"engine\": \"llama_cpp\",\n        \"repo\": \"unsloth/gemma-3-12b-it-GGUF\",\n        \"file\": \"gemma-3-12b-it-Q4_K_M.gguf\",\n        \"file_pattern\": \"*Q4_K_M*.gguf\",\n        \"mmproj_repo\": \"unsloth/gemma-3-12b-it-GGUF\",\n        \"mmproj_file\": \"mmproj-F16.gguf\",\n        \"mmproj_pattern\": \"*mmproj*F16*.gguf\",\n        \"family\": \"gemma3\",\n        \"server_flags\": [\"--jinja\"],\n        \"caption_sampling\": {\n            \"temperature\": 0.3,\n            \"top_p\": 0.95,\n            \"top_k\": 64,\n            \"repeat_penalty\": 1.0,\n            \"enable_thinking\": False,\n        },\n        \"prompt_routing\": \"user_merged\",\n        \"approx_size_gb\": 8.1,\n        \"desc\": \"Versi resmi Google Gemma 3 12B Vision Instruct dari Unsloth.\",\n    },\n    PIXAI_TAGGER_KEY: {\n        \"engine\": \"pixai_tagger\",\n        \"repo\": \"pixai-labs/pixai-tagger-v0.9\",\n        \"fallback_repo\": \"yabaiyobai/pixai-tagger-v0.9-private\",\n        \"encoder_repo\": \"hf_hub:SmilingWolf/wd-eva02-large-tagger-v3\",\n        \"file\": \"model_v0.9.pth\",\n        \"tags_file\": \"tags_v0.9_13k.json\",\n        \"ip_map_file\": \"char_ip_map.json\",\n        \"family\": \"pixai_tagger\",\n        \"server_flags\": [],\n        \"caption_sampling\": {\n            \"general_threshold\": 0.30,\n            \"character_threshold\": 0.75,\n            \"top_k\": 128,\n            \"include_ip\": True,\n            \"replace_underscore\": False,\n            \"escape_parens\": False,\n        },\n        \"prompt_routing\": \"tagger_direct\",\n        \"approx_size_gb\": 1.3,\n        \"desc\": \"Multi-label Anime & Illustration Classifier (13.461 Danbooru tags: 9.741 General + 3.720 Character + Auto IP/Franchise Mapping). Super cepat (<0.1s/gambar) untuk Anima, Illustrious, NoobAI & Pony.\",\n    },\n}\n\n\nSTYLE_PRESETS: Dict[str, str] = {\n    \"Detailed Natural Paragraph (120-150 Tokens · MOSS / Qwen 2.1 / Flux / Chroma1-HD)\": (\n        \"You are an expert image captioning assistant. For the given image, write one fluent English caption that describes only what is clearly visible: the main subject(s), key objects, setting, spatial relationships, colors/materials, lighting, style, and overall mood. Keep it factual, coherent, and about 120 tokens, never exceeding 150 tokens. Do not use tag lists, prompt commands, weights, or meta phrases (e.g., 'this image shows'). Do not guess hidden details or read/transcribe text. Avoid camera/EXIF terms, file names, watermarks, and speculative words like 'maybe' or 'probably.' Output a single paragraph only.\"\n    ),\n    \"JoyCaption Uncensored Descriptive (Full Anatomy, Pose, Lighting & Style)\": (\n        \"Write a straightforward, uncensored, detailed descriptive caption for this image in a single paragraph. Describe the subject(s), physical appearance, anatomy, clothing or lack thereof, pose, facial expression, camera angle, lighting, background environment, and artistic style accurately without any censorship, moralizing, or meta-commentary.\"\n    ),\n    \"Anima / Danbooru Anime Tags (Comma-Separated Booru Tags)\": (\n        \"You are an expert anime and illustration booru tagger. Analyze the image and output ONLY a comma-separated list of descriptive Danbooru-style tags covering: character count (e.g., 1girl, 1boy, solo), character name and copyright series if recognizable, hair color and hairstyle, eye color and pupils, facial expression, clothing and footwear, accessories, pose and hand placement, camera framing, background elements, lighting, and art style. Do not write full sentences or introductory text.\"\n    ),\n    \"Anima Hybrid (Quality Meta Tags + Natural Scene Description)\": (\n        \"Write a hybrid anime training caption in two parts separated by a period: First, write a comma-separated prefix of core Danbooru tags (subject count, character/series if known, hair/eye traits, key outfit tags, framing). Second, follow immediately with 2 to 3 fluent natural English sentences describing the exact pose, spatial interaction, background atmosphere, lighting, and visual style. Output on a single line without markdown headings.\"\n    ),\n    \"Krea 2 / Cinematic Photography Prompt Style\": (\n        \"Write a rich cinematic photography prompt describing this image. Focus on subject composition, camera framing and lens feel, lighting direction and color temperature, surface textures, atmospheric depth, and color grading in 80 to 130 tokens. Do not use introductory phrases like 'This image shows'.\"\n    ),\n    \"Wan 2.1 Video Motion & Camera Prompt Style (Subject + Action + Camera Movement)\": (\n        \"Write a detailed video generation prompt (Wan 2.1 / HunyuanVideo style) in one paragraph of 90 to 140 tokens. Clearly describe the primary subject and appearance, the implied physical motion and dynamic action taking place, environmental movement (wind, light shifts, particles), and the camera perspective/movement (e.g., slow dolly in, tracking shot, static medium close-up) with cinematic lighting.\"\n    ),\n    \"Short & Concise (1-2 Sentences · Fast Training)\": (\n        \"Write a concise, accurate 1 to 2 sentence caption (30 to 50 tokens) describing the main subject, action, setting, and visual style of this image directly without filler words.\"\n    ),\n    \"Custom\": (\n        \"You are an expert image captioning assistant. Describe the main subject(s), attributes, clothing, pose, setting, lighting, and artistic style of this image accurately in a single fluent paragraph without introductory meta-phrases.\"\n    ),\n}\n\n\ndef get_model_spec(model_key: str) -> Dict[str, Any]:\n    \"\"\"Return the model specification dict for model_key, falling back to DEFAULT_MODEL_KEY.\"\"\"\n    if model_key in MODEL_REGISTRY:\n        return MODEL_REGISTRY[model_key]\n    return MODEL_REGISTRY[DEFAULT_MODEL_KEY]\n",
  "core/__init__.py": "\"\"\"Core engine and dataset management package for Vision AI Studio.\"\"\"\nfrom .text_cleaner import (\n    clean_vlm_response,\n    extract_message_text,\n    process_caption_text,\n    estimate_token_count,\n)\n\n__all__ = [\n    \"clean_vlm_response\",\n    \"extract_message_text\",\n    \"process_caption_text\",\n    \"estimate_token_count\",\n]\n",
  "core/text_cleaner.py": "\"\"\"\ncore/text_cleaner.py\nAnti-leak VLM output cleaner:\n1. Handles empty `content` with populated `reasoning_content` fallback.\n2. Strips closed `<think>...</think>` blocks.\n3. Strips unclosed/truncated `<think>...$` blocks when cut off by max_tokens.\n4. Strips ANSI escape sequences, special chat template tokens, and conversational meta-preambles.\n5. Normalizes trigger words, commas, and provides word/token estimation.\n\"\"\"\n\nimport re\nfrom typing import Any, Dict, Optional\n\nMETA_PREAMBLE_PATTERNS = [\n    r\"^(?:here\\s+is|here's)\\s+(?:a|the|an|your)\\s+(?:detailed\\s+|uncensored\\s+|concise\\s+|fluent\\s+)?(?:caption|description|prompt|tag\\s+list)(?:\\s+for\\s+(?:the|this)\\s+image)?\\s*[:\\-–—]\\s*\",\n    r\"^(?:sure|certainly|of\\s+course)[,!\\.]?\\s+(?:here\\s+is|here's)\\s+[^:\\n]{1,60}:\\s*\",\n    r\"^(?:caption|description|prompt)\\s*[:\\-–—]\\s*\",\n]\n\n\ndef _extract_last_descriptive_paragraph(reasoning_text: str) -> str:\n    \"\"\"\n    Fallback extractor when a hybrid thinking model puts its response inside reasoning_content\n    and leaves content empty. Extracts the last substantial non-meta paragraph/sentence.\n    \"\"\"\n    if not reasoning_text:\n        return \"\"\n    cleaned = re.sub(r\"<think>|</think>\", \"\", reasoning_text, flags=re.IGNORECASE).strip()\n    paragraphs = [p.strip() for p in re.split(r\"\\n\\s*\\n\", cleaned) if p.strip()]\n    if not paragraphs:\n        return \"\"\n    candidate = paragraphs[-1]\n    candidate = re.sub(r'^[\"\\']|[\"\\']$', \"\", candidate).strip()\n    return candidate\n\n\ndef extract_message_text(message_obj: Dict[str, Any], allow_reasoning_fallback: bool = True) -> str:\n    \"\"\"\n    Safely extract text from an OpenAI-compatible `/v1/chat/completions` message object.\n    Handles both standard `content` and rare `reasoning_content` placement.\n    \"\"\"\n    if not isinstance(message_obj, dict):\n        return \"\"\n    content = (message_obj.get(\"content\") or \"\").strip()\n    reasoning = (message_obj.get(\"reasoning_content\") or \"\").strip()\n\n    if content:\n        # Check if content only contains an unclosed <think> block and nothing else\n        stripped_think = re.sub(r\"<think>.*?</think>\", \"\", content, flags=re.DOTALL | re.IGNORECASE)\n        stripped_think = re.sub(r\"<think>.*$\", \"\", stripped_think, flags=re.DOTALL | re.IGNORECASE).strip()\n        if stripped_think:\n            return content\n        if allow_reasoning_fallback:\n            inner = re.sub(r\"^.*?<think>\", \"\", content, flags=re.DOTALL | re.IGNORECASE)\n            return _extract_last_descriptive_paragraph(inner)\n\n    if not content and reasoning and allow_reasoning_fallback:\n        return _extract_last_descriptive_paragraph(reasoning)\n\n    return content\n\n\ndef clean_vlm_response(\n    text: str,\n    sys_prompt: str = \"\",\n    usr_prompt: str = \"\",\n    strip_meta_preamble: bool = True,\n    single_line: bool = True,\n) -> str:\n    \"\"\"\n    Clean raw VLM output from llama-server or llama-mtmd-cli:\n    - Removes closed <think>...</think> and unclosed <think>...$\n    - Removes ANSI escape codes and <|...|>, </s>, <end_of_turn> tokens\n    - Strips CLI prompt echo if present\n    - Optionally removes meta preambles (\"Here is a caption for the image:\")\n    \"\"\"\n    if not text:\n        return \"\"\n\n    # 1. Remove closed <think>...</think> blocks\n    text = re.sub(r\"<think>.*?</think>\", \"\", text, flags=re.DOTALL | re.IGNORECASE)\n\n    # 2. Remove unclosed/truncated <think>...$ blocks (when cut off by max_tokens)\n    if re.search(r\"<think>\", text, flags=re.IGNORECASE):\n        before_think = re.sub(r\"<think>.*$\", \"\", text, flags=re.DOTALL | re.IGNORECASE).strip()\n        if before_think:\n            text = before_think\n        else:\n            # If the entire text started with <think> and never closed, extract last paragraph\n            raw_inner = re.sub(r\"^.*?<think>\", \"\", text, flags=re.DOTALL | re.IGNORECASE)\n            text = _extract_last_descriptive_paragraph(raw_inner)\n\n    # 3. Also strip stray </think> if model emitted closing tag without opening tag\n    if \"</think>\" in text.lower():\n        text = re.split(r\"</think>\", text, flags=re.IGNORECASE)[-1]\n\n    # 4. Strip ANSI escape sequences and special tokens\n    text = re.sub(r\"\\x1b\\[[0-9;]*[a-zA-Z]\", \"\", text)\n    text = re.sub(r\"<\\|.*?\\|>\", \"\", text)\n    text = re.sub(r\"</?(?:s|bos|eos|end_of_turn|start_of_turn)>\", \"\", text, flags=re.IGNORECASE)\n\n    # 5. Strip CLI prompt echoes if any\n    if usr_prompt and f\"> {usr_prompt}\" in text:\n        text = text.split(f\"> {usr_prompt}\")[-1]\n    text = re.split(r\"\\[\\s*Prompt:\", text, flags=re.IGNORECASE)[0]\n    text = re.split(r\"\\bllama_perf_\", text, flags=re.IGNORECASE)[0]\n    text = re.split(r\"\\bExiting\\.\\.\\.\", text, flags=re.IGNORECASE)[0]\n\n    text = text.strip()\n\n    # 6. Strip conversational meta preambles for direct captioning\n    if strip_meta_preamble:\n        for pat in META_PREAMBLE_PATTERNS:\n            text = re.sub(pat, \"\", text, flags=re.IGNORECASE).strip()\n\n    if single_line:\n        text = \" \".join(line.strip() for line in text.splitlines() if line.strip()).strip()\n    else:\n        lines = [line.rstrip() for line in text.splitlines()]\n        text = \"\\n\".join(lines).strip()\n\n    return text\n\n\ndef process_caption_text(\n    text: str,\n    trigger: str,\n    pos: str = \"front\",\n    force_lower: bool = False,\n    skip_dup: bool = True,\n    fix_commas: bool = True,\n) -> Optional[str]:\n    \"\"\"\n    Insert a trigger word at the front or back of caption text, optionally normalizing case\n    and cleaning redundant commas. Returns None if skipped due to duplicate detection.\n    \"\"\"\n    text = (text or \"\").strip()\n    trigger = (trigger or \"\").strip()\n\n    if force_lower:\n        text = text.lower()\n        trigger = trigger.lower()\n\n    if not trigger:\n        if fix_commas and text:\n            text = _normalize_commas(text)\n        return text\n\n    if skip_dup and trigger.lower() in text.lower():\n        return None\n\n    if not text:\n        return trigger\n\n    if pos == \"front\":\n        cleaned_body = re.sub(r\"^[\\s,\\.\\-]+\", \"\", text)\n        new_text = f\"{trigger}, {cleaned_body}\"\n    else:\n        has_dot = text.endswith(\".\")\n        cleaned_body = re.sub(r\"[\\s,\\.\\-]+$\", \"\", text)\n        new_text = f\"{cleaned_body}, {trigger}\"\n        if has_dot:\n            new_text += \".\"\n\n    if fix_commas:\n        new_text = _normalize_commas(new_text)\n\n    return new_text.strip()\n\n\ndef _normalize_commas(text: str) -> str:\n    \"\"\"Clean duplicate commas and spacing around commas.\"\"\"\n    text = re.sub(r\",+\", \",\", text)\n    text = re.sub(r\"\\s*,\\s*\", \", \", text)\n    text = re.sub(r\"\\s+\", \" \", text)\n    text = re.sub(r\"^\\s*,\\s*\", \"\", text)\n    text = re.sub(r\"\\s*,\\s*$\", \"\", text)\n    return text.strip()\n\n\ndef estimate_token_count(text: str) -> Dict[str, int]:\n    \"\"\"\n    Calculate word count and estimated subword token count (~1.3x words + punctuation).\n    Useful for Diffusion/LoRA token budget indicators (75 / 150 / 225 tokens).\n    \"\"\"\n    s = (text or \"\").strip()\n    if not s:\n        return {\"words\": 0, \"tokens\": 0}\n    words = len(s.split())\n    comma_count = s.count(\",\")\n    est_tokens = max(words, int(round(words * 1.28 + comma_count * 0.35)))\n    return {\"words\": words, \"tokens\": est_tokens}\n",
  "core/tagger_manager.py": "\"\"\"\ncore/tagger_manager.py\nNative PyTorch + timm implementation for `pixai-labs/pixai-tagger-v0.9`.\nSupports:\n1. Standalone Ultra-Fast Booru Tagging (13,461 tags: 9,741 General + 3,720 Character + Auto IP/Copyright Mapping).\n2. Hybrid Tag-Assisted VLM Captioning (Grounding VLM prompts with character/IP/visual tags or merging tags with NL captions).\n3. Automatic zero-token fallback from gated `pixai-labs/pixai-tagger-v0.9` to ungated mirror `yabaiyobai/pixai-tagger-v0.9-private`.\n\"\"\"\n\nimport os\nimport re\nimport gc\nimport json\nimport subprocess\nimport threading\nfrom pathlib import Path\nfrom typing import Any, Dict, List, Optional, Tuple\n\nfrom PIL import Image\n\ntry:\n    from huggingface_hub import hf_hub_download\nexcept ImportError:\n    hf_hub_download = None\n\nfrom config.vlm_catalog import MODEL_REGISTRY, PIXAI_TAGGER_KEY\n\nTAGGER_CACHE_DIR = Path(os.environ.get(\"VLM_MODELS_DIR\", \"/content/vlm_models\")) / \"pixai_tagger_v09\"\n_TAGGER_LOCK = threading.Lock()\n\nTAGGER_STATE: Dict[str, Any] = {\n    \"status\": \"STANDBY\",  # STANDBY | LOADING | ONLINE | ERROR\n    \"device\": None,\n    \"model\": None,\n    \"transform\": None,\n    \"index_to_tag\": {},\n    \"gen_tag_count\": 9741,\n    \"char_tag_count\": 3720,\n    \"char_ip_map\": {},\n    \"weights_path\": None,\n    \"error\": None,\n}\n\n\ndef _pil_to_rgb_white_bg(image: Image.Image, bg_color: Tuple[int, int, int] = (255, 255, 255)) -> Image.Image:\n    \"\"\"Composite RGBA/P transparency onto a clean white RGB background (exact PixAI/WD14 preprocessing).\"\"\"\n    image.load()\n    if image.mode == \"RGBA\":\n        bg = Image.new(\"RGB\", image.size, bg_color)\n        bg.paste(image, mask=image.split()[3])\n        return bg\n    if image.mode == \"P\" or \"transparency\" in image.info:\n        rgba = image.convert(\"RGBA\")\n        bg = Image.new(\"RGB\", rgba.size, bg_color)\n        bg.paste(rgba, mask=rgba.split()[3])\n        return bg\n    return image.convert(\"RGB\")\n\n\ndef _download_tagger_file(filename: str) -> Path:\n    \"\"\"\n    Download required PixAI Tagger v0.9 file (`model_v0.9.pth`, `tags_v0.9_13k.json`, `char_ip_map.json`).\n    Tries official `pixai-labs/pixai-tagger-v0.9` first, then automatically falls back to\n    the ungated mirror `yabaiyobai/pixai-tagger-v0.9-private` if no HF_TOKEN or gated access is present.\n    \"\"\"\n    TAGGER_CACHE_DIR.mkdir(parents=True, exist_ok=True)\n    local_target = TAGGER_CACHE_DIR / filename\n    if local_target.is_file() and local_target.stat().st_size > 1024:\n        return local_target\n\n    spec = MODEL_REGISTRY[PIXAI_TAGGER_KEY]\n    repos_to_try = [spec[\"repo\"], spec[\"fallback_repo\"]]\n    hf_token = os.environ.get(\"HF_TOKEN\") or os.environ.get(\"HUGGING_FACE_HUB_TOKEN\")\n\n    if hf_hub_download is not None:\n        for repo_id in repos_to_try:\n            try:\n                print(f\"[+] Mengunduh komponen PixAI Tagger '{filename}' dari '{repo_id}'...\", flush=True)\n                dl_path = hf_hub_download(\n                    repo_id=repo_id,\n                    filename=filename,\n                    local_dir=str(TAGGER_CACHE_DIR),\n                    token=hf_token if repo_id == spec[\"repo\"] else None,\n                )\n                return Path(dl_path)\n            except Exception as e:\n                print(f\"[!] Info unduhan '{filename}' dari '{repo_id}' ({e}), mencoba mirror cadangan...\", flush=True)\n\n    # Direct wget fallback from ungated mirror\n    fallback_url = f\"https://huggingface.co/{spec['fallback_repo']}/resolve/main/{filename}\"\n    print(f\"[+] Mengunduh via wget fallback: {fallback_url}\", flush=True)\n    subprocess.run([\"wget\", \"-q\", \"-c\", fallback_url, \"-O\", str(local_target)], check=True)\n    return local_target\n\n\ndef is_tagger_downloaded() -> Tuple[bool, float]:\n    \"\"\"Return whether PixAI Tagger v0.9 files exist on disk and their total size in GB.\"\"\"\n    if not TAGGER_CACHE_DIR.exists():\n        return False, 0.0\n    req = [\"model_v0.9.pth\", \"tags_v0.9_13k.json\", \"char_ip_map.json\"]\n    total_bytes = 0\n    all_exist = True\n    for fname in req:\n        p = TAGGER_CACHE_DIR / fname\n        if p.is_file() and p.stat().st_size > 100:\n            total_bytes += p.stat().st_size\n        else:\n            all_exist = False\n    return all_exist, round(total_bytes / (1024 ** 3), 2)\n\n\ndef ensure_pixai_tagger() -> Dict[str, Any]:\n    \"\"\"\n    Build and load PixAI Tagger v0.9 (`EVA02-Large` + `TaggingHead(1024, 13461)`) into GPU/CPU memory.\n    Thread-safe and cached across calls.\n    \"\"\"\n    with _TAGGER_LOCK:\n        if TAGGER_STATE[\"status\"] == \"ONLINE\" and TAGGER_STATE[\"model\"] is not None:\n            return TAGGER_STATE\n\n        TAGGER_STATE[\"status\"] = \"LOADING\"\n        TAGGER_STATE[\"error\"] = None\n        try:\n            import torch\n            import timm\n            import torchvision.transforms as transforms\n\n            weights_file = _download_tagger_file(\"model_v0.9.pth\")\n            tags_file = _download_tagger_file(\"tags_v0.9_13k.json\")\n            mapping_file = _download_tagger_file(\"char_ip_map.json\")\n\n            with tags_file.open(\"r\", encoding=\"utf-8\") as f:\n                tag_info = json.load(f)\n            tag_map: Dict[str, int] = tag_info[\"tag_map\"]\n            tag_split: Dict[str, int] = tag_info[\"tag_split\"]\n            gen_tag_count = int(tag_split.get(\"gen_tag_count\", 9741))\n            char_tag_count = int(tag_split.get(\"character_tag_count\", 3720))\n            total_classes = gen_tag_count + char_tag_count\n\n            with mapping_file.open(\"r\", encoding=\"utf-8\") as f:\n                char_ip_map = json.load(f)\n\n            class TaggingHead(torch.nn.Module):\n                def __init__(self, input_dim: int, num_classes: int):\n                    super().__init__()\n                    self.head = torch.nn.Sequential(torch.nn.Linear(input_dim, num_classes))\n\n                def forward(self, x):\n                    logits = self.head(x)\n                    return torch.sigmoid(logits)\n\n            spec = MODEL_REGISTRY[PIXAI_TAGGER_KEY]\n            encoder = timm.create_model(spec[\"encoder_repo\"], pretrained=False)\n            encoder.reset_classifier(0)\n            decoder = TaggingHead(1024, total_classes)\n            model = torch.nn.Sequential(encoder, decoder)\n\n            device = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n            try:\n                state = torch.load(str(weights_file), map_location=device, weights_only=True)\n            except TypeError:\n                state = torch.load(str(weights_file), map_location=device)\n            model.load_state_dict(state)\n            model.to(device).eval()\n\n            transform = transforms.Compose([\n                transforms.Resize((448, 448)),\n                transforms.ToTensor(),\n                transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),\n            ])\n\n            TAGGER_STATE[\"model\"] = model\n            TAGGER_STATE[\"device\"] = device\n            TAGGER_STATE[\"transform\"] = transform\n            TAGGER_STATE[\"index_to_tag\"] = {int(v): k for k, v in tag_map.items()}\n            TAGGER_STATE[\"gen_tag_count\"] = gen_tag_count\n            TAGGER_STATE[\"char_tag_count\"] = char_tag_count\n            TAGGER_STATE[\"char_ip_map\"] = char_ip_map\n            TAGGER_STATE[\"weights_path\"] = str(weights_file)\n            TAGGER_STATE[\"status\"] = \"ONLINE\"\n            print(f\"[✓] PixAI Tagger v0.9 aktif di {device.upper()} ({total_classes} tags).\", flush=True)\n            return TAGGER_STATE\n        except Exception as e:\n            TAGGER_STATE[\"status\"] = \"ERROR\"\n            TAGGER_STATE[\"error\"] = str(e)\n            print(f\"[!] Gagal memuat PixAI Tagger v0.9: {e}\", flush=True)\n            raise\n\n\ndef unload_pixai_tagger() -> bool:\n    \"\"\"Unload PixAI Tagger v0.9 from GPU VRAM and run garbage collection.\"\"\"\n    with _TAGGER_LOCK:\n        TAGGER_STATE[\"model\"] = None\n        TAGGER_STATE[\"transform\"] = None\n        TAGGER_STATE[\"status\"] = \"STANDBY\"\n        gc.collect()\n        try:\n            import torch\n            if torch.cuda.is_available():\n                torch.cuda.empty_cache()\n        except Exception:\n            pass\n        print(\"[✓] PixAI Tagger v0.9 berhasil di-unload dari VRAM.\", flush=True)\n        return True\n\n\ndef format_tag_token(tag: str, replace_underscore: bool = False, escape_parens: bool = False) -> str:\n    \"\"\"Format a single Danbooru tag according to user preferences.\"\"\"\n    t = tag.strip()\n    if replace_underscore and not re.match(r\"^[:;><\\^v0-9_]{2,4}$\", t):\n        t = t.replace(\"_\", \" \")\n    if escape_parens:\n        t = t.replace(\"(\", r\"\\(\").replace(\")\", r\"\\)\")\n    return t\n\n\ndef run_pixai_tagger(\n    img_path: str,\n    general_threshold: float = 0.30,\n    character_threshold: float = 0.75,\n    top_k: int = 128,\n    include_ip: bool = True,\n    replace_underscore: bool = False,\n    escape_parens: bool = False,\n    exclude_tags: str = \"\",\n) -> Dict[str, Any]:\n    \"\"\"\n    Run PixAI Tagger v0.9 inference on a single image path.\n    Returns structured dict with `caption`, `general_tags`, `character_tags`, `ip_tags`, and `scores`.\n    \"\"\"\n    import torch\n\n    state = ensure_pixai_tagger()\n    model = state[\"model\"]\n    device = state[\"device\"]\n    transform = state[\"transform\"]\n    idx_to_tag = state[\"index_to_tag\"]\n    gen_count = state[\"gen_tag_count\"]\n    char_ip_map = state[\"char_ip_map\"]\n\n    excluded_set = set()\n    if exclude_tags:\n        for raw_ex in exclude_tags.split(\",\"):\n            norm_ex = raw_ex.strip().lower()\n            if norm_ex:\n                excluded_set.add(norm_ex)\n                excluded_set.add(norm_ex.replace(\" \", \"_\"))\n                excluded_set.add(norm_ex.replace(\"_\", \" \"))\n\n    with Image.open(img_path) as raw_im:\n        rgb_im = _pil_to_rgb_white_bg(raw_im)\n        tensor = transform(rgb_im).unsqueeze(0)\n\n    with torch.inference_mode():\n        if device == \"cuda\":\n            tensor = tensor.pin_memory().to(device, non_blocking=True)\n        else:\n            tensor = tensor.to(device)\n        probs = model(tensor)[0].detach().cpu()\n\n    gen_probs = probs[:gen_count]\n    char_probs = probs[gen_count:]\n\n    gen_mask = gen_probs >= float(general_threshold)\n    char_mask = char_probs >= float(character_threshold)\n\n    gen_indices = gen_mask.nonzero(as_tuple=True)[0].tolist()\n    char_indices = (char_mask.nonzero(as_tuple=True)[0] + gen_count).tolist()\n\n    # Sort both categories by probability descending\n    gen_Pairs = sorted(\n        [(idx_to_tag[i], float(probs[i].item())) for i in gen_indices if i in idx_to_tag],\n        key=lambda x: x[1],\n        reverse=True,\n    )\n    if top_k and int(top_k) > 0:\n        gen_Pairs = gen_Pairs[: int(top_k)]\n\n    char_Pairs = sorted(\n        [(idx_to_tag[i], float(probs[i].item())) for i in char_indices if i in idx_to_tag],\n        key=lambda x: x[1],\n        reverse=True,\n    )\n\n    raw_char_tags = [t for t, _ in char_Pairs if t.lower() not in excluded_set]\n    raw_gen_tags = [t for t, _ in gen_Pairs if t.lower() not in excluded_set]\n\n    raw_ip_tags: List[str] = []\n    if include_ip:\n        seen_ips = set()\n        for c_tag in raw_char_tags:\n            for ip in char_ip_map.get(c_tag, []):\n                if ip.lower() not in excluded_set and ip not in seen_ips:\n                    seen_ips.add(ip)\n                    raw_ip_tags.append(ip)\n\n    formatted_char = [format_tag_token(t, replace_underscore, escape_parens) for t in raw_char_tags]\n    formatted_ip = [format_tag_token(t, replace_underscore, escape_parens) for t in raw_ip_tags]\n    formatted_gen = [format_tag_token(t, replace_underscore, escape_parens) for t in raw_gen_tags]\n\n    # Standard Booru / Anima ordering: Character -> IP/Series -> General features\n    combined: List[str] = []\n    seen_out = set()\n    for t in formatted_char + formatted_ip + formatted_gen:\n        low_t = t.lower()\n        if low_t not in seen_out:\n            seen_out.add(low_t)\n            combined.append(t)\n\n    caption_str = \", \".join(combined)\n    return {\n        \"caption\": caption_str,\n        \"character_tags\": formatted_char,\n        \"ip_tags\": formatted_ip,\n        \"general_tags\": formatted_gen,\n        \"scores\": {\n            \"character\": {t: round(s, 4) for t, s in char_Pairs[:15]},\n            \"general\": {t: round(s, 4) for t, s in gen_Pairs[:30]},\n        },\n    }\n\n\ndef build_hybrid_vlm_context(tag_result: Dict[str, Any]) -> str:\n    \"\"\"\n    Build a concise grounding hint from PixAI Tagger v0.9 results to inject into the VLM prompt\n    so the VLM accurately names the anime character, copyright series, and visual traits.\n    \"\"\"\n    chars = tag_result.get(\"character_tags\", [])\n    ips = tag_result.get(\"ip_tags\", [])\n    gens = tag_result.get(\"general_tags\", [])[:28]\n\n    parts = []\n    if chars:\n        parts.append(f\"Detected Character(s): {', '.join(chars)}\")\n    if ips:\n        parts.append(f\"Source Series/Copyright: {', '.join(ips)}\")\n    if gens:\n        parts.append(f\"Key Visual Attributes: {', '.join(gens)}\")\n\n    if not parts:\n        return \"\"\n    return (\n        \"[Reference Visual Tags from PixAI Tagger v0.9 — Incorporate accurate character names, series, \"\n        \"and key traits naturally into your response without copying as a raw list unless requested]: \"\n        + \" | \".join(parts)\n    )\n",
  "core/llama_manager.py": "\"\"\"\ncore/llama_manager.py\nProduction Process & Memory Manager for llama-server & llama-mtmd-cli on NVIDIA Tesla T4 (15 GiB VRAM).\nFeatures:\n- Thread-safe model loading (`threading.Lock()`)\n- Optimal T4 server flags (`-ngl 99 -c 8192 -np 1 -cb -ub 512 -b 1024 --cache-type-k q8_0 --cache-type-v q8_0 --flash-attn on --jinja`)\n- Per-Model Prompt Routing (`system_and_user`, `joycaption_mode`, `user_merged`)\n- Dynamic Qwen3.5 thinking toggle per API call\n- Hybrid PixAI Tagger v0.9 integration (Grounding Prompt, Prepend Tags, Append Tags)\n- Global Subject / Concept Hint injection\n- Automatic server restart & `llama-mtmd-cli` fallback\n- Full GPU VRAM / System RAM / Disk Cache Telemetry & GGUF Cleaner\n\"\"\"\n\nimport os\nimport io\nimport re\nimport time\nimport glob\nimport shutil\nimport signal\nimport base64\nimport threading\nimport subprocess\nfrom pathlib import Path\nfrom typing import Any, Dict, List, Optional, Tuple\n\nimport requests\nfrom PIL import Image\n\ntry:\n    from huggingface_hub import hf_hub_download, list_repo_files\nexcept ImportError:\n    hf_hub_download = None\n    list_repo_files = None\n\nfrom config.vlm_catalog import (\n    MODEL_REGISTRY,\n    DEFAULT_MODEL_KEY,\n    PIXAI_TAGGER_KEY,\n    get_model_spec,\n)\nfrom core.text_cleaner import clean_vlm_response, extract_message_text, estimate_token_count\nfrom core.tagger_manager import (\n    TAGGER_STATE,\n    TAGGER_CACHE_DIR,\n    is_tagger_downloaded,\n    run_pixai_tagger,\n    build_hybrid_vlm_context,\n    unload_pixai_tagger,\n)\n\nos.environ[\"HF_HUB_DISABLE_XET\"] = \"1\"\nos.environ[\"HF_HUB_ENABLE_HF_TRANSFER\"] = \"0\"\n\nMODELS_CACHE_DIR = Path(os.environ.get(\"VLM_MODELS_DIR\", \"/content/vlm_models\"))\nSERVER_PORT = int(os.environ.get(\"VLM_SERVER_PORT\", \"8085\"))\nSERVER_URL = f\"http://127.0.0.1:{SERVER_PORT}\"\nSERVER_PID_FILE = Path(\"/content/vlm_llama_server.pid\")\nSERVER_LOG_FILE = Path(\"/content/vlm_llama_server.log\")\n\n_SERVER_LOCK = threading.Lock()\n_INFERENCE_LOCK = threading.Lock()\n\nACTIVE_STATE: Dict[str, Any] = {\n    \"status\": \"STANDBY\",  # STANDBY | LOADING | ONLINE | ERROR\n    \"model_key\": None,\n    \"model_path\": None,\n    \"mmproj_path\": None,\n    \"proc\": None,\n    \"last_error\": None,\n}\n\n\ndef get_llama_binaries() -> Tuple[str, str, str]:\n    \"\"\"Return resolved paths for llama-server, llama-mtmd-cli, and llama-cli.\"\"\"\n    srv = os.environ.get(\"VLM_LLAMA_SERVER\", \"/content/llama-bin/cuda-12.8/llama-server\")\n    mtmd = os.environ.get(\"VLM_LLAMA_MTMD_CLI\", \"\")\n    cli = os.environ.get(\"VLM_LLAMA_CLI\", \"/content/llama-bin/cuda-12.8/llama-cli\")\n    if not mtmd:\n        found = glob.glob(\"/content/llama-bin/**/llama-mtmd-cli\", recursive=True)\n        mtmd = found[0] if found else cli\n    return srv, mtmd, cli\n\n\ndef _repo_local_dir(repo_id: str) -> Path:\n    return MODELS_CACHE_DIR / repo_id.replace(\"/\", \"__\")\n\n\ndef resolve_and_download_hf(repo_id: str, preferred_filename: str, fallback_glob: str = \"*\") -> str:\n    \"\"\"\n    Download GGUF / MMProj file into MODELS_CACHE_DIR/<repo_id> so disk cache management\n    is predictable, inspectable, and cleanable from Tab 3. Includes smart glob matching & wget fallback.\n    \"\"\"\n    local_dir = _repo_local_dir(repo_id)\n    local_dir.mkdir(parents=True, exist_ok=True)\n\n    direct_candidate = local_dir / Path(preferred_filename).name\n    if direct_candidate.is_file() and direct_candidate.stat().st_size > 1024 * 1024:\n        return str(direct_candidate)\n\n    target_filename = preferred_filename\n    if list_repo_files is not None:\n        try:\n            files = list_repo_files(repo_id)\n            if preferred_filename not in files:\n                import fnmatch\n                matches = [f for f in files if fnmatch.fnmatch(f.lower(), fallback_glob.lower())]\n                if matches:\n                    target_filename = matches[0]\n        except Exception:\n            pass\n\n    candidate_after_resolve = local_dir / Path(target_filename).name\n    if candidate_after_resolve.is_file() and candidate_after_resolve.stat().st_size > 1024 * 1024:\n        return str(candidate_after_resolve)\n\n    print(f\"[+] Memastikan file '{target_filename}' dari '{repo_id}'...\", flush=True)\n    if hf_hub_download is not None:\n        try:\n            dl = hf_hub_download(\n                repo_id=repo_id,\n                filename=target_filename,\n                local_dir=str(local_dir),\n            )\n            return str(dl)\n        except Exception as e:\n            print(f\"[!] Fallback wget untuk {target_filename}: {e}\", flush=True)\n\n    url = f\"https://huggingface.co/{repo_id}/resolve/main/{target_filename}\"\n    out_path = local_dir / Path(target_filename).name\n    subprocess.run([\"wget\", \"-q\", \"-c\", url, \"-O\", str(out_path)], check=True)\n    return str(out_path)\n\n\ndef is_gguf_model_downloaded(model_key: str) -> Tuple[bool, float]:\n    \"\"\"Check whether both main GGUF and MMProj for a model exist on disk and return size in GB.\"\"\"\n    spec = get_model_spec(model_key)\n    if spec.get(\"engine\") == \"pixai_tagger\":\n        return is_tagger_downloaded()\n\n    main_dir = _repo_local_dir(spec[\"repo\"])\n    mmproj_dir = _repo_local_dir(spec[\"mmproj_repo\"])\n\n    main_file = main_dir / Path(spec[\"file\"]).name\n    mmproj_file = mmproj_dir / Path(spec[\"mmproj_file\"]).name\n\n    hf_cache_root = Path.home() / \".cache\" / \"huggingface\" / \"hub\"\n\n    total_bytes = 0\n    main_ok = False\n    mmproj_ok = False\n\n    if main_file.is_file() and main_file.stat().st_size > 1024 * 1024:\n        main_ok = True\n        total_bytes += main_file.stat().st_size\n    elif hf_cache_root.exists():\n        matches = list(hf_cache_root.rglob(Path(spec[\"file\"]).name))\n        if matches and matches[0].stat().st_size > 1024 * 1024:\n            main_ok = True\n            total_bytes += matches[0].stat().st_size\n\n    if mmproj_file.is_file() and mmproj_file.stat().st_size > 1024 * 1024:\n        mmproj_ok = True\n        total_bytes += mmproj_file.stat().st_size\n    elif hf_cache_root.exists():\n        matches = list(hf_cache_root.rglob(Path(spec[\"mmproj_file\"]).name))\n        if matches and matches[0].stat().st_size > 1024 * 1024:\n            mmproj_ok = True\n            total_bytes += matches[0].stat().st_size\n\n    return (main_ok and mmproj_ok), round(total_bytes / (1024 ** 3), 2)\n\n\ndef stop_llama_server() -> None:\n    \"\"\"Terminate active llama-server process and free GPU VRAM.\"\"\"\n    proc = ACTIVE_STATE.get(\"proc\")\n    if proc is not None and proc.poll() is None:\n        try:\n            proc.terminate()\n            proc.wait(timeout=5)\n        except Exception:\n            try:\n                proc.kill()\n            except Exception:\n                pass\n    ACTIVE_STATE[\"proc\"] = None\n    ACTIVE_STATE[\"status\"] = \"STANDBY\"\n    ACTIVE_STATE[\"model_key\"] = None\n\n    if SERVER_PID_FILE.exists():\n        try:\n            pid = int(SERVER_PID_FILE.read_text().strip())\n            os.kill(pid, signal.SIGTERM)\n            time.sleep(0.8)\n        except Exception:\n            pass\n        try:\n            SERVER_PID_FILE.unlink(missing_ok=True)\n        except Exception:\n            pass\n\n\ndef unload_vlm_server(include_tagger: bool = False) -> Dict[str, Any]:\n    \"\"\"Stop llama-server (and optionally PixAI Tagger) to release 100% of T4 VRAM.\"\"\"\n    with _SERVER_LOCK:\n        stop_llama_server()\n        if include_tagger:\n            unload_pixai_tagger()\n    return {\"ok\": True, \"status\": \"STANDBY\", \"msg\": \"Model berhasil di-unload dari VRAM GPU.\"}\n\n\ndef is_server_healthy() -> bool:\n    \"\"\"Ping llama-server `/health` endpoint.\"\"\"\n    proc = ACTIVE_STATE.get(\"proc\")\n    if proc is None or proc.poll() is not None:\n        return False\n    try:\n        r = requests.get(f\"{SERVER_URL}/health\", timeout=2)\n        return r.status_code == 200\n    except Exception:\n        return False\n\n\ndef ensure_llama_server(model_key: str) -> Tuple[str, str]:\n    \"\"\"\n    Thread-safe manager that downloads the requested GGUF + MMProj if needed and starts `llama-server`\n    with Tesla T4 optimal parameters (`-c 8192 -np 1 --cache-type-k q8_0 --cache-type-v q8_0 --flash-attn on`).\n    \"\"\"\n    if model_key not in MODEL_REGISTRY:\n        model_key = DEFAULT_MODEL_KEY\n\n    spec = get_model_spec(model_key)\n    if spec.get(\"engine\") == \"pixai_tagger\":\n        ensure_pixai_tagger()\n        return str(TAGGER_STATE.get(\"weights_path\") or \"\"), \"\"\n\n    with _SERVER_LOCK:\n        if ACTIVE_STATE.get(\"model_key\") == model_key and is_server_healthy():\n            ACTIVE_STATE[\"status\"] = \"ONLINE\"\n            return ACTIVE_STATE[\"model_path\"], ACTIVE_STATE[\"mmproj_path\"]\n\n        ACTIVE_STATE[\"status\"] = \"LOADING\"\n        ACTIVE_STATE[\"last_error\"] = None\n\n        try:\n            model_path = resolve_and_download_hf(spec[\"repo\"], spec[\"file\"], spec[\"file_pattern\"])\n            mmproj_path = resolve_and_download_hf(spec[\"mmproj_repo\"], spec[\"mmproj_file\"], spec[\"mmproj_pattern\"])\n\n            stop_llama_server()\n            ACTIVE_STATE[\"status\"] = \"LOADING\"\n\n            server_bin, _, _ = get_llama_binaries()\n            cmd = [\n                server_bin,\n                \"-m\", model_path,\n                \"--mmproj\", mmproj_path,\n                \"-ngl\", \"99\",\n                \"-c\", \"8192\",\n                \"-np\", \"1\",\n                \"-cb\",\n                \"-ub\", \"512\",\n                \"-b\", \"1024\",\n                \"-t\", \"2\",\n                \"--cache-type-k\", \"q8_0\",\n                \"--cache-type-v\", \"q8_0\",\n                \"--flash-attn\", \"on\",\n                \"--host\", \"127.0.0.1\",\n                \"--port\", str(SERVER_PORT),\n            ]\n            for extra_flag in spec.get(\"server_flags\", [\"--jinja\"]):\n                if extra_flag not in cmd:\n                    cmd.append(extra_flag)\n\n            log_f = open(SERVER_LOG_FILE, \"w\", encoding=\"utf-8\", errors=\"replace\")\n            proc = subprocess.Popen(\n                cmd,\n                stdout=log_f,\n                stderr=subprocess.STDOUT,\n                text=True,\n                env=os.environ.copy(),\n            )\n            SERVER_PID_FILE.write_text(str(proc.pid))\n\n            for _ in range(150):\n                if proc.poll() is not None:\n                    break\n                try:\n                    r = requests.get(f\"{SERVER_URL}/health\", timeout=2)\n                    if r.status_code == 200:\n                        ACTIVE_STATE[\"model_key\"] = model_key\n                        ACTIVE_STATE[\"model_path\"] = model_path\n                        ACTIVE_STATE[\"mmproj_path\"] = mmproj_path\n                        ACTIVE_STATE[\"proc\"] = proc\n                        ACTIVE_STATE[\"status\"] = \"ONLINE\"\n                        print(f\"[✓] llama-server siap dengan model: {model_key}\", flush=True)\n                        return model_path, mmproj_path\n                except Exception:\n                    pass\n                time.sleep(1)\n\n            tail_log = \"\"\n            if SERVER_LOG_FILE.exists():\n                tail_log = \"\\n\".join(SERVER_LOG_FILE.read_text(errors=\"ignore\").splitlines()[-25:])\n            ACTIVE_STATE[\"status\"] = \"ERROR\"\n            ACTIVE_STATE[\"last_error\"] = f\"llama-server gagal mencapai status healthy.\\n{tail_log}\"\n            ACTIVE_STATE[\"model_key\"] = model_key\n            ACTIVE_STATE[\"model_path\"] = model_path\n            ACTIVE_STATE[\"mmproj_path\"] = mmproj_path\n            return model_path, mmproj_path\n        except Exception as e:\n            ACTIVE_STATE[\"status\"] = \"ERROR\"\n            ACTIVE_STATE[\"last_error\"] = str(e)\n            raise\n\n\ndef image_to_data_uri(img_path: str, max_side: int = 1280) -> str:\n    \"\"\"Resize image in memory (without altering disk file) and encode as JPEG base64 Data URI.\"\"\"\n    with Image.open(img_path) as im:\n        if im.mode in (\"RGBA\", \"P\"):\n            bg = Image.new(\"RGB\", im.size, (255, 255, 255))\n            rgba = im.convert(\"RGBA\")\n            bg.paste(rgba, mask=rgba.split()[3])\n            im = bg\n        else:\n            im = im.convert(\"RGB\")\n        w, h = im.size\n        if max(w, h) > max_side:\n            scale = max_side / float(max(w, h))\n            im = im.resize((int(w * scale), int(h * scale)), Image.Resampling.LANCZOS)\n        buf = io.BytesIO()\n        im.save(buf, format=\"JPEG\", quality=92)\n        b64 = base64.b64encode(buf.getvalue()).decode(\"utf-8\")\n        return f\"data:image/jpeg;base64,{b64}\"\n\n\ndef build_routed_messages(\n    model_key: str,\n    sys_prompt: str,\n    usr_prompt: str,\n    image_data_uri: Optional[str] = None,\n    subject_hint: str = \"\",\n    hybrid_tag_context: str = \"\",\n) -> List[Dict[str, Any]]:\n    \"\"\"\n    Build OpenAI-compatible multimodal `messages` list tailored to each VLM's chat template:\n    - `joycaption_mode`: System locked to `\"You are a helpful image captioner.\"`, style instructions placed in user prompt.\n    - `user_merged` (Gemma-3): Merges system + user instructions into a single `user` role message for 100% adherence.\n    - `system_and_user` (Qwen3-VL / Qwen3.5): Keeps distinct `system` and `user` roles.\n    \"\"\"\n    spec = get_model_spec(model_key)\n    routing = spec.get(\"prompt_routing\", \"system_and_user\")\n\n    clean_sys = (sys_prompt or \"\").strip()\n    clean_usr = (usr_prompt or \"Describe this image in detail.\").strip()\n\n    extra_context_blocks = []\n    if subject_hint and subject_hint.strip():\n        extra_context_blocks.append(\n            f\"[Global Subject / Concept Context — Must be respected in the caption: {subject_hint.strip()}]\"\n        )\n    if hybrid_tag_context and hybrid_tag_context.strip():\n        extra_context_blocks.append(hybrid_tag_context.strip())\n\n    extra_block_str = \"\\n\".join(extra_context_blocks)\n\n    if routing == \"joycaption_mode\":\n        locked_sys = spec.get(\"joycaption_system_prompt\", \"You are a helpful image captioner.\")\n        combined_usr = clean_sys if clean_sys else clean_usr\n        if clean_sys and clean_usr and clean_usr.lower() != \"describe this image in detail.\":\n            combined_usr = f\"{clean_sys}\\n\\n{clean_usr}\"\n        if extra_block_str:\n            combined_usr = f\"{combined_usr}\\n\\n{extra_block_str}\"\n\n        u_content: List[Dict[str, Any]] = []\n        if image_data_uri:\n            u_content.append({\"type\": \"image_url\", \"image_url\": {\"url\": image_data_uri}})\n        u_content.append({\"type\": \"text\", \"text\": combined_usr})\n        return [\n            {\"role\": \"system\", \"content\": locked_sys},\n            {\"role\": \"user\", \"content\": u_content},\n        ]\n\n    if routing == \"user_merged\":\n        merged_text = f\"{clean_sys}\\n\\n{clean_usr}\".strip() if clean_sys else clean_usr\n        if extra_block_str:\n            merged_text = f\"{merged_text}\\n\\n{extra_block_str}\"\n        u_content = []\n        if image_data_uri:\n            u_content.append({\"type\": \"image_url\", \"image_url\": {\"url\": image_data_uri}})\n        u_content.append({\"type\": \"text\", \"text\": merged_text})\n        return [{\"role\": \"user\", \"content\": u_content}]\n\n    # Default: system_and_user (Qwen3-VL / Qwen3.5)\n    final_usr = f\"{clean_usr}\\n\\n{extra_block_str}\".strip() if extra_block_str else clean_usr\n    u_content = []\n    if image_data_uri:\n        u_content.append({\"type\": \"image_url\", \"image_url\": {\"url\": image_data_uri}})\n    u_content.append({\"type\": \"text\", \"text\": final_usr})\n\n    messages: List[Dict[str, Any]] = []\n    if clean_sys:\n        messages.append({\"role\": \"system\", \"content\": clean_sys})\n    messages.append({\"role\": \"user\", \"content\": u_content})\n    return messages\n\n\ndef _run_mtmd_cli_fallback(\n    img_path: str,\n    model_path: str,\n    mmproj_path: str,\n    sys_prompt: str,\n    usr_prompt: str,\n    max_tokens: int,\n    temperature: float,\n) -> str:\n    \"\"\"\n    Fallback execution using `llama-mtmd-cli` with combined `-p` prompt (compatible with all libmtmd releases).\n    \"\"\"\n    _, mtmd_bin, cli_bin = get_llama_binaries()\n    runner_bin = mtmd_bin or cli_bin\n    combined_prompt = f\"{sys_prompt.strip()}\\n\\n{usr_prompt.strip()}\".strip()\n\n    cmd = [\n        runner_bin,\n        \"-m\", str(model_path),\n        \"--mmproj\", str(mmproj_path),\n        \"-ngl\", \"99\",\n        \"-c\", \"4096\",\n        \"-t\", \"2\",\n        \"--flash-attn\", \"on\",\n        \"--image\", str(img_path),\n        \"-p\", combined_prompt,\n        \"-n\", str(int(max_tokens)),\n        \"--temp\", str(float(temperature)),\n    ]\n    res = subprocess.run(\n        cmd,\n        stdin=subprocess.DEVNULL,\n        capture_output=True,\n        text=True,\n        encoding=\"utf-8\",\n        errors=\"replace\",\n        timeout=240,\n    )\n    return clean_vlm_response(res.stdout, sys_prompt, usr_prompt)\n\n\ndef run_vlm_caption_single(\n    img_path: str,\n    model_key: str,\n    sys_prompt: str,\n    usr_prompt: str = \"Describe this image in detail.\",\n    max_tokens: int = 220,\n    temperature: Optional[float] = None,\n    subject_hint: str = \"\",\n    hybrid_tagger_enabled: bool = False,\n    hybrid_mode: str = \"grounding\",  # grounding | prepend | append\n    general_threshold: float = 0.30,\n    character_threshold: float = 0.75,\n    replace_underscore: bool = False,\n    escape_parens: bool = False,\n    exclude_tags: str = \"\",\n) -> Dict[str, Any]:\n    \"\"\"\n    Execute single-image captioning or tagging:\n    - If `model_key == PIXAI_TAGGER_KEY`, runs standalone ultra-fast PixAI Tagger v0.9.\n    - Otherwise runs VLM via `llama-server` (with optional Hybrid PixAI Tagger v0.9 assistance).\n    \"\"\"\n    spec = get_model_spec(model_key)\n\n    # 1. Standalone PixAI Tagger v0.9 Mode\n    if spec.get(\"engine\") == \"pixai_tagger\":\n        tag_res = run_pixai_tagger(\n            img_path=img_path,\n            general_threshold=general_threshold,\n            character_threshold=character_threshold,\n            include_ip=True,\n            replace_underscore=replace_underscore,\n            escape_parens=escape_parens,\n            exclude_tags=exclude_tags,\n        )\n        cap = tag_res[\"caption\"]\n        stats = estimate_token_count(cap)\n        return {\n            \"ok\": bool(cap),\n            \"caption\": cap,\n            \"words\": stats[\"words\"],\n            \"tokens\": stats[\"tokens\"],\n            \"tagger_meta\": tag_res,\n        }\n\n    # 2. Optional Hybrid PixAI Tagger Pre-Pass\n    tag_res: Optional[Dict[str, Any]] = None\n    hybrid_context_str = \"\"\n    if hybrid_tagger_enabled:\n        try:\n            tag_res = run_pixai_tagger(\n                img_path=img_path,\n                general_threshold=general_threshold,\n                character_threshold=character_threshold,\n                include_ip=True,\n                replace_underscore=replace_underscore,\n                escape_parens=escape_parens,\n                exclude_tags=exclude_tags,\n            )\n            if hybrid_mode in (\"grounding\", \"both\"):\n                hybrid_context_str = build_hybrid_vlm_context(tag_res)\n        except Exception as e:\n            print(f\"[!] Peringatan Hybrid PixAI Tagger ({e}), melanjutkan dengan VLM murni...\", flush=True)\n\n    # 3. Ensure llama-server is running with requested VLM\n    model_path, mmproj_path = ensure_llama_server(model_key)\n    sampling = dict(spec.get(\"caption_sampling\", {}))\n    eff_temp = float(temperature) if temperature is not None else float(sampling.get(\"temperature\", 0.3))\n\n    data_uri = image_to_data_uri(img_path)\n    messages = build_routed_messages(\n        model_key=model_key,\n        sys_prompt=sys_prompt,\n        usr_prompt=usr_prompt,\n        image_data_uri=data_uri,\n        subject_hint=subject_hint,\n        hybrid_tag_context=hybrid_context_str,\n    )\n\n    payload: Dict[str, Any] = {\n        \"messages\": messages,\n        \"max_tokens\": int(max_tokens),\n        \"temperature\": eff_temp,\n        \"top_p\": float(sampling.get(\"top_p\", 0.9)),\n        \"top_k\": int(sampling.get(\"top_k\", 40)),\n        \"repeat_penalty\": float(sampling.get(\"repeat_penalty\", 1.05)),\n        \"chat_template_kwargs\": {\"enable_thinking\": False},\n        \"reasoning_format\": \"none\",\n    }\n    if \"min_p\" in sampling:\n        payload[\"min_p\"] = float(sampling[\"min_p\"])\n    if \"presence_penalty\" in sampling:\n        payload[\"presence_penalty\"] = float(sampling[\"presence_penalty\"])\n    if \"reasoning_budget\" in sampling:\n        payload[\"reasoning_budget\"] = int(sampling[\"reasoning_budget\"])\n\n    cleaned_caption = \"\"\n    with _INFERENCE_LOCK:\n        for attempt in range(2):\n            try:\n                r = requests.post(f\"{SERVER_URL}/v1/chat/completions\", json=payload, timeout=180)\n                if r.ok:\n                    resp_json = r.json()\n                    msg_obj = resp_json[\"choices\"][0][\"message\"]\n                    raw_text = extract_message_text(msg_obj, allow_reasoning_fallback=True)\n                    cleaned_caption = clean_vlm_response(raw_text, sys_prompt, usr_prompt)\n                    if cleaned_caption:\n                        break\n            except Exception as e:\n                print(f\"[!] Request llama-server percobaan {attempt + 1} gagal ({e})\", flush=True)\n                if attempt == 0:\n                    stop_llama_server()\n                    ensure_llama_server(model_key)\n\n        if not cleaned_caption:\n            print(\"[!] Menjalankan fallback llama-mtmd-cli...\", flush=True)\n            stop_llama_server()\n            cleaned_caption = _run_mtmd_cli_fallback(\n                img_path, model_path, mmproj_path, sys_prompt, usr_prompt, int(max_tokens), eff_temp\n            )\n\n    # 4. Merge Hybrid Tags if prepend/append/both was selected\n    if tag_res and tag_res.get(\"caption\") and cleaned_caption:\n        tag_str = tag_res[\"caption\"].strip()\n        if hybrid_mode in (\"prepend\", \"both\"):\n            cleaned_caption = f\"{tag_str}. {cleaned_caption}\"\n        elif hybrid_mode == \"append\":\n            cleaned_caption = f\"{cleaned_caption.rstrip('. ')}. Tags: {tag_str}\"\n\n    stats = estimate_token_count(cleaned_caption)\n    return {\n        \"ok\": bool(cleaned_caption),\n        \"caption\": cleaned_caption,\n        \"words\": stats[\"words\"],\n        \"tokens\": stats[\"tokens\"],\n        \"tagger_meta\": tag_res,\n    }\n\n\ndef run_multimodal_chat(\n    model_key: str,\n    sys_prompt: str,\n    history: List[Dict[str, Any]],\n    user_message: str,\n    image_data_uri: Optional[str] = None,\n    temperature: float = 0.7,\n    max_tokens: int = 768,\n    enable_thinking: bool = False,\n) -> Dict[str, Any]:\n    \"\"\"\n    Execute multi-turn multimodal chat.\n    Preserves up to the 2 most recent images in conversation history so follow-up questions\n    continue to see the image without overflowing the 8192 context window on Tesla T4.\n    \"\"\"\n    spec = get_model_spec(model_key)\n    if spec.get(\"engine\") == \"pixai_tagger\":\n        model_key = DEFAULT_MODEL_KEY\n        spec = get_model_spec(model_key)\n\n    ensure_llama_server(model_key)\n    routing = spec.get(\"prompt_routing\", \"system_and_user\")\n\n    turns_with_img = [idx for idx, h in enumerate(history) if h.get(\"image_data_uri\")]\n    keep_img_indices = set(turns_with_img[-1:] if image_data_uri else turns_with_img[-2:])\n\n    messages: List[Dict[str, Any]] = []\n    if routing != \"user_merged\" and sys_prompt.strip():\n        messages.append({\"role\": \"system\", \"content\": sys_prompt.strip()})\n\n    first_user_merged = False\n    for idx, h in enumerate(history):\n        role = h.get(\"role\", \"user\")\n        text_val = (h.get(\"content\") or \"\").strip()\n        h_img = h.get(\"image_data_uri\") if idx in keep_img_indices else None\n\n        if role == \"user\":\n            if routing == \"user_merged\" and not first_user_merged and sys_prompt.strip():\n                text_val = f\"{sys_prompt.strip()}\\n\\n{text_val}\".strip()\n                first_user_merged = True\n            if h_img:\n                messages.append({\n                    \"role\": \"user\",\n                    \"content\": [\n                        {\"type\": \"image_url\", \"image_url\": {\"url\": h_img}},\n                        {\"type\": \"text\", \"text\": text_val or \"Describe this image.\"},\n                    ],\n                })\n            else:\n                messages.append({\"role\": \"user\", \"content\": text_val})\n        else:\n            messages.append({\"role\": \"assistant\", \"content\": text_val})\n\n    cur_text = (user_message or \"Describe this image in detail.\").strip()\n    if routing == \"user_merged\" and not first_user_merged and sys_prompt.strip():\n        cur_text = f\"{sys_prompt.strip()}\\n\\n{cur_text}\".strip()\n\n    u_content: List[Dict[str, Any]] = []\n    if image_data_uri:\n        u_content.append({\"type\": \"image_url\", \"image_url\": {\"url\": image_data_uri}})\n    u_content.append({\"type\": \"text\", \"text\": cur_text})\n    messages.append({\"role\": \"user\", \"content\": u_content})\n\n    use_thinking = bool(enable_thinking and spec.get(\"family\") == \"qwen35\")\n    payload: Dict[str, Any] = {\n        \"messages\": messages,\n        \"max_tokens\": int(max_tokens),\n        \"temperature\": float(temperature),\n        \"chat_template_kwargs\": {\"enable_thinking\": use_thinking},\n        \"reasoning_format\": \"deepseek\" if use_thinking else \"none\",\n    }\n    if not use_thinking:\n        payload[\"reasoning_budget\"] = 0\n\n    with _INFERENCE_LOCK:\n        try:\n            r = requests.post(f\"{SERVER_URL}/v1/chat/completions\", json=payload, timeout=180)\n            r.raise_for_status()\n            msg_obj = r.json()[\"choices\"][0][\"message\"]\n            raw_reply = extract_message_text(msg_obj, allow_reasoning_fallback=True)\n            thinking_trace = (msg_obj.get(\"reasoning_content\") or \"\").strip() if use_thinking else \"\"\n            cleaned_reply = clean_vlm_response(\n                raw_reply, strip_meta_preamble=False, single_line=False\n            )\n            return {\n                \"ok\": True,\n                \"reply\": cleaned_reply,\n                \"thinking\": thinking_trace,\n                \"model_used\": model_key,\n            }\n        except Exception as e:\n            return {\n                \"ok\": False,\n                \"reply\": f\"⚠️ Error saat menghubungi mesin VLM: {e}\",\n                \"thinking\": \"\",\n                \"model_used\": model_key,\n            }\n\n\ndef list_downloaded_models() -> List[Dict[str, Any]]:\n    \"\"\"Return status and disk usage of all 8 models in MODEL_REGISTRY.\"\"\"\n    results = []\n    for key, spec in MODEL_REGISTRY.items():\n        downloaded, size_gb = is_gguf_model_downloaded(key)\n        is_active = (\n            (ACTIVE_STATE.get(\"model_key\") == key and is_server_healthy())\n            if spec.get(\"engine\") == \"llama_cpp\"\n            else (TAGGER_STATE.get(\"status\") == \"ONLINE\")\n        )\n        results.append({\n            \"key\": key,\n            \"engine\": spec.get(\"engine\", \"llama_cpp\"),\n            \"family\": spec.get(\"family\", \"\"),\n            \"desc\": spec.get(\"desc\", \"\"),\n            \"downloaded\": downloaded,\n            \"size_gb\": size_gb if downloaded else spec.get(\"approx_size_gb\", 5.8),\n            \"active_in_vram\": is_active,\n            \"sampling\": spec.get(\"caption_sampling\", {}),\n        })\n    return results\n\n\ndef delete_downloaded_model(model_key: str) -> Dict[str, Any]:\n    \"\"\"Delete cached model files on disk for `model_key` (unloading from VRAM first if active).\"\"\"\n    if model_key not in MODEL_REGISTRY:\n        return {\"ok\": False, \"msg\": f\"Model '{model_key}' tidak dikenal.\"}\n\n    spec = MODEL_REGISTRY[model_key]\n    if spec.get(\"engine\") == \"pixai_tagger\":\n        unload_pixai_tagger()\n        freed_bytes = 0\n        if TAGGER_CACHE_DIR.exists():\n            for p in TAGGER_CACHE_DIR.rglob(\"*\"):\n                if p.is_file():\n                    freed_bytes += p.stat().st_size\n            shutil.rmtree(TAGGER_CACHE_DIR, ignore_errors=True)\n        return {\n            \"ok\": True,\n            \"msg\": f\"Berhasil menghapus cache PixAI Tagger v0.9 ({freed_bytes / (1024**3):.2f} GB dibebaskan).\",\n        }\n\n    if ACTIVE_STATE.get(\"model_key\") == model_key:\n        stop_llama_server()\n\n    freed_bytes = 0\n    target_names = {Path(spec[\"file\"]).name, Path(spec[\"mmproj_file\"]).name}\n\n    repo_dir = _repo_local_dir(spec[\"repo\"])\n    if repo_dir.exists():\n        for p in repo_dir.rglob(\"*\"):\n            if p.is_file() and (p.name == Path(spec[\"file\"]).name or spec[\"repo\"] != spec[\"mmproj_repo\"]):\n                freed_bytes += p.stat().st_size\n                p.unlink(missing_ok=True)\n\n    hf_cache_root = Path.home() / \".cache\" / \"huggingface\" / \"hub\"\n    if hf_cache_root.exists():\n        for fname in target_names:\n            for m in hf_cache_root.rglob(fname):\n                try:\n                    real_p = m.resolve()\n                    if real_p.is_file():\n                        freed_bytes += real_p.stat().st_size\n                        real_p.unlink(missing_ok=True)\n                    m.unlink(missing_ok=True)\n                except Exception:\n                    pass\n\n    return {\n        \"ok\": True,\n        \"msg\": f\"Berhasil menghapus file GGUF '{model_key.split('[')[0].strip()}' ({freed_bytes / (1024**3):.2f} GB dibebaskan).\",\n    }\n\n\ndef get_system_telemetry() -> Dict[str, Any]:\n    \"\"\"\n    Collect live GPU VRAM, System RAM, Disk Free Space, llama-server status,\n    PixAI Tagger status, and per-model disk cache inventory.\n    \"\"\"\n    vram_used_gb = 0.0\n    vram_total_gb = 15.0\n    gpu_name = \"NVIDIA Tesla T4\"\n\n    try:\n        res = subprocess.run(\n            [\"nvidia-smi\", \"--query-gpu=name,memory.used,memory.total\", \"--format=csv,noheader,nounits\"],\n            capture_output=True,\n            text=True,\n            timeout=3,\n        )\n        if res.returncode == 0 and res.stdout.strip():\n            parts = [x.strip() for x in res.stdout.strip().splitlines()[0].split(\",\")]\n            if len(parts) >= 3:\n                gpu_name = parts[0]\n                vram_used_gb = round(float(parts[1]) / 1024.0, 2)\n                vram_total_gb = round(float(parts[2]) / 1024.0, 2)\n    except Exception:\n        try:\n            import torch\n            if torch.cuda.is_available():\n                gpu_name = torch.cuda.get_device_name(0)\n                free_b, tot_b = torch.cuda.mem_get_info(0)\n                vram_total_gb = round(tot_b / (1024 ** 3), 2)\n                vram_used_gb = round((tot_b - free_b) / (1024 ** 3), 2)\n        except Exception:\n            pass\n\n    check_disk_path = \"/content\" if os.path.exists(\"/content\") else \"/\"\n    usage = shutil.disk_usage(check_disk_path)\n    disk_free_gb = round(usage.free / (1024 ** 3), 2)\n    disk_total_gb = round(usage.total / (1024 ** 3), 2)\n\n    srv_status = ACTIVE_STATE.get(\"status\", \"STANDBY\")\n    if srv_status == \"ONLINE\" and not is_server_healthy():\n        srv_status = \"STANDBY\"\n        ACTIVE_STATE[\"status\"] = \"STANDBY\"\n\n    return {\n        \"server_status\": srv_status,\n        \"active_model\": ACTIVE_STATE.get(\"model_key\"),\n        \"tagger_status\": TAGGER_STATE.get(\"status\", \"STANDBY\"),\n        \"gpu_name\": gpu_name,\n        \"vram_used_gb\": vram_used_gb,\n        \"vram_total_gb\": vram_total_gb,\n        \"disk_free_gb\": disk_free_gb,\n        \"disk_total_gb\": disk_total_gb,\n        \"models_inventory\": list_downloaded_models(),\n    }\n",
  "core/dataset_manager.py": "\"\"\"\ncore/dataset_manager.py\nDataset & File Operations Manager:\n- Fast dataset scanning with image dimensions (width/height, low-res <512px flag), word counts, and token estimates\n- Safe `.zip` extraction ignoring `__MACOSX`, `.DS_Store`, `._*`, hidden files, with collision-free flattening and Pillow verification\n- Single-item caption save & single-item image+txt deletion\n- Bulk Trigger Word & Find/Replace with Auto-Backup & Restore\n- Smart File Manager (9 Actions + One-Click Google Drive Export/Import)\n\"\"\"\n\nimport os\nimport glob\nimport time\nimport shutil\nimport zipfile\nimport tempfile\nfrom pathlib import Path\nfrom typing import Any, Dict, List, Optional, Tuple\n\nfrom PIL import Image\n\nfrom core.text_cleaner import process_caption_text, estimate_token_count\n\n\ndef _resolve_writable_default(preferred_path: str, fallback_subdir: str) -> str:\n    \"\"\"Use `/content/...` on Google Colab when writable; otherwise fall back cleanly to `/tmp/...`.\"\"\"\n    try:\n        os.makedirs(preferred_path, exist_ok=True)\n        return preferred_path\n    except PermissionError:\n        alt = os.path.join(tempfile.gettempdir(), fallback_subdir)\n        os.makedirs(alt, exist_ok=True)\n        return alt\n\n\nDEFAULT_DATASET_DIR = _resolve_writable_default(\n    os.environ.get(\"VLM_DATASET_DIR\", \"/content/dataset\"), \"vlm_dataset\"\n)\nBACKUP_DIR = _resolve_writable_default(\n    os.environ.get(\"VLM_BACKUP_DIR\", \"/content/txt_backup\"), \"vlm_txt_backup\"\n)\nIMAGE_EXTENSIONS = {\".jpg\", \".jpeg\", \".png\", \".webp\", \".bmp\"}\n\n\ndef verify_and_inspect_image(img_path: Path) -> Tuple[bool, int, int]:\n    \"\"\"\n    Safely verify an image file with Pillow (`verify()` on first handle, dimensions on second handle).\n    If corrupted, returns `(False, 0, 0)`.\n    \"\"\"\n    try:\n        with Image.open(img_path) as im:\n            im.verify()\n        with Image.open(img_path) as im2:\n            w, h = im2.size\n            return True, int(w), int(h)\n    except Exception:\n        return False, 0, 0\n\n\ndef scan_dataset_items(folder_path: str = DEFAULT_DATASET_DIR) -> List[Dict[str, Any]]:\n    \"\"\"\n    Scan dataset directory and return sorted list of image items with caption, words,\n    estimated tokens, image dimensions (`width`, `height`), and quality flags (`low_res`).\n    \"\"\"\n    folder = Path(folder_path)\n    try:\n        folder.mkdir(parents=True, exist_ok=True)\n    except PermissionError:\n        folder = Path(DEFAULT_DATASET_DIR)\n        folder.mkdir(parents=True, exist_ok=True)\n\n    imgs = sorted(\n        [p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS and not p.name.startswith(\".\")],\n        key=lambda p: p.name.lower(),\n    )\n\n    items: List[Dict[str, Any]] = []\n    for p in imgs:\n        txt_p = p.with_suffix(\".txt\")\n        cap = \"\"\n        if txt_p.is_file():\n            try:\n                cap = txt_p.read_text(encoding=\"utf-8\", errors=\"ignore\").strip()\n            except Exception:\n                pass\n\n        width, height = 0, 0\n        try:\n            with Image.open(p) as im:\n                width, height = im.size\n        except Exception:\n            pass\n\n        stats = estimate_token_count(cap)\n        low_res = bool(width > 0 and height > 0 and min(width, height) < 512)\n\n        items.append({\n            \"name\": p.name,\n            \"img_path\": str(p),\n            \"txt_path\": str(txt_p),\n            \"caption\": cap,\n            \"words\": stats[\"words\"],\n            \"tokens\": stats[\"tokens\"],\n            \"width\": width,\n            \"height\": height,\n            \"low_res\": low_res,\n        })\n    return items\n\n\ndef save_single_caption(img_path: str, caption: str) -> Dict[str, Any]:\n    \"\"\"Save or delete `.txt` caption file corresponding to `img_path`.\"\"\"\n    p = Path(img_path)\n    txt_path = p.with_suffix(\".txt\")\n    clean_cap = (caption or \"\").strip()\n    if clean_cap:\n        txt_path.write_text(clean_cap, encoding=\"utf-8\")\n    elif txt_path.exists():\n        txt_path.unlink(missing_ok=True)\n    stats = estimate_token_count(clean_cap)\n    return {\"ok\": True, \"words\": stats[\"words\"], \"tokens\": stats[\"tokens\"]}\n\n\ndef delete_single_item(img_path: str) -> Dict[str, Any]:\n    \"\"\"Delete an image and its companion `.txt` file from the dataset.\"\"\"\n    p = Path(img_path)\n    txt_p = p.with_suffix(\".txt\")\n    removed = False\n    if p.is_file():\n        p.unlink(missing_ok=True)\n        removed = True\n    if txt_p.is_file():\n        txt_p.unlink(missing_ok=True)\n    return {\"ok\": removed, \"deleted_path\": str(p)}\n\n\ndef _is_junk_archive_member(member_name: str) -> bool:\n    \"\"\"Return True if a ZIP entry is a macOS/Linux hidden or metadata artifact.\"\"\"\n    norm = member_name.replace(\"\\\\\", \"/\")\n    parts = [pt for pt in norm.split(\"/\") if pt]\n    if not parts:\n        return True\n    for pt in parts:\n        if pt in (\"__MACOSX\", \".DS_Store\", \"Thumbs.db\") or pt.startswith(\"._\") or pt.startswith(\".\"):\n            return True\n    return False\n\n\ndef _unique_stem_in_folder(folder: Path, desired_stem: str, img_ext: str) -> str:\n    \"\"\"Return a stem that does not collide with any existing image or txt file in `folder`.\"\"\"\n    candidate = desired_stem\n    counter = 1\n    while (folder / f\"{candidate}{img_ext}\").exists() or (folder / f\"{candidate}.txt\").exists():\n        candidate = f\"{desired_stem}_{counter}\"\n        counter += 1\n    return candidate\n\n\ndef safe_extract_zip_to_dataset(zip_path: Path, target_folder: Path) -> Tuple[int, int]:\n    \"\"\"\n    Safely extract a `.zip` archive into `target_folder`:\n    - Ignores `__MACOSX`, `.DS_Store`, `._*`, and hidden files.\n    - Flattens subdirectories while keeping image + `.txt` pairs matched even when renamed for collision avoidance.\n    - Verifies every extracted image with Pillow and deletes corrupted image files automatically.\n    Returns `(added_images_count, rejected_corrupt_count)`.\n    \"\"\"\n    tmp_extract = target_folder / f\"__tmp_zip_{int(time.time() * 1000)}\"\n    tmp_extract.mkdir(parents=True, exist_ok=True)\n    added = 0\n    corrupt = 0\n\n    try:\n        with zipfile.ZipFile(zip_path, \"r\") as zf:\n            for member in zf.infolist():\n                if member.is_dir() or _is_junk_archive_member(member.filename):\n                    continue\n                zf.extract(member, path=str(tmp_extract))\n\n        all_files = [p for p in tmp_extract.rglob(\"*\") if p.is_file() and not _is_junk_archive_member(p.name)]\n        img_files = [p for p in all_files if p.suffix.lower() in IMAGE_EXTENSIONS]\n        orphan_txts = set(p for p in all_files if p.suffix.lower() == \".txt\")\n\n        for img_p in sorted(img_files, key=lambda x: str(x).lower()):\n            ok_img, _, _ = verify_and_inspect_image(img_p)\n            companion_txt = img_p.with_suffix(\".txt\")\n            if companion_txt in orphan_txts:\n                orphan_txts.remove(companion_txt)\n\n            if not ok_img:\n                corrupt += 1\n                img_p.unlink(missing_ok=True)\n                continue\n\n            ext = img_p.suffix.lower()\n            safe_stem = _unique_stem_in_folder(target_folder, img_p.stem, ext)\n            dst_img = target_folder / f\"{safe_stem}{ext}\"\n            shutil.move(str(img_p), str(dst_img))\n            added += 1\n\n            if companion_txt.is_file():\n                dst_txt = target_folder / f\"{safe_stem}.txt\"\n                shutil.move(str(companion_txt), str(dst_txt))\n\n        for txt_p in orphan_txts:\n            dst_txt = target_folder / txt_p.name\n            if not dst_txt.exists():\n                shutil.move(str(txt_p), str(dst_txt))\n\n    finally:\n        shutil.rmtree(tmp_extract, ignore_errors=True)\n        zip_path.unlink(missing_ok=True)\n\n    return added, corrupt\n\n\ndef process_uploaded_file_bytes(\n    folder_path: str,\n    filename: str,\n    content_bytes: bytes,\n) -> Tuple[int, int]:\n    \"\"\"\n    Save an uploaded file (image, `.txt`, or `.zip`) into `folder_path` with corruption verification.\n    Returns `(added_count, corrupt_count)`.\n    \"\"\"\n    folder = Path(folder_path)\n    folder.mkdir(parents=True, exist_ok=True)\n    safe_name = Path(filename or f\"upload_{int(time.time())}.png\").name\n    if _is_junk_archive_member(safe_name):\n        return 0, 0\n\n    suffix = Path(safe_name).suffix.lower()\n    if suffix == \".zip\":\n        tmp_zip = folder / f\"__upload_{int(time.time() * 1000)}_{safe_name}\"\n        tmp_zip.write_bytes(content_bytes)\n        return safe_extract_zip_to_dataset(tmp_zip, folder)\n\n    if suffix in IMAGE_EXTENSIONS:\n        dst_path = folder / safe_name\n        dst_path.write_bytes(content_bytes)\n        ok_img, _, _ = verify_and_inspect_image(dst_path)\n        if not ok_img:\n            dst_path.unlink(missing_ok=True)\n            return 0, 1\n        return 1, 0\n\n    if suffix == \".txt\":\n        dst_path = folder / safe_name\n        dst_path.write_bytes(content_bytes)\n        return 1, 0\n\n    return 0, 0\n\n\ndef create_dataset_zip(folder_path: str = DEFAULT_DATASET_DIR) -> Path:\n    \"\"\"Create a downloadable `.zip` archive of all valid files in `folder_path`.\"\"\"\n    fdir = Path(folder_path)\n    files = [\n        p for p in fdir.iterdir()\n        if p.is_file() and not p.name.startswith(\".\") and not p.name.startswith(\"__tmp_\")\n    ] if fdir.exists() else []\n\n    zip_dir = Path(\"/content\") if os.access(\"/content\", os.W_OK) else Path(tempfile.gettempdir())\n    out_zip = zip_dir / f\"dataset_labeled_{time.strftime('%Y%m%d_%H%M%S')}.zip\"\n    with zipfile.ZipFile(out_zip, \"w\", zipfile.ZIP_DEFLATED) as zf:\n        for p in sorted(files, key=lambda x: x.name.lower()):\n            zf.write(p, arcname=p.name)\n    return out_zip\n\n\ndef run_bulk_caption_edit(\n    folder_path: str,\n    action_type: str = \"apply\",\n    trigger: str = \"\",\n    pos: str = \"front\",\n    find_str: str = \"\",\n    replace_str: str = \"\",\n    lowercase: bool = False,\n    skip_dup: bool = True,\n    clean_commas: bool = True,\n    backup: bool = True,\n    selected_img_paths: Optional[List[str]] = None,\n) -> Dict[str, Any]:\n    \"\"\"\n    Apply bulk trigger word insertion and/or find-and-replace across `.txt` files in `folder_path`\n    (or only on `selected_img_paths` if provided), with automatic backup to `BACKUP_DIR`.\n    \"\"\"\n    folder = Path(folder_path)\n    folder.mkdir(parents=True, exist_ok=True)\n    os.makedirs(BACKUP_DIR, exist_ok=True)\n\n    if action_type == \"restore\":\n        bfiles = glob.glob(os.path.join(BACKUP_DIR, \"*.txt\"))\n        for bf in bfiles:\n            shutil.copy2(bf, os.path.join(folder_path, os.path.basename(bf)))\n        return {\n            \"ok\": True,\n            \"msg\": f\"Berhasil memulihkan {len(bfiles)} file .txt dari cadangan (backup).\",\n            \"items\": scan_dataset_items(folder_path),\n        }\n\n    if selected_img_paths:\n        target_txts = []\n        for img_p in selected_img_paths:\n            tp = Path(img_p).with_suffix(\".txt\")\n            if not tp.exists() and trigger.strip():\n                tp.write_text(\"\", encoding=\"utf-8\")\n            if tp.exists():\n                target_txts.append(str(tp))\n    else:\n        target_txts = sorted(glob.glob(os.path.join(folder_path, \"*.txt\")))\n\n    pos_code = \"front\" if pos == \"front\" else \"back\"\n    mod_cnt = 0\n\n    for fpath in target_txts:\n        fname = os.path.basename(fpath)\n        orig = Path(fpath).read_text(encoding=\"utf-8\", errors=\"ignore\")\n        if backup:\n            bp = os.path.join(BACKUP_DIR, fname)\n            if not os.path.exists(bp):\n                Path(bp).write_text(orig, encoding=\"utf-8\")\n\n        cur = orig.replace(find_str, replace_str) if find_str else orig\n        if trigger.strip():\n            updated = process_caption_text(cur, trigger, pos_code, lowercase, skip_dup, clean_commas)\n        else:\n            updated = process_caption_text(cur, \"\", pos_code, lowercase, False, clean_commas)\n\n        if updated is not None and updated != orig:\n            Path(fpath).write_text(updated, encoding=\"utf-8\")\n            mod_cnt += 1\n\n    return {\n        \"ok\": True,\n        \"msg\": f\"Berhasil memperbarui {mod_cnt} file caption (.txt).\",\n        \"items\": scan_dataset_items(folder_path),\n    }\n\n\ndef run_smart_file_manager(\n    action: str,\n    source_path: str,\n    destination_path: str = \"/content/dataset.zip\",\n    rename_prefix: str = \"dataset_img\",\n    active_dataset_dir: str = DEFAULT_DATASET_DIR,\n) -> Dict[str, Any]:\n    \"\"\"\n    Execute any of the 9 Smart File Manager actions + Google Drive Export/Import actions.\n    \"\"\"\n    log = [f\"=== SMART FILE MANAGER: {action.upper()} ===\", f\"[+] Source Path      : {source_path}\"]\n    if destination_path:\n        log.append(f\"[+] Destination Path : {destination_path}\")\n\n    try:\n        if action == \"Count Statistics\":\n            if not os.path.exists(source_path):\n                log.append(\"[!] Path sumber tidak ditemukan.\")\n            elif os.path.isfile(source_path):\n                log.append(f\"-> Ukuran File: {os.path.getsize(source_path) / (1024 * 1024):.2f} MB\")\n            else:\n                all_f = [\n                    f for f in glob.glob(os.path.join(source_path, \"**/*\"), recursive=True)\n                    if os.path.isfile(f)\n                ]\n                ext_counts: Dict[str, int] = {}\n                tot_sz = 0\n                for f in all_f:\n                    ext = os.path.splitext(f)[1].lower() or \"no_ext\"\n                    ext_counts[ext] = ext_counts.get(ext, 0) + 1\n                    tot_sz += os.path.getsize(f)\n                items = scan_dataset_items(source_path)\n                cap_cnt = sum(1 for x in items if x[\"caption\"])\n                low_res_cnt = sum(1 for x in items if x[\"low_res\"])\n                log.append(f\"-> Total File          : {len(all_f)}\")\n                log.append(f\"-> Total Ukuran        : {tot_sz / (1024 * 1024):.2f} MB\")\n                log.append(f\"-> Gambar Ber-Caption  : {cap_cnt} / {len(items)} gambar\")\n                log.append(f\"-> Gambar Low-Res <512 : {low_res_cnt} gambar\")\n                for ext, c in sorted(ext_counts.items()):\n                    log.append(f\"   • {ext:<10} : {c} file\")\n\n        elif action == \"Zip Folder\":\n            if os.path.isdir(source_path):\n                out_zip = destination_path if destination_path.endswith(\".zip\") else destination_path + \".zip\"\n                os.makedirs(os.path.dirname(out_zip) or \".\", exist_ok=True)\n                shutil.make_archive(out_zip[:-4], \"zip\", source_path)\n                log.append(f\"[✓] Berhasil di-ZIP ke '{out_zip}' ({os.path.getsize(out_zip) / (1024 * 1024):.2f} MB)\")\n            else:\n                log.append(\"[!] Folder sumber tidak ditemukan.\")\n\n        elif action == \"Unzip File\":\n            if os.path.isfile(source_path) and source_path.endswith(\".zip\"):\n                dest_dir = destination_path[:-4] if destination_path.endswith(\".zip\") else destination_path\n                os.makedirs(dest_dir, exist_ok=True)\n                tmp_copy = Path(dest_dir) / f\"__tmp_mgr_{int(time.time())}.zip\"\n                shutil.copy2(source_path, tmp_copy)\n                added, corrupt = safe_extract_zip_to_dataset(tmp_copy, Path(dest_dir))\n                log.append(f\"[✓] Ekstraksi aman selesai ke '{dest_dir}' ({added} gambar valid, {corrupt} korup diabaikan).\")\n            else:\n                log.append(\"[!] File ZIP sumber tidak ditemukan.\")\n\n        elif action in (\"Copy\", \"Export to Google Drive\", \"Import from Google Drive\"):\n            if not os.path.exists(source_path):\n                log.append(f\"[!] Path sumber '{source_path}' tidak ditemukan. Pastikan Google Drive sudah di-mount jika mengakses /content/drive.\")\n            else:\n                if os.path.isdir(source_path):\n                    os.makedirs(destination_path, exist_ok=True)\n                    shutil.copytree(source_path, destination_path, dirs_exist_ok=True)\n                else:\n                    os.makedirs(os.path.dirname(destination_path) or \".\", exist_ok=True)\n                    shutil.copy2(source_path, destination_path)\n                log.append(f\"[✓] Berhasil disalin ke '{destination_path}'\")\n\n        elif action == \"Move\":\n            if os.path.exists(source_path):\n                os.makedirs(os.path.dirname(destination_path) or \".\", exist_ok=True)\n                shutil.move(source_path, destination_path)\n                log.append(f\"[✓] Berhasil dipindahkan ke '{destination_path}'\")\n            else:\n                log.append(\"[!] Path sumber tidak ditemukan.\")\n\n        elif action == \"Delete\":\n            if os.path.isdir(source_path):\n                shutil.rmtree(source_path)\n                os.makedirs(source_path, exist_ok=True)\n                log.append(f\"[✓] Isi folder '{source_path}' berhasil dikosongkan.\")\n            elif os.path.isfile(source_path):\n                os.remove(source_path)\n                log.append(f\"[✓] File '{source_path}' berhasil dihapus.\")\n            else:\n                log.append(\"[!] Path sumber tidak ditemukan.\")\n\n        elif action == \"Sync Images & Captions\":\n            if os.path.isdir(source_path):\n                img_map, txt_map = {}, {}\n                for f in os.listdir(source_path):\n                    p = os.path.join(source_path, f)\n                    if os.path.isfile(p):\n                        b, e = os.path.splitext(f)\n                        if e.lower() in IMAGE_EXTENSIONS:\n                            img_map[b] = f\n                        elif e.lower() == \".txt\":\n                            txt_map[b] = f\n                orphaned_txt = [txt_map[b] for b in txt_map if b not in img_map and b != \"classes\"]\n                for f in orphaned_txt:\n                    os.remove(os.path.join(source_path, f))\n                    log.append(f\"   [Deleted Orphan TXT] {f}\")\n                log.append(f\"[✓] Sinkronisasi selesai ({len(orphaned_txt)} file .txt yatim dihapus).\")\n            else:\n                log.append(\"[!] Folder sumber tidak ditemukan.\")\n\n        elif action == \"Batch Rename\":\n            if os.path.isdir(source_path):\n                pairs, seen = [], set()\n                for f in sorted(os.listdir(source_path)):\n                    p = os.path.join(source_path, f)\n                    if os.path.isfile(p):\n                        b, e = os.path.splitext(f)\n                        if e.lower() in IMAGE_EXTENSIONS and b not in seen:\n                            txt_f = b + \".txt\"\n                            has_txt = os.path.exists(os.path.join(source_path, txt_f))\n                            pairs.append((f, txt_f if has_txt else None))\n                            seen.add(b)\n                temp_pairs = []\n                for idx, (img_f, txt_f) in enumerate(pairs, 1):\n                    new_b = f\"{rename_prefix}_{str(idx).zfill(3)}\"\n                    _, ext = os.path.splitext(img_f)\n                    t_img = f\"__tmp_{new_b}{ext}\"\n                    os.rename(os.path.join(source_path, img_f), os.path.join(source_path, t_img))\n                    t_txt = None\n                    if txt_f:\n                        t_txt = f\"__tmp_{new_b}.txt\"\n                        os.rename(os.path.join(source_path, txt_f), os.path.join(source_path, t_txt))\n                    temp_pairs.append((t_img, t_txt, new_b + ext, new_b + \".txt\" if txt_f else None))\n                for t_img, t_txt, f_img, f_txt in temp_pairs:\n                    os.rename(os.path.join(source_path, t_img), os.path.join(source_path, f_img))\n                    if t_txt and f_txt:\n                        os.rename(os.path.join(source_path, t_txt), os.path.join(source_path, f_txt))\n                log.append(f\"[✓] Batch rename selesai untuk {len(pairs)} pasang gambar + caption!\")\n            else:\n                log.append(\"[!] Folder sumber tidak ditemukan.\")\n\n        elif action == \"Clean Empty Files\":\n            if os.path.isdir(source_path):\n                empty = [\n                    f for f in glob.glob(os.path.join(source_path, \"*\"))\n                    if os.path.isfile(f) and os.path.getsize(f) == 0\n                ]\n                for f in empty:\n                    os.remove(f)\n                    log.append(f\"   [Deleted Empty] {os.path.basename(f)}\")\n                log.append(f\"[✓] Dihapus {len(empty)} file berukuran 0 byte.\")\n            else:\n                log.append(\"[!] Folder sumber tidak ditemukan.\")\n\n    except Exception as e:\n        log.append(f\"[!] Error saat menjalankan '{action}': {e}\")\n\n    return {\n        \"ok\": True,\n        \"log\": \"\\n\".join(log),\n        \"items\": scan_dataset_items(active_dataset_dir),\n    }\n",
  "ui/__init__.py": "\"\"\"UI package for Vision AI Studio.\"\"\"\nfrom .studio_routes import register_studio_routes, build_studio_html\n\n__all__ = [\"register_studio_routes\", \"build_studio_html\"]\n",
  "ui/studio_routes.py": "\"\"\"\nui/studio_routes.py\n100% Non-Blocking FastAPI Endpoints for Vision AI Dataset Labeling, Booru Tagging,\nMultimodal Chat & Smart File Manager Studio.\nAll heavy I/O and GPU operations are dispatched via `asyncio.to_thread(...)` so UI telemetry,\nimage thumbnails, and caption saving respond in <10ms even during heavy batch inference.\n\"\"\"\n\nimport os\nimport json\nimport asyncio\nfrom pathlib import Path\nfrom typing import Any, Dict\n\nfrom fastapi import FastAPI, Request, UploadFile, File, Form\nfrom fastapi.responses import HTMLResponse, JSONResponse, FileResponse\n\nfrom config.vlm_catalog import (\n    MODEL_REGISTRY,\n    STYLE_PRESETS,\n    DEFAULT_MODEL_KEY,\n    PIXAI_TAGGER_KEY,\n    get_model_spec,\n)\nfrom core.llama_manager import (\n    ACTIVE_STATE,\n    ensure_llama_server,\n    unload_vlm_server,\n    run_vlm_caption_single,\n    run_multimodal_chat,\n    get_system_telemetry,\n    delete_downloaded_model,\n    image_to_data_uri,\n)\nfrom core.tagger_manager import (\n    ensure_pixai_tagger,\n    unload_pixai_tagger,\n    run_pixai_tagger,\n)\nfrom core.dataset_manager import (\n    DEFAULT_DATASET_DIR,\n    scan_dataset_items,\n    save_single_caption,\n    delete_single_item,\n    process_uploaded_file_bytes,\n    create_dataset_zip,\n    run_bulk_caption_edit,\n    run_smart_file_manager,\n)\nfrom core.text_cleaner import estimate_token_count\n\nTEMPLATE_PATH = Path(__file__).parent / \"templates\" / \"studio.html\"\n\n\ndef get_initial_boot_payload(folder: str = DEFAULT_DATASET_DIR) -> Dict[str, Any]:\n    \"\"\"Build initial JSON payload injected directly into studio.html for 0ms boot.\"\"\"\n    default_key = os.environ.get(\"VLM_DEFAULT_MODEL\", DEFAULT_MODEL_KEY)\n    if default_key not in MODEL_REGISTRY:\n        default_key = DEFAULT_MODEL_KEY\n\n    models_list = []\n    for k, v in MODEL_REGISTRY.items():\n        models_list.append({\n            \"key\": k,\n            \"engine\": v.get(\"engine\", \"llama_cpp\"),\n            \"family\": v.get(\"family\", \"\"),\n            \"desc\": v.get(\"desc\", \"\"),\n            \"approx_size_gb\": v.get(\"approx_size_gb\", 5.8),\n            \"sampling\": v.get(\"caption_sampling\", {}),\n        })\n\n    return {\n        \"folder\": folder,\n        \"models\": models_list,\n        \"default_model\": ACTIVE_STATE.get(\"model_key\") or default_key,\n        \"pixai_tagger_key\": PIXAI_TAGGER_KEY,\n        \"styles\": STYLE_PRESETS,\n    }\n\n\ndef build_studio_html(folder: str = DEFAULT_DATASET_DIR) -> str:\n    \"\"\"Read `ui/templates/studio.html` and inject `INITIAL_BOOT_JSON`.\"\"\"\n    raw_html = TEMPLATE_PATH.read_text(encoding=\"utf-8\")\n    boot_json = json.dumps(get_initial_boot_payload(folder))\n    return raw_html.replace(\"__INITIAL_BOOT_JSON__\", boot_json)\n\n\ndef register_studio_routes(app: FastAPI) -> None:\n    \"\"\"Register all `/studio_ui` and `/studio_api/*` routes on the FastAPI `app` instance.\"\"\"\n\n    @app.get(\"/studio_ui\", response_class=HTMLResponse)\n    async def route_studio_ui(folder: str = DEFAULT_DATASET_DIR):\n        html_content = await asyncio.to_thread(build_studio_html, folder)\n        return HTMLResponse(html_content)\n\n    @app.get(\"/studio_api/init\")\n    async def route_init(folder: str = DEFAULT_DATASET_DIR):\n        items, telemetry = await asyncio.gather(\n            asyncio.to_thread(scan_dataset_items, folder),\n            asyncio.to_thread(get_system_telemetry),\n        )\n        payload = get_initial_boot_payload(folder)\n        payload[\"items\"] = items\n        payload[\"telemetry\"] = telemetry\n        return payload\n\n    @app.get(\"/studio_api/telemetry\")\n    async def route_telemetry():\n        data = await asyncio.to_thread(get_system_telemetry)\n        return data\n\n    @app.get(\"/studio_api/image\")\n    async def route_image(path: str, as_data_uri: int = 0):\n        p = Path(path)\n        if not p.is_file():\n            return JSONResponse({\"error\": \"Image file not found\"}, status_code=404)\n        if as_data_uri == 1:\n            uri = await asyncio.to_thread(image_to_data_uri, str(p))\n            return {\"ok\": True, \"name\": p.name, \"data_uri\": uri}\n        return FileResponse(str(p))\n\n    @app.post(\"/studio_api/save_caption\")\n    async def route_save_caption(req: Request):\n        data = await req.json()\n        img_path = data.get(\"img_path\", \"\")\n        caption = data.get(\"caption\", \"\")\n        res = await asyncio.to_thread(save_single_caption, img_path, caption)\n        return res\n\n    @app.post(\"/studio_api/delete_item\")\n    async def route_delete_item(req: Request):\n        data = await req.json()\n        img_path = data.get(\"img_path\", \"\")\n        folder = data.get(\"folder\", DEFAULT_DATASET_DIR)\n        res = await asyncio.to_thread(delete_single_item, img_path)\n        items = await asyncio.to_thread(scan_dataset_items, folder)\n        res[\"items\"] = items\n        return res\n\n    @app.post(\"/studio_api/upload\")\n    async def route_upload(folder: str = Form(DEFAULT_DATASET_DIR), files: list[UploadFile] = File(...)):\n        added_total = 0\n        corrupt_total = 0\n        for uf in files:\n            fname = uf.filename or \"upload.png\"\n            content = await uf.read()\n            added, corrupt = await asyncio.to_thread(process_uploaded_file_bytes, folder, fname, content)\n            added_total += added\n            corrupt_total += corrupt\n        items = await asyncio.to_thread(scan_dataset_items, folder)\n        return {\n            \"ok\": True,\n            \"added\": added_total,\n            \"corrupt\": corrupt_total,\n            \"items\": items,\n        }\n\n    @app.post(\"/studio_api/switch_model\")\n    async def route_switch_model(req: Request):\n        data = await req.json()\n        model_key = data.get(\"model_key\", DEFAULT_MODEL_KEY)\n        spec = get_model_spec(model_key)\n        try:\n            if spec.get(\"engine\") == \"pixai_tagger\":\n                await asyncio.to_thread(ensure_pixai_tagger)\n            else:\n                await asyncio.to_thread(ensure_llama_server, model_key)\n            telemetry = await asyncio.to_thread(get_system_telemetry)\n            return {\n                \"ok\": True,\n                \"model_key\": model_key,\n                \"msg\": f\"Model '{model_key.split('[')[0].strip()}' siap di VRAM GPU.\",\n                \"telemetry\": telemetry,\n            }\n        except Exception as e:\n            telemetry = await asyncio.to_thread(get_system_telemetry)\n            return JSONResponse(\n                {\"ok\": False, \"msg\": f\"Gagal memuat model: {e}\", \"telemetry\": telemetry},\n                status_code=500,\n            )\n\n    @app.post(\"/studio_api/unload_vram\")\n    async def route_unload_vram(req: Request):\n        data = await req.json()\n        target = data.get(\"target\", \"all\")  # vlm | tagger | all\n        if target == \"tagger\":\n            await asyncio.to_thread(unload_pixai_tagger)\n            msg = \"PixAI Tagger v0.9 berhasil di-unload dari VRAM.\"\n        elif target == \"vlm\":\n            await asyncio.to_thread(unload_vlm_server, False)\n            msg = \"Model VLM (llama-server) berhasil di-unload dari VRAM.\"\n        else:\n            await asyncio.to_thread(unload_vlm_server, True)\n            msg = \"Seluruh model VLM & PixAI Tagger berhasil di-unload dari VRAM.\"\n        telemetry = await asyncio.to_thread(get_system_telemetry)\n        return {\"ok\": True, \"msg\": msg, \"telemetry\": telemetry}\n\n    @app.post(\"/studio_api/delete_model_cache\")\n    async def route_delete_model_cache(req: Request):\n        data = await req.json()\n        model_key = data.get(\"model_key\", \"\")\n        res = await asyncio.to_thread(delete_downloaded_model, model_key)\n        telemetry = await asyncio.to_thread(get_system_telemetry)\n        res[\"telemetry\"] = telemetry\n        return res\n\n    @app.post(\"/studio_api/generate_one\")\n    async def route_generate_one(req: Request):\n        data = await req.json()\n        img_path = data[\"img_path\"]\n        model_key = data.get(\"model_key\", DEFAULT_MODEL_KEY)\n        sys_prompt = data.get(\"sys_prompt\", \"\")\n        usr_prompt = data.get(\"usr_prompt\", \"Describe this image in detail.\")\n        max_tokens = int(data.get(\"max_tokens\", 220))\n        temperature = float(data.get(\"temperature\", 0.2))\n        subject_hint = data.get(\"subject_hint\", \"\")\n        hybrid_tagger_enabled = bool(data.get(\"hybrid_tagger_enabled\", False))\n        hybrid_mode = data.get(\"hybrid_mode\", \"grounding\")\n        general_threshold = float(data.get(\"general_threshold\", 0.30))\n        character_threshold = float(data.get(\"character_threshold\", 0.75))\n        replace_underscore = bool(data.get(\"replace_underscore\", False))\n        escape_parens = bool(data.get(\"escape_parens\", False))\n        exclude_tags = data.get(\"exclude_tags\", \"\")\n\n        res = await asyncio.to_thread(\n            run_vlm_caption_single,\n            img_path=img_path,\n            model_key=model_key,\n            sys_prompt=sys_prompt,\n            usr_prompt=usr_prompt,\n            max_tokens=max_tokens,\n            temperature=temperature,\n            subject_hint=subject_hint,\n            hybrid_tagger_enabled=hybrid_tagger_enabled,\n            hybrid_mode=hybrid_mode,\n            general_threshold=general_threshold,\n            character_threshold=character_threshold,\n            replace_underscore=replace_underscore,\n            escape_parens=escape_parens,\n            exclude_tags=exclude_tags,\n        )\n        if res.get(\"ok\") and res.get(\"caption\"):\n            await asyncio.to_thread(save_single_caption, img_path, res[\"caption\"])\n        return res\n\n    @app.post(\"/studio_api/tag_one\")\n    async def route_tag_one(req: Request):\n        \"\"\"Run standalone PixAI Tagger v0.9 on a single image and optionally save to .txt.\"\"\"\n        data = await req.json()\n        img_path = data[\"img_path\"]\n        save_to_file = bool(data.get(\"save_to_file\", True))\n        general_threshold = float(data.get(\"general_threshold\", 0.30))\n        character_threshold = float(data.get(\"character_threshold\", 0.75))\n        replace_underscore = bool(data.get(\"replace_underscore\", False))\n        escape_parens = bool(data.get(\"escape_parens\", False))\n        exclude_tags = data.get(\"exclude_tags\", \"\")\n\n        tag_res = await asyncio.to_thread(\n            run_pixai_tagger,\n            img_path=img_path,\n            general_threshold=general_threshold,\n            character_threshold=character_threshold,\n            include_ip=True,\n            replace_underscore=replace_underscore,\n            escape_parens=escape_parens,\n            exclude_tags=exclude_tags,\n        )\n        cap = tag_res.get(\"caption\", \"\")\n        stats = estimate_token_count(cap)\n        if save_to_file and cap:\n            await asyncio.to_thread(save_single_caption, img_path, cap)\n        return {\n            \"ok\": bool(cap),\n            \"caption\": cap,\n            \"words\": stats[\"words\"],\n            \"tokens\": stats[\"tokens\"],\n            \"tagger_meta\": tag_res,\n        }\n\n    @app.get(\"/studio_api/download_zip\")\n    async def route_download_zip(folder: str = DEFAULT_DATASET_DIR):\n        out_zip = await asyncio.to_thread(create_dataset_zip, folder)\n        return FileResponse(str(out_zip), filename=out_zip.name, media_type=\"application/zip\")\n\n    @app.post(\"/studio_api/bulk_edit\")\n    async def route_bulk_edit(req: Request):\n        d = await req.json()\n        res = await asyncio.to_thread(\n            run_bulk_caption_edit,\n            folder_path=d.get(\"folder\", DEFAULT_DATASET_DIR),\n            action_type=d.get(\"action_type\", \"apply\"),\n            trigger=d.get(\"trigger\", \"\"),\n            pos=d.get(\"pos\", \"front\"),\n            find_str=d.get(\"find\", \"\"),\n            replace_str=d.get(\"replace\", \"\"),\n            lowercase=bool(d.get(\"lowercase\", False)),\n            skip_dup=bool(d.get(\"skip_dup\", True)),\n            clean_commas=bool(d.get(\"clean_commas\", True)),\n            backup=bool(d.get(\"backup\", True)),\n            selected_img_paths=d.get(\"selected_img_paths\"),\n        )\n        return res\n\n    @app.post(\"/studio_api/chat\")\n    async def route_chat(req: Request):\n        d = await req.json()\n        res = await asyncio.to_thread(\n            run_multimodal_chat,\n            model_key=d.get(\"model_key\", DEFAULT_MODEL_KEY),\n            sys_prompt=d.get(\"sys_prompt\", \"You are a helpful, uncensored multimodal AI assistant.\"),\n            history=d.get(\"history\", []),\n            user_message=d.get(\"message\", \"\"),\n            image_data_uri=d.get(\"image_data_uri\"),\n            temperature=float(d.get(\"temperature\", 0.7)),\n            max_tokens=int(d.get(\"max_tokens\", 768)),\n            enable_thinking=bool(d.get(\"enable_thinking\", False)),\n        )\n        return res\n\n    @app.post(\"/studio_api/manager\")\n    async def route_manager(req: Request):\n        d = await req.json()\n        res = await asyncio.to_thread(\n            run_smart_file_manager,\n            action=d.get(\"action\", \"Count Statistics\"),\n            source_path=d.get(\"source_path\", DEFAULT_DATASET_DIR),\n            destination_path=d.get(\"destination_path\", \"/content/dataset.zip\"),\n            rename_prefix=d.get(\"rename_prefix\", \"dataset_img\"),\n            active_dataset_dir=d.get(\"active_folder\", DEFAULT_DATASET_DIR),\n        )\n        return res\n",
  "ui/templates/studio.html": "<!DOCTYPE html>\n<html lang=\"en\">\n<head>\n<meta charset=\"UTF-8\">\n<meta name=\"viewport\" content=\"width=device-width, initial-scale=1.0\">\n<title>Vision AI Dataset Labeling, Booru Tagger & Multimodal Chat Studio</title>\n<style>\n  :root {\n    --bg-main: #090b10;\n    --bg-panel: #11151f;\n    --bg-card: #161b28;\n    --bg-input: #0d1017;\n    --border: #22293a;\n    --border-hover: #364156;\n    --accent: #1d9bf0;\n    --accent-glow: rgba(29, 155, 240, 0.22);\n    --purple: #8b5cf6;\n    --text: #e2e8f0;\n    --text-muted: #8b95a7;\n    --success: #10b981;\n    --warning: #f59e0b;\n    --danger: #ef4444;\n  }\n  * { box-sizing: border-box; margin: 0; padding: 0; font-family: 'Inter', -apple-system, BlinkMacSystemFont, sans-serif; }\n  body { background: var(--bg-main); color: var(--text); height: 100vh; display: flex; flex-direction: column; overflow: hidden; }\n\n  /* NAVBAR */\n  .navbar {\n    display: flex; align-items: center; justify-content: space-between;\n    padding: 8px 16px; background: var(--bg-panel); border-bottom: 1px solid var(--border);\n    flex-shrink: 0; gap: 12px; flex-wrap: wrap;\n  }\n  .nav-brand { display: flex; align-items: center; gap: 10px; font-weight: 700; font-size: 14.5px; color: #f8fafc; }\n  .telemetry-strip {\n    display: flex; align-items: center; gap: 8px; font-size: 11.5px;\n    background: var(--bg-input); padding: 4px 10px; border-radius: 8px; border: 1px solid var(--border);\n  }\n  .tel-badge { display: inline-flex; align-items: center; gap: 5px; padding: 2px 7px; border-radius: 5px; background: #161d2b; color: #cbd5e1; }\n  .dot { width: 7px; height: 7px; border-radius: 50%; display: inline-block; }\n  .dot-online { background: var(--success); box-shadow: 0 0 6px var(--success); }\n  .dot-loading { background: var(--warning); box-shadow: 0 0 6px var(--warning); }\n  .dot-standby { background: #64748b; }\n\n  .nav-tabs { display: flex; gap: 4px; background: var(--bg-input); padding: 3px; border-radius: 9px; border: 1px solid var(--border); }\n  .nav-tab {\n    padding: 6px 13px; border-radius: 6px; font-size: 12.5px; font-weight: 600;\n    color: var(--text-muted); cursor: pointer; border: none; background: transparent; transition: 0.15s;\n  }\n  .nav-tab.active { background: var(--bg-card); color: #fff; border: 1px solid var(--border); }\n  .nav-right { display: flex; align-items: center; gap: 8px; }\n\n  /* SUBBAR */\n  .subbar {\n    display: flex; align-items: center; justify-content: space-between;\n    padding: 8px 16px; border-bottom: 1px solid var(--border); background: #0c0f17;\n    flex-shrink: 0; gap: 10px; flex-wrap: wrap;\n  }\n  .subbar-left { display: flex; align-items: center; gap: 8px; flex-wrap: wrap; }\n  .filter-pills { display: flex; gap: 4px; background: var(--bg-panel); padding: 3px; border-radius: 8px; border: 1px solid var(--border); }\n  .f-pill {\n    padding: 4px 10px; border-radius: 6px; font-size: 12px; font-weight: 500;\n    color: var(--text-muted); cursor: pointer; border: none; background: transparent; transition: 0.15s;\n  }\n  .f-pill.active { background: #1e293b; color: #38bdf8; font-weight: 600; }\n  .subbar-right { display: flex; align-items: center; gap: 7px; font-size: 12px; color: var(--text-muted); flex-wrap: wrap; }\n\n  /* BUTTONS & INPUTS */\n  .btn {\n    padding: 6px 11px; border-radius: 7px; font-size: 12px; font-weight: 500;\n    border: 1px solid var(--border); background: var(--bg-card); color: var(--text);\n    cursor: pointer; transition: 0.15s; display: inline-flex; align-items: center; gap: 5px; text-decoration: none;\n  }\n  .btn:hover { border-color: var(--border-hover); background: #1f2637; }\n  .btn-xs { padding: 3px 8px; font-size: 11px; border-radius: 6px; }\n  .btn-primary {\n    background: linear-gradient(180deg, #2596ff 0%, #1d7ed8 100%);\n    border-color: #38bdf8; color: #fff; font-weight: 600;\n  }\n  .btn-primary:hover { filter: brightness(1.08); }\n  .btn-purple {\n    background: linear-gradient(180deg, #8b5cf6 0%, #6d28d9 100%);\n    border-color: #a78bfa; color: #fff; font-weight: 600;\n  }\n  .btn-danger:hover { border-color: var(--danger); color: #fca5a5; background: rgba(239,68,68,0.12); }\n  .btn-block { width: 100%; justify-content: center; padding: 9px; font-size: 13px; border-radius: 8px; }\n\n  .input, .select, .textarea {\n    width: 100%; background: var(--bg-input); border: 1px solid var(--border);\n    color: var(--text); padding: 7px 10px; border-radius: 7px; font-size: 12px; outline: none;\n  }\n  .input:focus, .select:focus, .textarea:focus { border-color: var(--accent); }\n  .field-label { font-size: 11.5px; font-weight: 600; color: #94a3b8; margin-bottom: 4px; display: flex; justify-content: space-between; align-items: center; }\n\n  /* TABS */\n  .tab-view { display: none; flex: 1; overflow: hidden; }\n  .tab-view.active { display: flex; }\n\n  /* TAB 1: LIST PANE & CARDS */\n  .list-pane { flex: 1; display: flex; flex-direction: column; overflow: hidden; padding: 12px 16px; gap: 8px; }\n  .upload-drawer {\n    display: none; background: var(--bg-panel); border: 1.5px dashed #38bdf8;\n    border-radius: 9px; padding: 12px 16px; align-items: center; justify-content: space-between;\n    flex-shrink: 0; gap: 12px; flex-wrap: wrap;\n  }\n  .upload-drawer.open { display: flex; }\n  .rows-scroll { flex: 1; overflow-y: auto; display: flex; flex-direction: column; gap: 8px; padding-right: 4px; }\n\n  .label-row {\n    display: flex; align-items: stretch; gap: 12px;\n    background: var(--bg-panel); border: 1.5px solid var(--border);\n    border-radius: 9px; padding: 9px 12px; transition: 0.15s; position: relative;\n    flex-shrink: 0;\n  }\n  .label-row:hover { border-color: #334155; }\n  .label-row.selected {\n    border-color: var(--accent);\n    box-shadow: 0 0 0 1px var(--accent-glow), inset 0 0 14px rgba(29, 155, 240, 0.04);\n    background: #121826;\n  }\n  .row-check-wrap { display: flex; align-items: flex-start; padding-top: 3px; }\n  .row-check { width: 17px; height: 17px; accent-color: var(--accent); cursor: pointer; }\n  .row-thumb-wrap { position: relative; width: 88px; height: 88px; flex-shrink: 0; }\n  .row-thumb {\n    width: 88px; height: 88px; border-radius: 7px; object-fit: cover;\n    background: #080a0f; border: 1px solid var(--border); cursor: zoom-in; display: block;\n  }\n  .res-badge {\n    position: absolute; bottom: 3px; left: 3px; right: 3px;\n    background: rgba(9, 11, 16, 0.82); color: #cbd5e1; font-size: 9.5px;\n    text-align: center; padding: 1px 3px; border-radius: 4px; pointer-events: none;\n  }\n  .res-badge.low-res { background: rgba(239, 68, 68, 0.88); color: #fff; font-weight: 600; }\n\n  .row-body { flex: 1; display: flex; flex-direction: column; gap: 5px; min-width: 0; }\n  .row-meta { display: flex; align-items: center; justify-content: space-between; font-size: 11.5px; color: var(--text-muted); gap: 8px; flex-wrap: wrap; }\n  .row-actions { display: flex; align-items: center; gap: 5px; flex-wrap: wrap; }\n  .tok-badge { padding: 2px 7px; border-radius: 5px; font-size: 11px; font-weight: 600; background: #1e293b; color: #94a3b8; }\n  .tok-ok { background: rgba(16, 185, 129, 0.14); color: #34d399; }\n  .tok-warn { background: rgba(245, 158, 11, 0.16); color: #fbbf24; }\n  .tok-over { background: rgba(239, 68, 68, 0.18); color: #f87171; }\n\n  .row-textarea {\n    flex: 1; width: 100%; background: transparent; border: none; color: var(--text);\n    font-size: 12.8px; line-height: 1.48; resize: none; outline: none; min-height: 62px;\n  }\n  .row-textarea::placeholder { color: #475569; font-style: italic; }\n\n  .pagination-bar {\n    display: flex; align-items: center; justify-content: space-between;\n    padding: 7px 12px; background: var(--bg-panel); border: 1px solid var(--border);\n    border-radius: 8px; font-size: 12px; color: var(--text-muted); flex-shrink: 0;\n  }\n\n  /* SIDEBAR */\n  .sidebar {\n    width: 390px; background: var(--bg-panel); border-left: 1px solid var(--border);\n    padding: 13px 15px; display: flex; flex-direction: column; gap: 10px; overflow-y: auto; flex-shrink: 0;\n  }\n  .model-desc { font-size: 11px; color: #64748b; line-height: 1.38; margin-top: 3px; }\n  .drawer-box {\n    display: none; background: var(--bg-input); border: 1px solid var(--border);\n    border-radius: 8px; padding: 10px; flex-direction: column; gap: 7px;\n  }\n  .drawer-box.open { display: flex; }\n  .grid-2 { display: grid; grid-template-columns: 1fr 1fr; gap: 7px; }\n  .grid-3 { display: grid; grid-template-columns: 1fr 1fr 1fr; gap: 6px; }\n\n  /* CHAT TAB */\n  .chat-main { flex: 1; display: flex; flex-direction: column; overflow: hidden; padding: 14px 20px; gap: 10px; }\n  .chat-messages { flex: 1; overflow-y: auto; display: flex; flex-direction: column; gap: 10px; padding-right: 6px; }\n  .msg { max-width: 84%; padding: 11px 15px; border-radius: 11px; font-size: 13px; line-height: 1.55; white-space: pre-wrap; position: relative; }\n  .msg-user { align-self: flex-end; background: #1e3a8a; color: #f8fafc; border: 1px solid #2563eb; }\n  .msg-ai { align-self: flex-start; background: var(--bg-panel); color: var(--text); border: 1px solid var(--border); }\n  .msg-toolbar { display: flex; justify-content: flex-end; gap: 6px; margin-top: 7px; padding-top: 5px; border-top: 1px solid rgba(255,255,255,0.06); }\n  .chat-bar { display: flex; gap: 8px; align-items: flex-end; background: var(--bg-panel); padding: 9px; border-radius: 10px; border: 1px solid var(--border); }\n\n  /* LIGHTBOX SPLIT EDITOR */\n  .lightbox {\n    display: none; position: fixed; inset: 0; background: rgba(5, 7, 12, 0.92);\n    z-index: 9999; align-items: center; justify-content: center; padding: 24px;\n  }\n  .lightbox.open { display: flex; }\n  .lightbox-card {\n    background: var(--bg-panel); border: 1px solid var(--border); border-radius: 12px;\n    width: 94vw; max-width: 1280px; height: 86vh; display: flex; overflow: hidden; box-shadow: 0 20px 50px rgba(0,0,0,0.7);\n  }\n  .lb-img-pane {\n    flex: 1.3; background: #05070b; display: flex; align-items: center; justify-content: center;\n    position: relative; padding: 14px; overflow: hidden;\n  }\n  .lb-img-pane img { max-width: 100%; max-height: 100%; object-fit: contain; border-radius: 6px; }\n  .lb-edit-pane {\n    width: 440px; border-left: 1px solid var(--border); padding: 16px;\n    display: flex; flex-direction: column; gap: 10px; overflow-y: auto;\n  }\n\n  /* MANAGER TABLE */\n  .inv-table { width: 100%; border-collapse: collapse; font-size: 12px; }\n  .inv-table th, .inv-table td { padding: 8px 10px; border-bottom: 1px solid var(--border); text-align: left; }\n  .inv-table th { color: #94a3b8; font-weight: 600; background: var(--bg-input); }\n</style>\n</head>\n<body>\n\n<!-- ==================== TOP NAVBAR & LIVE TELEMETRY ==================== -->\n<div class=\"navbar\">\n  <div class=\"nav-brand\">\n    <span>⚡ Vision AI Labeling, PixAI Booru & Chat Studio</span>\n  </div>\n\n  <div class=\"telemetry-strip\">\n    <span class=\"tel-badge\" title=\"Status Mesin VLM (llama-server)\">\n      <span class=\"dot dot-standby\" id=\"telVlmDot\"></span>\n      <span id=\"telVlmText\">VLM: Standby</span>\n    </span>\n    <span class=\"tel-badge\" title=\"Status Mesin PixAI Tagger v0.9\">\n      <span class=\"dot dot-standby\" id=\"telTaggerDot\"></span>\n      <span id=\"telTaggerText\">PixAI: Standby</span>\n    </span>\n    <span class=\"tel-badge\" id=\"telVramBadge\">VRAM: 0.0 / 15.0 GB</span>\n    <span class=\"tel-badge\" id=\"telDiskBadge\">Disk Free: -- GB</span>\n  </div>\n\n  <div class=\"nav-tabs\">\n    <button type=\"button\" class=\"nav-tab active\" id=\"tabBtn-labeling\" onclick=\"switchTab('labeling', this)\">🏷️ Labeling & Tagging</button>\n    <button type=\"button\" class=\"nav-tab\" id=\"tabBtn-chat\" onclick=\"switchTab('chat', this)\">💬 Multimodal AI Chat</button>\n    <button type=\"button\" class=\"nav-tab\" id=\"tabBtn-manager\" onclick=\"switchTab('manager', this)\">🗂️ File, Drive & VRAM Manager</button>\n  </div>\n\n  <div class=\"nav-right\">\n    <button type=\"button\" class=\"btn\" onclick=\"toggleUploadDrawer()\">📤 Upload Images / .ZIP</button>\n    <a class=\"btn btn-primary\" id=\"downloadZipLink\" href=\"./studio_api/download_zip\" target=\"_blank\">📥 Download (.zip)</a>\n  </div>\n</div>\n\n<!-- ==================== TAB 1: LABELING & BOORU TAGGING ==================== -->\n<div id=\"tab-labeling\" class=\"tab-view active\" style=\"flex-direction: column;\">\n  <div class=\"subbar\">\n    <div class=\"subbar-left\">\n      <div class=\"filter-pills\">\n        <button type=\"button\" class=\"f-pill active\" id=\"pill-all\" onclick=\"setFilter('all')\">All <b id=\"cnt-all\">0</b></button>\n        <button type=\"button\" class=\"f-pill\" id=\"pill-cap\" onclick=\"setFilter('captioned')\">Captioned <b id=\"cnt-cap\">0</b></button>\n        <button type=\"button\" class=\"f-pill\" id=\"pill-uncap\" onclick=\"setFilter('uncaptioned')\">Uncaptioned <b id=\"cnt-uncap\">0</b></button>\n        <button type=\"button\" class=\"f-pill\" id=\"pill-short\" onclick=\"setFilter('short')\" title=\"Caption kurang dari 15 kata\">Short &lt;15w <b id=\"cnt-short\">0</b></button>\n        <button type=\"button\" class=\"f-pill\" id=\"pill-lowres\" onclick=\"setFilter('lowres')\" title=\"Resolusi sisi terpendek di bawah 512px\">Low-Res <b id=\"cnt-lowres\">0</b></button>\n      </div>\n\n      <input type=\"text\" id=\"searchQuery\" class=\"input\" style=\"width:185px; padding:5px 9px;\" placeholder=\"🔍 Cari nama file / isi caption...\" oninput=\"onSearchInput()\">\n\n      <div style=\"display:flex; align-items:center; gap:4px;\">\n        <input type=\"text\" id=\"activeFolderInput\" class=\"input\" style=\"width:165px; padding:5px 8px;\" value=\"/content/dataset\" title=\"Folder Dataset Aktif\">\n        <button type=\"button\" class=\"btn btn-xs\" onclick=\"changeActiveFolder()\">📂 Buka</button>\n      </div>\n    </div>\n\n    <div class=\"subbar-right\">\n      <span id=\"statusBanner\" style=\"color:#38bdf8; font-weight:600;\">Ready</span>\n      <span>·</span>\n      <span id=\"selCounter\">0 selected</span>\n      <button type=\"button\" class=\"btn btn-xs\" onclick=\"selectPageRows(true)\" title=\"Pilih semua item di halaman ini\">Select Page</button>\n      <button type=\"button\" class=\"btn btn-xs\" onclick=\"selectAllFilteredRows(true)\" title=\"Pilih seluruh item di filter saat ini\">Select All</button>\n      <button type=\"button\" class=\"btn btn-xs\" onclick=\"selectAllFilteredRows(false)\">Deselect</button>\n      <button type=\"button\" class=\"btn btn-xs\" onclick=\"loadDataset()\">🔄 Refresh</button>\n    </div>\n  </div>\n\n  <div style=\"display:flex; flex:1; overflow:hidden;\">\n    <div class=\"list-pane\">\n      <div class=\"upload-drawer\" id=\"uploadDrawer\">\n        <div>\n          <b style=\"color:#38bdf8; font-size:12.5px;\">📤 Upload Gambar (.png, .jpg, .webp) atau Arsip .ZIP Dataset (Otomatis Anti-__MACOSX & Verifikasi Korup)</b>\n          <p style=\"font-size:11.5px; color:var(--text-muted); margin-top:2px;\">Mendukung ekstraksi sub-folder otomatis beserta pasangan file .txt-nya.</p>\n        </div>\n        <div style=\"display:flex; gap:7px; align-items:center;\">\n          <input type=\"file\" id=\"fileUploadInput\" multiple accept=\".png,.jpg,.jpeg,.webp,.bmp,.txt,.zip\" class=\"input\" style=\"width:250px;\">\n          <button type=\"button\" class=\"btn btn-primary\" onclick=\"uploadFilesNow()\">⬆️ Upload Sekarang</button>\n          <button type=\"button\" class=\"btn\" onclick=\"toggleUploadDrawer()\">✕</button>\n        </div>\n      </div>\n\n      <div class=\"rows-scroll\" id=\"rowsContainer\"></div>\n\n      <!-- PAGINATION BAR -->\n      <div class=\"pagination-bar\">\n        <div style=\"display:flex; align-items:center; gap:8px;\">\n          <span>Baris per halaman:</span>\n          <select id=\"pageSizeSelect\" class=\"select\" style=\"width:70px; padding:3px 6px;\" onchange=\"onPageSizeChange()\">\n            <option value=\"30\" selected>30</option>\n            <option value=\"50\">50</option>\n            <option value=\"100\">100</option>\n          </select>\n          <span id=\"pageRangeInfo\">Menampilkan 0–0 dari 0</span>\n        </div>\n        <div style=\"display:flex; align-items:center; gap:6px;\">\n          <span style=\"font-size:11px; color:#64748b;\">Pintasan: <b>Ctrl+Enter</b> = Generate 1 · <b>Alt+↑/↓</b> = Pindah Baris</span>\n          <button type=\"button\" class=\"btn btn-xs\" id=\"btnPrevPage\" onclick=\"changePage(-1)\">◀ Prev</button>\n          <span id=\"pageIndicator\" style=\"font-weight:600; color:#e2e8f0; min-width:85px; text-align:center;\">Page 1 / 1</span>\n          <button type=\"button\" class=\"btn btn-xs\" id=\"btnNextPage\" onclick=\"changePage(1)\">Next ▶</button>\n        </div>\n      </div>\n    </div>\n\n    <!-- RIGHT SIDEBAR -->\n    <div class=\"sidebar\">\n      <div class=\"grid-2\">\n        <button type=\"button\" class=\"btn btn-primary btn-block\" id=\"btnGenerateBatch\" onclick=\"generateSelectedCaptions()\">\n          ✨ Generate (Selected)\n        </button>\n        <button type=\"button\" class=\"btn btn-purple btn-block\" id=\"btnTagBatch\" onclick=\"tagSelectedWithPixAI()\" title=\"Tag cepat gambar terpilih dengan PixAI Tagger v0.9\">\n          🏷️ PixAI Tag (Selected)\n        </button>\n      </div>\n\n      <button type=\"button\" class=\"btn btn-block\" onclick=\"toggleDrawer('editLabelsDrawer')\">\n        🔧 Trigger Word & Bulk Find/Replace\n      </button>\n\n      <div class=\"drawer-box\" id=\"editLabelsDrawer\">\n        <div>\n          <label class=\"field-label\">Trigger Word</label>\n          <input type=\"text\" id=\"trigWord\" class=\"input\" placeholder=\"e.g. ohwx_woman, makima_(chainsaw_man)\">\n        </div>\n        <div class=\"grid-2\">\n          <div>\n            <label class=\"field-label\">Posisi Trigger</label>\n            <select id=\"trigPos\" class=\"select\">\n              <option value=\"front\" selected>Depan (Front)</option>\n              <option value=\"back\">Belakang (Back)</option>\n            </select>\n          </div>\n          <div>\n            <label class=\"field-label\">Target File</label>\n            <select id=\"bulkTargetScope\" class=\"select\">\n              <option value=\"all\" selected>Semua File (.txt)</option>\n              <option value=\"selected\">Hanya yang Dicentang</option>\n            </select>\n          </div>\n        </div>\n        <div style=\"display:flex; gap:12px; font-size:11.5px;\">\n          <label><input type=\"checkbox\" id=\"trigLower\"> Lowercase</label>\n          <label><input type=\"checkbox\" id=\"trigSkip\" checked> Skip Duplicate</label>\n        </div>\n        <div class=\"grid-2\">\n          <input type=\"text\" id=\"findStr\" class=\"input\" placeholder=\"Find word/tag...\">\n          <input type=\"text\" id=\"repStr\" class=\"input\" placeholder=\"Replace with...\">\n        </div>\n        <div class=\"grid-2\">\n          <button type=\"button\" class=\"btn btn-primary\" onclick=\"runBulkEdit('apply')\">✓ Terapkan</button>\n          <button type=\"button\" class=\"btn\" onclick=\"runBulkEdit('restore')\">♻️ Restore Backup</button>\n        </div>\n      </div>\n\n      <!-- MODEL SELECTOR + VRAM CONTROLS -->\n      <div>\n        <div class=\"field-label\">\n          <span>Model Utama (7 VLM GGUF + PixAI Tagger v0.9)</span>\n          <span id=\"modelDiskBadge\" style=\"font-size:10.5px; color:#38bdf8;\"></span>\n        </div>\n        <select id=\"modelSelect\" class=\"select\" onchange=\"onModelChange()\"></select>\n        <div class=\"model-desc\" id=\"modelDesc\"></div>\n        <div class=\"grid-2\" style=\"margin-top:6px;\">\n          <button type=\"button\" class=\"btn btn-xs\" style=\"justify-content:center;\" onclick=\"loadSelectedModelNow()\">⚡ Load Model ke VRAM</button>\n          <button type=\"button\" class=\"btn btn-xs btn-danger\" style=\"justify-content:center;\" onclick=\"unloadVramNow('all')\">🧹 Unload VRAM</button>\n        </div>\n      </div>\n\n      <!-- HYBRID PIXAI TAGGER V0.9 ASSISTANT -->\n      <div style=\"background:var(--bg-input); border:1px solid var(--border); border-radius:8px; padding:9px;\">\n        <div style=\"display:flex; align-items:center; justify-content:space-between; gap:6px;\">\n          <label style=\"font-size:11.8px; font-weight:600; color:#c4b5fd; display:flex; align-items:center; gap:6px; cursor:pointer;\">\n            <input type=\"checkbox\" id=\"hybridTaggerChk\" onchange=\"toggleHybridDrawer()\">\n            🔗 Asisten Hybrid PixAI Tagger v0.9\n          </label>\n          <button type=\"button\" class=\"btn btn-xs\" onclick=\"toggleDrawer('pixaiSettingsBox')\">⚙️ Threshold</button>\n        </div>\n        <div class=\"drawer-box\" id=\"pixaiSettingsBox\" style=\"margin-top:7px; padding:8px;\">\n          <div>\n            <label class=\"field-label\">Mode Integrasi saat VLM Aktif</label>\n            <select id=\"hybridModeSelect\" class=\"select\">\n              <option value=\"grounding\" selected>Grounding Prompt (Bantu VLM Kenali Karakter/IP)</option>\n              <option value=\"prepend\">Anima Hybrid: Tag PixAI di Depan + Paragraf VLM</option>\n              <option value=\"append\">Paragraf VLM + Tag PixAI di Belakang</option>\n            </select>\n          </div>\n          <div class=\"grid-2\">\n            <div>\n              <label class=\"field-label\">General Thresh</label>\n              <input type=\"number\" id=\"pixaiGenTh\" class=\"input\" value=\"0.30\" step=\"0.05\" min=\"0.05\" max=\"0.95\">\n            </div>\n            <div>\n              <label class=\"field-label\">Character Thresh</label>\n              <input type=\"number\" id=\"pixaiCharTh\" class=\"input\" value=\"0.75\" step=\"0.05\" min=\"0.10\" max=\"0.99\">\n            </div>\n          </div>\n          <div style=\"display:flex; gap:10px; font-size:11px; color:var(--text-muted);\">\n            <label><input type=\"checkbox\" id=\"pixaiNoUnderscore\"> Hapus Underscore (_)</label>\n            <label><input type=\"checkbox\" id=\"pixaiEscapeParen\"> Escape \\( \\)</label>\n          </div>\n          <div>\n            <input type=\"text\" id=\"pixaiExcludeTags\" class=\"input\" placeholder=\"Exclude tags (e.g. signature, watermark)\">\n          </div>\n        </div>\n      </div>\n\n      <!-- GLOBAL SUBJECT / CONCEPT HINT -->\n      <div>\n        <label class=\"field-label\">\n          <span>Global Subject / Concept Hint (Opsional)</span>\n        </label>\n        <input type=\"text\" id=\"subjectHintInput\" class=\"input\" placeholder=\"Misal: karakter bernama Maya memakai jaket merah...\">\n      </div>\n\n      <!-- STYLE PRESET & SYSTEM PROMPT -->\n      <div>\n        <label class=\"field-label\">Style Preference</label>\n        <select id=\"styleSelect\" class=\"select\" onchange=\"onStyleChange()\"></select>\n      </div>\n\n      <div style=\"flex:1; display:flex; flex-direction:column; min-height:110px;\">\n        <label class=\"field-label\">Caption System Prompt</label>\n        <textarea id=\"captionPrompt\" class=\"textarea\" style=\"flex:1; min-height:105px; resize:vertical; line-height:1.42;\"></textarea>\n      </div>\n\n      <div class=\"grid-2\">\n        <div>\n          <label class=\"field-label\">Max Tokens Target</label>\n          <input type=\"number\" id=\"maxTokens\" class=\"input\" value=\"220\" min=\"30\" max=\"1024\" onchange=\"renderRows()\">\n        </div>\n        <div>\n          <label class=\"field-label\">Temperature</label>\n          <input type=\"number\" id=\"temperature\" class=\"input\" value=\"0.2\" step=\"0.05\" min=\"0\" max=\"1.5\">\n        </div>\n      </div>\n\n      <label style=\"font-size:11.5px; color:var(--text-muted); display:flex; align-items:center; gap:6px; cursor:pointer;\">\n        <input type=\"checkbox\" id=\"overwriteChk\" checked> Timpa gambar yang sudah punya caption saat Batch Generate\n      </label>\n    </div>\n  </div>\n</div>\n\n<!-- ==================== TAB 2: MULTIMODAL CHAT ==================== -->\n<div id=\"tab-chat\" class=\"tab-view\">\n  <div class=\"chat-main\">\n    <div class=\"chat-messages\" id=\"chatMessages\">\n      <div class=\"msg msg-ai\">Halo! Saya siap membantu Anda merancang prompt T2I/I2V (MOSS-Image, Qwen 2.1, Flux, Chroma1-HD, Anima, Krea 2, Wan 2.1), menganalisis gambar (reverse-prompting/OCR), atau berdiskusi bebas tanpa sensor. Anda juga bisa menekan tombol <b>📌 Ke Chat</b> pada gambar mana pun di Tab 1!</div>\n    </div>\n    <div id=\"chatImgPreviewBar\" style=\"display:none; align-items:center; gap:10px; background:var(--bg-panel); padding:6px 12px; border-radius:8px; border:1px solid var(--border);\">\n      <img id=\"chatImgPreview\" src=\"\" style=\"height:44px; border-radius:5px;\">\n      <span id=\"chatImgName\" style=\"font-size:12px; color:var(--text-muted); flex:1;\"></span>\n      <button type=\"button\" class=\"btn btn-xs\" onclick=\"clearChatImage()\">✕ Hapus Lampiran</button>\n    </div>\n    <div class=\"chat-bar\">\n      <label class=\"btn\" style=\"height:38px; cursor:pointer;\">\n        🖼️ Gambar\n        <input type=\"file\" id=\"chatImgInput\" accept=\"image/*\" style=\"display:none;\" onchange=\"onChatImagePicked(event)\">\n      </label>\n      <textarea id=\"chatInput\" class=\"textarea\" rows=\"2\" placeholder=\"Ketik pertanyaan atau instruksi tentang gambar... ( Gambar tetap diingat pada pertanyaan lanjutan! )\" onkeydown=\"if(event.key==='Enter' && !event.shiftKey){event.preventDefault(); sendChatMessage();}\"></textarea>\n      <button type=\"button\" class=\"btn btn-primary\" style=\"height:38px; padding:0 16px;\" onclick=\"sendChatMessage()\">🚀 Kirim</button>\n    </div>\n  </div>\n  <div class=\"sidebar\">\n    <label class=\"field-label\">System Prompt Chat</label>\n    <textarea id=\"chatSysPrompt\" class=\"textarea\" rows=\"5\">You are an uncensored, helpful multimodal AI assistant and expert prompt engineer for diffusion and video models (Flux, Qwen-Image 2.1, MOSS, Chroma1-HD, Anima, Krea 2, Wan 2.1).</textarea>\n    <div class=\"grid-2\">\n      <div>\n        <label class=\"field-label\">Temperature</label>\n        <input type=\"number\" id=\"chatTemp\" class=\"input\" value=\"0.7\" step=\"0.05\">\n      </div>\n      <div>\n        <label class=\"field-label\">Max Tokens</label>\n        <input type=\"number\" id=\"chatMaxTok\" class=\"input\" value=\"768\" step=\"64\">\n      </div>\n    </div>\n    <label style=\"font-size:12px; color:#38bdf8; display:flex; align-items:center; gap:6px; cursor:pointer; margin-top:4px;\">\n      <input type=\"checkbox\" id=\"chatThinkingChk\"> 🧠 Aktifkan Thinking Mode (Khusus Qwen3.5)\n    </label>\n    <p style=\"font-size:11px; color:var(--text-muted); line-height:1.4;\">\n      💡 <b>Persistent Vision Memory:</b> Gambar yang Anda unggah di chat otomatis disimpan di histori percakapan (maks 2 gambar terakhir) sehingga Anda bisa terus bertanya tanpa perlu mengunggah ulang gambar!\n    </p>\n    <button type=\"button\" class=\"btn\" onclick=\"clearChatHistory()\">🗑️ Bersihkan Riwayat Chat</button>\n  </div>\n</div>\n\n<!-- ==================== TAB 3: SMART FILE, DRIVE & VRAM MANAGER ==================== -->\n<div id=\"tab-manager\" class=\"tab-view\" style=\"padding:16px; gap:16px; overflow-y:auto;\">\n  <!-- LEFT: SMART FILE MANAGER + GOOGLE DRIVE SYNC -->\n  <div style=\"width:410px; background:var(--bg-panel); border:1px solid var(--border); border-radius:11px; padding:15px; display:flex; flex-direction:column; gap:10px; flex-shrink:0;\">\n    <h3 style=\"font-size:14px;\">🛠️ Smart File Manager & Google Drive Backup</h3>\n    <div>\n      <label class=\"field-label\">Pilih Aksi Operasi</label>\n      <select id=\"mgrAction\" class=\"select\" onchange=\"onManagerActionChange()\">\n        <option>Count Statistics</option>\n        <option>Batch Rename</option>\n        <option>Sync Images & Captions</option>\n        <option>Clean Empty Files</option>\n        <option>Zip Folder</option>\n        <option>Unzip File</option>\n        <option>Export to Google Drive</option>\n        <option>Import from Google Drive</option>\n        <option>Copy</option>\n        <option>Move</option>\n        <option>Delete</option>\n      </select>\n    </div>\n    <div>\n      <label class=\"field-label\">Source Path</label>\n      <input type=\"text\" id=\"mgrSrc\" class=\"input\" value=\"/content/dataset\">\n    </div>\n    <div>\n      <label class=\"field-label\">Destination Path</label>\n      <input type=\"text\" id=\"mgrDst\" class=\"input\" value=\"/content/dataset.zip\">\n    </div>\n    <div>\n      <label class=\"field-label\">Rename Prefix (Khusus Batch Rename)</label>\n      <input type=\"text\" id=\"mgrPrefix\" class=\"input\" value=\"dataset_img\">\n    </div>\n    <button type=\"button\" class=\"btn btn-primary btn-block\" onclick=\"runSmartManager()\">▶️ Jalankan Aksi</button>\n\n    <div style=\"border-top:1px solid var(--border); padding-top:10px; margin-top:4px;\">\n      <label class=\"field-label\">Pintasan Cepat Google Drive (/content/drive/MyDrive)</label>\n      <div class=\"grid-2\">\n        <button type=\"button\" class=\"btn btn-xs\" onclick=\"quickDrivePreset('export')\">☁️ Backup ke Drive</button>\n        <button type=\"button\" class=\"btn btn-xs\" onclick=\"quickDrivePreset('import')\">📥 Ambil dari Drive</button>\n      </div>\n    </div>\n\n    <div style=\"flex:1; display:flex; flex-direction:column; min-height:140px;\">\n      <label class=\"field-label\">Log Eksekusi File Manager</label>\n      <pre id=\"mgrLog\" style=\"flex:1; background:var(--bg-input); border:1px solid var(--border); border-radius:7px; padding:10px; font-family:monospace; font-size:11.5px; color:#38bdf8; overflow:auto; white-space:pre-wrap;\">Pilih aksi di atas lalu klik Jalankan Aksi.</pre>\n    </div>\n  </div>\n\n  <!-- RIGHT: VLM & PIXAI TAGGER DISK CACHE & VRAM MANAGER -->\n  <div style=\"flex:1; background:var(--bg-panel); border:1px solid var(--border); border-radius:11px; padding:15px; display:flex; flex-direction:column; gap:10px; overflow:hidden;\">\n    <div style=\"display:flex; align-items:center; justify-content:space-between; flex-wrap:wrap; gap:8px;\">\n      <div>\n        <h3 style=\"font-size:14px;\">💾 VLM GGUF & PixAI Tagger Disk Cache / VRAM Manager</h3>\n        <p style=\"font-size:11.5px; color:var(--text-muted);\">Kelola model yang aktif di VRAM Tesla T4 atau hapus file GGUF/PTH lama untuk menghemat ruang disk Colab.</p>\n      </div>\n      <div style=\"display:flex; gap:6px;\">\n        <button type=\"button\" class=\"btn btn-xs\" onclick=\"refreshTelemetry()\">🔄 Refresh Status</button>\n        <button type=\"button\" class=\"btn btn-xs btn-danger\" onclick=\"unloadVramNow('vlm')\">🧹 Unload VLM</button>\n        <button type=\"button\" class=\"btn btn-xs btn-danger\" onclick=\"unloadVramNow('tagger')\">🧹 Unload PixAI Tagger</button>\n      </div>\n    </div>\n\n    <div style=\"flex:1; overflow-y:auto; border:1px solid var(--border); border-radius:8px;\">\n      <table class=\"inv-table\">\n        <thead>\n          <tr>\n            <th>Model Name</th>\n            <th>Engine</th>\n            <th>Status Disk</th>\n            <th>Ukuran</th>\n            <th>Status VRAM</th>\n            <th>Aksi</th>\n          </tr>\n        </thead>\n        <tbody id=\"modelsInventoryBody\"></tbody>\n      </table>\n    </div>\n  </div>\n</div>\n\n<!-- ==================== SPLIT-VIEW LIGHTBOX EDITOR MODAL ==================== -->\n<div class=\"lightbox\" id=\"lightbox\" onclick=\"if(event.target===this) closeLightbox()\">\n  <div class=\"lightbox-card\">\n    <div class=\"lb-img-pane\">\n      <img id=\"lightboxImg\" src=\"\" alt=\"Preview\">\n      <div style=\"position:absolute; bottom:12px; left:12px; display:flex; gap:6px;\">\n        <button type=\"button\" class=\"btn btn-xs\" onclick=\"stepLightbox(-1)\">◀ Gambar Sebelumnya</button>\n        <button type=\"button\" class=\"btn btn-xs\" onclick=\"stepLightbox(1)\">Gambar Berikutnya ▶</button>\n      </div>\n    </div>\n    <div class=\"lb-edit-pane\">\n      <div style=\"display:flex; justify-content:space-between; align-items:center;\">\n        <b id=\"lbFileName\" style=\"font-size:13.5px; color:#38bdf8;\">image.png</b>\n        <button type=\"button\" class=\"btn btn-xs\" onclick=\"closeLightbox()\">✕ Tutup (Esc)</button>\n      </div>\n      <div id=\"lbMetaInfo\" style=\"font-size:11.5px; color:var(--text-muted);\"></div>\n      <div class=\"grid-3\">\n        <button type=\"button\" class=\"btn btn-primary btn-xs\" style=\"justify-content:center;\" id=\"lbBtnGen\" onclick=\"lightboxGenerateOne()\">⚡ Generate VLM</button>\n        <button type=\"button\" class=\"btn btn-purple btn-xs\" style=\"justify-content:center;\" id=\"lbBtnTag\" onclick=\"lightboxTagOne()\">🏷️ PixAI Tag</button>\n        <button type=\"button\" class=\"btn btn-xs\" style=\"justify-content:center;\" onclick=\"lightboxSendToChat()\">📌 Ke Chat</button>\n      </div>\n      <label class=\"field-label\">Edit Caption Langsung (Auto-Save)</label>\n      <textarea id=\"lbTextarea\" class=\"textarea\" style=\"flex:1; min-height:180px; font-size:13px; line-height:1.5; resize:none;\"></textarea>\n      <div id=\"lbPixaiInspector\" style=\"display:none; background:var(--bg-input); border:1px solid var(--border); border-radius:7px; padding:8px; font-size:11px; max-height:150px; overflow-y:auto;\"></div>\n    </div>\n  </div>\n</div>\n\n<script>\nconst BOOT_DATA = __INITIAL_BOOT_JSON__;\n\nlet STATE = {\n  folder: BOOT_DATA.folder || \"/content/dataset\",\n  items: [],\n  selected: new Set(),\n  filter: \"all\",\n  search: \"\",\n  page: 1,\n  pageSize: 30,\n  models: BOOT_DATA.models || [],\n  styles: BOOT_DATA.styles || {},\n  pixaiKey: BOOT_DATA.pixai_tagger_key || \"PixAI Tagger v0.9 [Fast Anime & Character/IP Booru Tagger]\",\n  isGenerating: false,\n  stopRequested: false,\n  chatHistory: [],\n  chatImageDataUri: null,\n  lightboxImgPath: null,\n  telemetry: null\n};\n\nfunction switchTab(name, btnEl) {\n  document.querySelectorAll(\".tab-view\").forEach(el => el.classList.remove(\"active\"));\n  document.querySelectorAll(\".nav-tab\").forEach(el => el.classList.remove(\"active\"));\n  const target = document.getElementById(\"tab-\" + name);\n  if (target) target.classList.add(\"active\");\n  if (btnEl) btnEl.classList.add(\"active\");\n  else {\n    const b = document.getElementById(\"tabBtn-\" + name);\n    if (b) b.classList.add(\"active\");\n  }\n}\n\nfunction updateDownloadLink() {\n  const link = document.getElementById(\"downloadZipLink\");\n  if (link) {\n    link.href = \"./studio_api/download_zip?folder=\" + encodeURIComponent(STATE.folder);\n  }\n}\n\nfunction initSelectors() {\n  document.getElementById(\"activeFolderInput\").value = STATE.folder;\n  updateDownloadLink();\n\n  const mSel = document.getElementById(\"modelSelect\");\n  if (mSel && !mSel.options.length) {\n    STATE.models.forEach(m => {\n      const opt = document.createElement(\"option\");\n      opt.value = m.key;\n      opt.textContent = m.key;\n      if (m.key === BOOT_DATA.default_model) opt.selected = true;\n      mSel.appendChild(opt);\n    });\n    onModelChange();\n  }\n\n  const sSel = document.getElementById(\"styleSelect\");\n  if (sSel && !sSel.options.length) {\n    Object.keys(STATE.styles).forEach(k => {\n      const opt = document.createElement(\"option\");\n      opt.value = k;\n      opt.textContent = k;\n      sSel.appendChild(opt);\n    });\n    onStyleChange();\n  }\n}\n\nfunction onModelChange() {\n  const val = document.getElementById(\"modelSelect\").value;\n  const found = STATE.models.find(x => x.key === val);\n  document.getElementById(\"modelDesc\").textContent = found ? found.desc : \"\";\n  if (found && found.sampling && found.sampling.temperature !== undefined) {\n    document.getElementById(\"temperature\").value = found.sampling.temperature;\n  }\n  if (found && found.engine === \"pixai_tagger\") {\n    document.getElementById(\"pixaiSettingsBox\").classList.add(\"open\");\n  }\n  updateModelDiskBadge();\n}\n\nfunction updateModelDiskBadge() {\n  const badge = document.getElementById(\"modelDiskBadge\");\n  if (!badge || !STATE.telemetry || !STATE.telemetry.models_inventory) return;\n  const val = document.getElementById(\"modelSelect\").value;\n  const item = STATE.telemetry.models_inventory.find(m => m.key === val);\n  if (!item) {\n    badge.textContent = \"\";\n    return;\n  }\n  if (item.active_in_vram) {\n    badge.textContent = \"🟢 Aktif di VRAM\";\n    badge.style.color = \"#10b981\";\n  } else if (item.downloaded) {\n    badge.textContent = \"✅ Tersedia di Disk (\" + item.size_gb + \" GB)\";\n    badge.style.color = \"#38bdf8\";\n  } else {\n    badge.textContent = \"☁️ Perlu Diunduh (~\" + item.size_gb + \" GB)\";\n    badge.style.color = \"#f59e0b\";\n  }\n}\n\nfunction onStyleChange() {\n  const val = document.getElementById(\"styleSelect\").value;\n  if (STATE.styles[val]) {\n    document.getElementById(\"captionPrompt\").value = STATE.styles[val];\n  }\n}\n\nfunction toggleHybridDrawer() {\n  const chk = document.getElementById(\"hybridTaggerChk\");\n  if (chk && chk.checked) {\n    document.getElementById(\"pixaiSettingsBox\").classList.add(\"open\");\n  }\n}\n\nfunction countWordsAndTokens(str) {\n  const s = (str || \"\").trim();\n  if (!s) return {words: 0, tokens: 0};\n  const words = s.split(/\\s+/).filter(Boolean).length;\n  const commas = (s.match(/,/g) || []).length;\n  const tokens = Math.max(words, Math.round(words * 1.28 + commas * 0.35));\n  return {words, tokens};\n}\n\nfunction formatTokenBadgeClass(tokens) {\n  const maxTok = parseInt(document.getElementById(\"maxTokens\").value || \"220\", 10);\n  if (!tokens) return \"tok-badge\";\n  if (tokens > maxTok) return \"tok-badge tok-over\";\n  if (tokens >= Math.round(maxTok * 0.85)) return \"tok-badge tok-warn\";\n  return \"tok-badge tok-ok\";\n}\n\nasync function loadDataset() {\n  initSelectors();\n  try {\n    const res = await fetch(\"./studio_api/init?folder=\" + encodeURIComponent(STATE.folder));\n    const data = await res.json();\n    STATE.items = data.items || [];\n    if (data.telemetry) applyTelemetry(data.telemetry);\n  } catch (e) {\n    console.error(\"Failed loading dataset:\", e);\n  }\n  renderRows();\n}\n\nfunction changeActiveFolder() {\n  const inp = document.getElementById(\"activeFolderInput\").value.trim();\n  if (!inp) return;\n  STATE.folder = inp;\n  STATE.page = 1;\n  STATE.selected.clear();\n  updateDownloadLink();\n  document.getElementById(\"mgrSrc\").value = inp;\n  loadDataset();\n}\n\nasync function refreshTelemetry() {\n  try {\n    const res = await fetch(\"./studio_api/telemetry\");\n    const data = await res.json();\n    applyTelemetry(data);\n  } catch (e) {}\n}\n\nfunction applyTelemetry(t) {\n  if (!t) return;\n  STATE.telemetry = t;\n  const vDot = document.getElementById(\"telVlmDot\");\n  const vTxt = document.getElementById(\"telVlmText\");\n  const st = t.server_status || \"STANDBY\";\n  vDot.className = \"dot \" + (st === \"ONLINE\" ? \"dot-online\" : (st === \"LOADING\" ? \"dot-loading\" : \"dot-standby\"));\n  const shortModel = t.active_model ? t.active_model.split(\"[\")[0].trim() : \"Standby\";\n  vTxt.textContent = \"VLM: \" + (st === \"ONLINE\" ? shortModel : st);\n\n  const tDot = document.getElementById(\"telTaggerDot\");\n  const tTxt = document.getElementById(\"telTaggerText\");\n  const tst = t.tagger_status || \"STANDBY\";\n  tDot.className = \"dot \" + (tst === \"ONLINE\" ? \"dot-online\" : (tst === \"LOADING\" ? \"dot-loading\" : \"dot-standby\"));\n  tTxt.textContent = \"PixAI: \" + tst;\n\n  document.getElementById(\"telVramBadge\").textContent = \"VRAM: \" + t.vram_used_gb + \" / \" + t.vram_total_gb + \" GB\";\n  document.getElementById(\"telDiskBadge\").textContent = \"Disk Free: \" + t.disk_free_gb + \" GB\";\n\n  updateModelDiskBadge();\n  renderInventoryTable(t.models_inventory || []);\n}\n\nfunction renderInventoryTable(list) {\n  const tbody = document.getElementById(\"modelsInventoryBody\");\n  if (!tbody) return;\n  tbody.innerHTML = \"\";\n  list.forEach(m => {\n    const tr = document.createElement(\"tr\");\n    const diskBadge = m.downloaded\n      ? '<span style=\"color:#10b981;font-weight:600;\">✅ Terunduh</span>'\n      : '<span style=\"color:#64748b;\">☁️ Belum Diunduh</span>';\n    const vramBadge = m.active_in_vram\n      ? '<span style=\"color:#38bdf8;font-weight:600;\">🟢 Online di VRAM</span>'\n      : '<span style=\"color:#64748b;\">⚪ Standby</span>';\n\n    tr.innerHTML =\n      '<td><b>' + m.key + '</b></td>' +\n      '<td>' + (m.engine === \"pixai_tagger\" ? \"PyTorch EVA02\" : \"llama-server GGUF\") + '</td>' +\n      '<td>' + diskBadge + '</td>' +\n      '<td>' + m.size_gb + ' GB</td>' +\n      '<td>' + vramBadge + '</td>' +\n      '<td style=\"display:flex; gap:5px;\">' +\n        '<button type=\"button\" class=\"btn btn-xs\" data-act=\"load\">⚡ Load</button>' +\n        (m.downloaded ? '<button type=\"button\" class=\"btn btn-xs btn-danger\" data-act=\"del\">🗑️ Hapus Disk</button>' : '') +\n      '</td>';\n\n    const btnLoad = tr.querySelector('[data-act=\"load\"]');\n    if (btnLoad) btnLoad.onclick = () => switchSpecificModel(m.key);\n    const btnDel = tr.querySelector('[data-act=\"del\"]');\n    if (btnDel) btnDel.onclick = () => deleteModelCache(m.key);\n    tbody.appendChild(tr);\n  });\n}\n\nasync function loadSelectedModelNow() {\n  const key = document.getElementById(\"modelSelect\").value;\n  await switchSpecificModel(key);\n}\n\nasync function switchSpecificModel(model_key) {\n  const banner = document.getElementById(\"statusBanner\");\n  banner.textContent = \"⏳ Memuat model '\" + model_key.split(\"[\")[0].trim() + \"' ke VRAM...\";\n  try {\n    const res = await fetch(\"./studio_api/switch_model\", {\n      method: \"POST\",\n      headers: {\"Content-Type\": \"application/json\"},\n      body: JSON.stringify({model_key})\n    });\n    const data = await res.json();\n    if (data.telemetry) applyTelemetry(data.telemetry);\n    banner.textContent = data.ok ? (\"✅ \" + data.msg) : (\"⚠️ \" + data.msg);\n  } catch (e) {\n    banner.textContent = \"⚠️ Gagal memuat model.\";\n  }\n}\n\nasync function unloadVramNow(target) {\n  const banner = document.getElementById(\"statusBanner\");\n  banner.textContent = \"⏳ Mengosongkan VRAM GPU...\";\n  const res = await fetch(\"./studio_api/unload_vram\", {\n    method: \"POST\",\n    headers: {\"Content-Type\": \"application/json\"},\n    body: JSON.stringify({target})\n  });\n  const data = await res.json();\n  if (data.telemetry) applyTelemetry(data.telemetry);\n  banner.textContent = \"✅ \" + data.msg;\n}\n\nasync function deleteModelCache(model_key) {\n  if (!confirm(\"Hapus file model '\" + model_key + \"' dari disk Colab?\")) return;\n  const banner = document.getElementById(\"statusBanner\");\n  banner.textContent = \"⏳ Menghapus cache model dari disk...\";\n  const res = await fetch(\"./studio_api/delete_model_cache\", {\n    method: \"POST\",\n    headers: {\"Content-Type\": \"application/json\"},\n    body: JSON.stringify({model_key})\n  });\n  const data = await res.json();\n  if (data.telemetry) applyTelemetry(data.telemetry);\n  banner.textContent = \"✅ \" + data.msg;\n}\n\nfunction updateCounts() {\n  const tot = STATE.items.length;\n  const cap = STATE.items.filter(x => x.caption && x.caption.trim()).length;\n  const uncap = tot - cap;\n  const shortCnt = STATE.items.filter(x => x.caption && x.caption.trim() && x.words > 0 && x.words < 15).length;\n  const lowresCnt = STATE.items.filter(x => x.low_res).length;\n\n  document.getElementById(\"cnt-all\").textContent = tot;\n  document.getElementById(\"cnt-cap\").textContent = cap;\n  document.getElementById(\"cnt-uncap\").textContent = uncap;\n  document.getElementById(\"cnt-short\").textContent = shortCnt;\n  document.getElementById(\"cnt-lowres\").textContent = lowresCnt;\n  document.getElementById(\"selCounter\").textContent = STATE.selected.size + \" selected\";\n}\n\nfunction setFilter(f) {\n  STATE.filter = f;\n  STATE.page = 1;\n  [\"all\", \"cap\", \"uncap\", \"short\", \"lowres\"].forEach(id => {\n    const mapKey = {all: \"all\", cap: \"captioned\", uncap: \"uncaptioned\", short: \"short\", lowres: \"lowres\"}[id];\n    const el = document.getElementById(\"pill-\" + id);\n    if (el) el.classList.toggle(\"active\", f === mapKey);\n  });\n  renderRows();\n}\n\nfunction onSearchInput() {\n  STATE.search = (document.getElementById(\"searchQuery\").value || \"\").trim().toLowerCase();\n  STATE.page = 1;\n  renderRows();\n}\n\nfunction getFilteredItems() {\n  let list = STATE.items;\n  if (STATE.filter === \"captioned\") list = list.filter(x => x.caption && x.caption.trim());\n  else if (STATE.filter === \"uncaptioned\") list = list.filter(x => !x.caption || !x.caption.trim());\n  else if (STATE.filter === \"short\") list = list.filter(x => x.caption && x.caption.trim() && x.words > 0 && x.words < 15);\n  else if (STATE.filter === \"lowres\") list = list.filter(x => x.low_res);\n\n  if (STATE.search) {\n    list = list.filter(x =>\n      x.name.toLowerCase().includes(STATE.search) ||\n      (x.caption || \"\").toLowerCase().includes(STATE.search)\n    );\n  }\n  return list;\n}\n\nfunction onPageSizeChange() {\n  STATE.pageSize = parseInt(document.getElementById(\"pageSizeSelect\").value || \"30\", 10);\n  STATE.page = 1;\n  renderRows();\n}\n\nfunction changePage(delta) {\n  const filtered = getFilteredItems();\n  const totalPages = Math.max(1, Math.ceil(filtered.length / STATE.pageSize));\n  STATE.page = Math.min(totalPages, Math.max(1, STATE.page + delta));\n  renderRows();\n}\n\nfunction renderRows() {\n  updateCounts();\n  const container = document.getElementById(\"rowsContainer\");\n  container.innerHTML = \"\";\n  const filtered = getFilteredItems();\n  const totalItems = filtered.length;\n  const totalPages = Math.max(1, Math.ceil(totalItems / STATE.pageSize));\n  if (STATE.page > totalPages) STATE.page = totalPages;\n\n  const startIdx = (STATE.page - 1) * STATE.pageSize;\n  const endIdx = Math.min(totalItems, startIdx + STATE.pageSize);\n  const pageSlice = filtered.slice(startIdx, endIdx);\n\n  document.getElementById(\"pageRangeInfo\").textContent =\n    totalItems ? (\"Menampilkan \" + (startIdx + 1) + \"–\" + endIdx + \" dari \" + totalItems) : \"0 item\";\n  document.getElementById(\"pageIndicator\").textContent = \"Page \" + STATE.page + \" / \" + totalPages;\n  document.getElementById(\"btnPrevPage\").disabled = STATE.page <= 1;\n  document.getElementById(\"btnNextPage\").disabled = STATE.page >= totalPages;\n\n  if (!pageSlice.length) {\n    container.innerHTML =\n      '<div style=\"text-align:center; padding:55px 20px; color:var(--text-muted); border:1px dashed var(--border); border-radius:10px;\">' +\n      'Belum ada gambar yang cocok di kategori/filter ini. Klik tombol <b>📤 Upload Images / .ZIP</b> di kanan atas untuk mengunggah dataset.' +\n      '</div>';\n    return;\n  }\n\n  pageSlice.forEach((it, idx) => {\n    const isSel = STATE.selected.has(it.img_path);\n    const row = document.createElement(\"div\");\n    row.className = \"label-row\" + (isSel ? \" selected\" : \"\");\n\n    const imgUrl = \"./studio_api/image?path=\" + encodeURIComponent(it.img_path);\n    const resText = (it.width && it.height) ? (it.width + \"×\" + it.height + (it.low_res ? \" ⚠️\" : \"\")) : \"\";\n    const tokStats = countWordsAndTokens(it.caption);\n    it.words = tokStats.words;\n    it.tokens = tokStats.tokens;\n\n    row.innerHTML =\n      '<div class=\"row-check-wrap\">' +\n        '<input type=\"checkbox\" class=\"row-check\" ' + (isSel ? \"checked\" : \"\") + '>' +\n      '</div>' +\n      '<div class=\"row-thumb-wrap\">' +\n        '<img class=\"row-thumb\" src=\"' + imgUrl + '\" loading=\"lazy\" title=\"Klik untuk membuka Split-View Lightbox Editor\">' +\n        (resText ? '<div class=\"res-badge ' + (it.low_res ? 'low-res' : '') + '\">' + resText + '</div>' : '') +\n      '</div>' +\n      '<div class=\"row-body\">' +\n        '<div class=\"row-meta\">' +\n          '<div><b style=\"color:#e2e8f0;\">' + it.name + '</b></div>' +\n          '<div class=\"row-actions\">' +\n            '<span class=\"' + formatTokenBadgeClass(it.tokens) + ' word-badge\">' + it.words + ' words · ~' + it.tokens + ' tok</span>' +\n            '<button type=\"button\" class=\"btn btn-xs\" data-act=\"gen1\" title=\"Generate caption untuk 1 gambar ini saja (Ctrl+Enter)\">⚡ Generate 1 Ini</button>' +\n            '<button type=\"button\" class=\"btn btn-xs\" data-act=\"tag1\" title=\"Tag 1 gambar ini dengan PixAI Tagger v0.9\">🏷️ PixAI Tag</button>' +\n            '<button type=\"button\" class=\"btn btn-xs\" data-act=\"chat1\" title=\"Kirim gambar ini ke Tab Multimodal Chat\">📌 Ke Chat</button>' +\n            '<button type=\"button\" class=\"btn btn-xs btn-danger\" data-act=\"del1\" title=\"Hapus gambar & caption ini\">🗑️</button>' +\n          '</div>' +\n        '</div>' +\n        '<textarea class=\"row-textarea\" placeholder=\"- belum ada caption (klik ⚡ Generate 1 Ini atau ketik manual) -\"></textarea>' +\n      '</div>';\n\n    const ta = row.querySelector(\".row-textarea\");\n    ta.value = it.caption || \"\";\n\n    const chk = row.querySelector(\".row-check\");\n    chk.addEventListener(\"change\", () => {\n      if (chk.checked) STATE.selected.add(it.img_path);\n      else STATE.selected.delete(it.img_path);\n      row.classList.toggle(\"selected\", chk.checked);\n      updateCounts();\n    });\n\n    const thumb = row.querySelector(\".row-thumb\");\n    thumb.addEventListener(\"click\", () => openLightbox(it.img_path));\n\n    const badge = row.querySelector(\".word-badge\");\n    let saveTimer = null;\n    ta.addEventListener(\"input\", () => {\n      it.caption = ta.value;\n      const st = countWordsAndTokens(ta.value);\n      it.words = st.words;\n      it.tokens = st.tokens;\n      badge.className = formatTokenBadgeClass(st.tokens) + \" word-badge\";\n      badge.textContent = st.words + \" words · ~\" + st.tokens + \" tok · saving...\";\n      clearTimeout(saveTimer);\n      saveTimer = setTimeout(async () => {\n        await fetch(\"./studio_api/save_caption\", {\n          method: \"POST\",\n          headers: {\"Content-Type\": \"application/json\"},\n          body: JSON.stringify({img_path: it.img_path, caption: ta.value})\n        });\n        badge.textContent = st.words + \" words · ~\" + st.tokens + \" tok · ✓\";\n        updateCounts();\n      }, 400);\n    });\n\n    // Keyboard shortcuts inside row textarea: Ctrl+Enter = Generate 1 Ini, Alt+Up/Down = Navigate rows\n    ta.addEventListener(\"keydown\", (ev) => {\n      if (ev.ctrlKey && ev.key === \"Enter\") {\n        ev.preventDefault();\n        generateSingleRowItem(it);\n      } else if (ev.altKey && (ev.key === \"ArrowDown\" || ev.key === \"ArrowUp\")) {\n        ev.preventDefault();\n        const allTas = Array.from(container.querySelectorAll(\".row-textarea\"));\n        const targetTa = allTas[idx + (ev.key === \"ArrowDown\" ? 1 : -1)];\n        if (targetTa) targetTa.focus();\n      }\n    });\n\n    row.querySelector('[data-act=\"gen1\"]').onclick = () => generateSingleRowItem(it);\n    row.querySelector('[data-act=\"tag1\"]').onclick = () => tagSingleRowItem(it);\n    row.querySelector('[data-act=\"chat1\"]').onclick = () => sendItemToChatTab(it);\n    row.querySelector('[data-act=\"del1\"]').onclick = () => deleteSingleRowItem(it);\n\n    it._domRow = row;\n    it._domTa = ta;\n    it._domBadge = badge;\n    container.appendChild(row);\n  });\n}\n\nfunction selectPageRows(flag) {\n  const filtered = getFilteredItems();\n  const startIdx = (STATE.page - 1) * STATE.pageSize;\n  const slice = filtered.slice(startIdx, startIdx + STATE.pageSize);\n  slice.forEach(it => {\n    if (flag) STATE.selected.add(it.img_path);\n    else STATE.selected.delete(it.img_path);\n  });\n  renderRows();\n}\n\nfunction selectAllFilteredRows(flag) {\n  if (!flag) {\n    STATE.selected.clear();\n  } else {\n    getFilteredItems().forEach(it => STATE.selected.add(it.img_path));\n  }\n  renderRows();\n}\n\nfunction toggleUploadDrawer() {\n  document.getElementById(\"uploadDrawer\").classList.toggle(\"open\");\n}\n\nfunction toggleDrawer(id) {\n  document.getElementById(id).classList.toggle(\"open\");\n}\n\nasync function uploadFilesNow() {\n  const inp = document.getElementById(\"fileUploadInput\");\n  if (!inp.files.length) return alert(\"Pilih file gambar atau .zip terlebih dahulu!\");\n  const fd = new FormData();\n  fd.append(\"folder\", STATE.folder);\n  for (const f of inp.files) fd.append(\"files\", f);\n\n  const banner = document.getElementById(\"statusBanner\");\n  banner.textContent = \"⏳ Mengunggah, memverifikasi gambar & mengekstrak file...\";\n  const res = await fetch(\"./studio_api/upload\", {method: \"POST\", body: fd});\n  const data = await res.json();\n  STATE.items = data.items || [];\n  inp.value = \"\";\n  document.getElementById(\"uploadDrawer\").classList.remove(\"open\");\n  banner.textContent = \"✅ Ditambahkan \" + data.added + \" file\" + (data.corrupt ? (\" (\" + data.corrupt + \" file korup ditolak)\") : \"!\");\n  renderRows();\n}\n\nfunction getGenerationPayload(img_path) {\n  return {\n    img_path,\n    model_key: document.getElementById(\"modelSelect\").value,\n    sys_prompt: document.getElementById(\"captionPrompt\").value,\n    max_tokens: parseInt(document.getElementById(\"maxTokens\").value || \"220\", 10),\n    temperature: parseFloat(document.getElementById(\"temperature\").value || \"0.2\"),\n    subject_hint: document.getElementById(\"subjectHintInput\").value,\n    hybrid_tagger_enabled: document.getElementById(\"hybridTaggerChk\").checked,\n    hybrid_mode: document.getElementById(\"hybridModeSelect\").value,\n    general_threshold: parseFloat(document.getElementById(\"pixaiGenTh\").value || \"0.30\"),\n    character_threshold: parseFloat(document.getElementById(\"pixaiCharTh\").value || \"0.75\"),\n    replace_underscore: document.getElementById(\"pixaiNoUnderscore\").checked,\n    escape_parens: document.getElementById(\"pixaiEscapeParen\").checked,\n    exclude_tags: document.getElementById(\"pixaiExcludeTags\").value\n  };\n}\n\nfunction applyCaptionUpdateToItem(it, data) {\n  if (!data || !data.caption) return;\n  it.caption = data.caption;\n  it.words = data.words;\n  it.tokens = data.tokens;\n  if (it._domTa) it._domTa.value = data.caption;\n  if (it._domBadge) {\n    it._domBadge.className = formatTokenBadgeClass(data.tokens) + \" word-badge\";\n    it._domBadge.textContent = data.words + \" words · ~\" + data.tokens + \" tok · ✓\";\n  }\n  if (STATE.lightboxImgPath === it.img_path) {\n    document.getElementById(\"lbTextarea\").value = data.caption;\n    if (data.tagger_meta) renderLightboxPixaiMeta(data.tagger_meta);\n  }\n  updateCounts();\n}\n\nasync function generateSingleRowItem(it) {\n  const banner = document.getElementById(\"statusBanner\");\n  banner.textContent = \"⚡ Menganalisis \" + it.name + \"...\";\n  if (it._domBadge) it._domBadge.textContent = \"⏳ generating...\";\n  try {\n    const res = await fetch(\"./studio_api/generate_one\", {\n      method: \"POST\",\n      headers: {\"Content-Type\": \"application/json\"},\n      body: JSON.stringify(getGenerationPayload(it.img_path))\n    });\n    const data = await res.json();\n    if (data.ok && data.caption) {\n      applyCaptionUpdateToItem(it, data);\n      banner.textContent = \"✅ Selesai: \" + it.name;\n    } else {\n      banner.textContent = \"⚠️ Gagal men-generate caption untuk \" + it.name;\n    }\n    refreshTelemetry();\n  } catch (e) {\n    banner.textContent = \"⚠️ Error koneksi saat memproses \" + it.name;\n  }\n}\n\nasync function tagSingleRowItem(it) {\n  const banner = document.getElementById(\"statusBanner\");\n  banner.textContent = \"🏷️ PixAI Tagger v0.9 memindai \" + it.name + \"...\";\n  if (it._domBadge) it._domBadge.textContent = \"⏳ tagging...\";\n  try {\n    const res = await fetch(\"./studio_api/tag_one\", {\n      method: \"POST\",\n      headers: {\"Content-Type\": \"application/json\"},\n      body: JSON.stringify({\n        img_path: it.img_path,\n        save_to_file: true,\n        general_threshold: parseFloat(document.getElementById(\"pixaiGenTh\").value || \"0.30\"),\n        character_threshold: parseFloat(document.getElementById(\"pixaiCharTh\").value || \"0.75\"),\n        replace_underscore: document.getElementById(\"pixaiNoUnderscore\").checked,\n        escape_parens: document.getElementById(\"pixaiEscapeParen\").checked,\n        exclude_tags: document.getElementById(\"pixaiExcludeTags\").value\n      })\n    });\n    const data = await res.json();\n    if (data.ok && data.caption) {\n      applyCaptionUpdateToItem(it, data);\n      banner.textContent = \"✅ PixAI Tag selesai: \" + it.name;\n    }\n    refreshTelemetry();\n  } catch (e) {\n    banner.textContent = \"⚠️ Gagal menjalankan PixAI Tagger.\";\n  }\n}\n\nasync function deleteSingleRowItem(it) {\n  if (!confirm(\"Hapus gambar '\" + it.name + \"' beserta file .txt-nya?\")) return;\n  const res = await fetch(\"./studio_api/delete_item\", {\n    method: \"POST\",\n    headers: {\"Content-Type\": \"application/json\"},\n    body: JSON.stringify({img_path: it.img_path, folder: STATE.folder})\n  });\n  const data = await res.json();\n  STATE.selected.delete(it.img_path);\n  STATE.items = data.items || [];\n  document.getElementById(\"statusBanner\").textContent = \"🗑️ Dihapus: \" + it.name;\n  renderRows();\n}\n\nasync function sendItemToChatTab(it) {\n  const banner = document.getElementById(\"statusBanner\");\n  banner.textContent = \"📌 Mengirim \" + it.name + \" ke Tab Multimodal Chat...\";\n  const res = await fetch(\"./studio_api/image?path=\" + encodeURIComponent(it.img_path) + \"&as_data_uri=1\");\n  const data = await res.json();\n  if (data.ok && data.data_uri) {\n    STATE.chatImageDataUri = data.data_uri;\n    document.getElementById(\"chatImgPreview\").src = data.data_uri;\n    document.getElementById(\"chatImgName\").textContent = it.name;\n    document.getElementById(\"chatImgPreviewBar\").style.display = \"flex\";\n    closeLightbox();\n    switchTab(\"chat\");\n    document.getElementById(\"chatInput\").focus();\n    banner.textContent = \"Ready\";\n  }\n}\n\nasync function generateSelectedCaptions() {\n  const btn = document.getElementById(\"btnGenerateBatch\");\n  if (STATE.isGenerating) {\n    STATE.stopRequested = true;\n    btn.textContent = \"⏹️ Menghentikan...\";\n    return;\n  }\n\n  let targets = STATE.items.filter(x => STATE.selected.has(x.img_path));\n  if (!targets.length) return alert(\"Centang minimal 1 gambar (atau klik Select All / Select Page) terlebih dahulu!\");\n\n  const overwrite = document.getElementById(\"overwriteChk\").checked;\n  if (!overwrite) targets = targets.filter(x => !x.caption || !x.caption.trim());\n  if (!targets.length) return alert(\"Semua gambar yang dipilih sudah memiliki caption!\");\n\n  STATE.isGenerating = true;\n  STATE.stopRequested = false;\n  const banner = document.getElementById(\"statusBanner\");\n\n  for (let i = 0; i < targets.length; i++) {\n    if (STATE.stopRequested) break;\n    const it = targets[i];\n    btn.textContent = \"⏹️ Stop (\" + (i + 1) + \"/\" + targets.length + \")\";\n    banner.textContent = \"⚡ [\" + (i + 1) + \"/\" + targets.length + \"] Menganalisis \" + it.name + \"...\";\n    try {\n      const res = await fetch(\"./studio_api/generate_one\", {\n        method: \"POST\",\n        headers: {\"Content-Type\": \"application/json\"},\n        body: JSON.stringify(getGenerationPayload(it.img_path))\n      });\n      const data = await res.json();\n      if (data.ok && data.caption) applyCaptionUpdateToItem(it, data);\n    } catch (e) {\n      console.error(e);\n    }\n  }\n\n  STATE.isGenerating = false;\n  btn.textContent = \"✨ Generate (Selected)\";\n  banner.textContent = STATE.stopRequested ? \"⏹️ Proses batch dihentikan.\" : \"✅ Selesai memproses \" + targets.length + \" gambar!\";\n  refreshTelemetry();\n}\n\nasync function tagSelectedWithPixAI() {\n  const btn = document.getElementById(\"btnTagBatch\");\n  if (STATE.isGenerating) {\n    STATE.stopRequested = true;\n    return;\n  }\n  let targets = STATE.items.filter(x => STATE.selected.has(x.img_path));\n  if (!targets.length) return alert(\"Centang minimal 1 gambar terlebih dahulu!\");\n\n  const overwrite = document.getElementById(\"overwriteChk\").checked;\n  if (!overwrite) targets = targets.filter(x => !x.caption || !x.caption.trim());\n  if (!targets.length) return alert(\"Semua gambar terpilih sudah punya caption!\");\n\n  STATE.isGenerating = true;\n  STATE.stopRequested = false;\n  const banner = document.getElementById(\"statusBanner\");\n\n  for (let i = 0; i < targets.length; i++) {\n    if (STATE.stopRequested) break;\n    const it = targets[i];\n    btn.textContent = \"⏹️ Stop (\" + (i + 1) + \"/\" + targets.length + \")\";\n    banner.textContent = \"🏷️ [\" + (i + 1) + \"/\" + targets.length + \"] PixAI Tagging \" + it.name + \"...\";\n    try {\n      const res = await fetch(\"./studio_api/tag_one\", {\n        method: \"POST\",\n        headers: {\"Content-Type\": \"application/json\"},\n        body: JSON.stringify({\n          img_path: it.img_path,\n          save_to_file: true,\n          general_threshold: parseFloat(document.getElementById(\"pixaiGenTh\").value || \"0.30\"),\n          character_threshold: parseFloat(document.getElementById(\"pixaiCharTh\").value || \"0.75\"),\n          replace_underscore: document.getElementById(\"pixaiNoUnderscore\").checked,\n          escape_parens: document.getElementById(\"pixaiEscapeParen\").checked,\n          exclude_tags: document.getElementById(\"pixaiExcludeTags\").value\n        })\n      });\n      const data = await res.json();\n      if (data.ok && data.caption) applyCaptionUpdateToItem(it, data);\n    } catch (e) {}\n  }\n\n  STATE.isGenerating = false;\n  btn.textContent = \"🏷️ PixAI Tag (Selected)\";\n  banner.textContent = \"✅ Selesai PixAI Tagging \" + targets.length + \" gambar!\";\n  refreshTelemetry();\n}\n\nasync function runBulkEdit(action_type) {\n  const scope = document.getElementById(\"bulkTargetScope\").value;\n  const selList = scope === \"selected\" ? Array.from(STATE.selected) : null;\n  if (action_type === \"apply\" && scope === \"selected\" && (!selList || !selList.length)) {\n    return alert(\"Centang gambar yang ingin di-edit terlebih dahulu, atau ubah Target File ke 'Semua File (.txt)'!\");\n  }\n  const banner = document.getElementById(\"statusBanner\");\n  banner.textContent = \"⏳ Memproses bulk edit file .txt...\";\n  const res = await fetch(\"./studio_api/bulk_edit\", {\n    method: \"POST\",\n    headers: {\"Content-Type\": \"application/json\"},\n    body: JSON.stringify({\n      action_type,\n      folder: STATE.folder,\n      trigger: document.getElementById(\"trigWord\").value,\n      pos: document.getElementById(\"trigPos\").value,\n      find: document.getElementById(\"findStr\").value,\n      replace: document.getElementById(\"repStr\").value,\n      lowercase: document.getElementById(\"trigLower\").checked,\n      skip_dup: document.getElementById(\"trigSkip\").checked,\n      clean_commas: true,\n      backup: true,\n      selected_img_paths: selList\n    })\n  });\n  const data = await res.json();\n  STATE.items = data.items || [];\n  banner.textContent = \"✅ \" + data.msg;\n  renderRows();\n}\n\n/* ==================== SPLIT-VIEW LIGHTBOX EDITOR ==================== */\nfunction openLightbox(img_path) {\n  const it = STATE.items.find(x => x.img_path === img_path);\n  if (!it) return;\n  STATE.lightboxImgPath = img_path;\n  document.getElementById(\"lightboxImg\").src = \"./studio_api/image?path=\" + encodeURIComponent(img_path);\n  document.getElementById(\"lbFileName\").textContent = it.name;\n  document.getElementById(\"lbMetaInfo\").textContent =\n    \"Resolusi: \" + (it.width || \"?\") + \"×\" + (it.height || \"?\") + \" px · \" + it.words + \" words (~\" + it.tokens + \" tokens)\";\n  const lbTa = document.getElementById(\"lbTextarea\");\n  lbTa.value = it.caption || \"\";\n  document.getElementById(\"lbPixaiInspector\").style.display = \"none\";\n  document.getElementById(\"lightbox\").classList.add(\"open\");\n\n  lbTa.oninput = () => {\n    it.caption = lbTa.value;\n    const st = countWordsAndTokens(lbTa.value);\n    it.words = st.words;\n    it.tokens = st.tokens;\n    document.getElementById(\"lbMetaInfo\").textContent =\n      \"Resolusi: \" + (it.width || \"?\") + \"×\" + (it.height || \"?\") + \" px · \" + st.words + \" words (~\" + st.tokens + \" tokens)\";\n    if (it._domTa) it._domTa.value = lbTa.value;\n    fetch(\"./studio_api/save_caption\", {\n      method: \"POST\",\n      headers: {\"Content-Type\": \"application/json\"},\n      body: JSON.stringify({img_path: it.img_path, caption: lbTa.value})\n    });\n    updateCounts();\n  };\n}\n\nfunction closeLightbox() {\n  STATE.lightboxImgPath = null;\n  document.getElementById(\"lightbox\").classList.remove(\"open\");\n}\n\nfunction stepLightbox(delta) {\n  const list = getFilteredItems();\n  const idx = list.findIndex(x => x.img_path === STATE.lightboxImgPath);\n  if (idx === -1) return;\n  const nextIt = list[idx + delta];\n  if (nextIt) openLightbox(nextIt.img_path);\n}\n\nasync function lightboxGenerateOne() {\n  const it = STATE.items.find(x => x.img_path === STATE.lightboxImgPath);\n  if (it) await generateSingleRowItem(it);\n}\n\nasync function lightboxTagOne() {\n  const it = STATE.items.find(x => x.img_path === STATE.lightboxImgPath);\n  if (it) await tagSingleRowItem(it);\n}\n\nfunction lightboxSendToChat() {\n  const it = STATE.items.find(x => x.img_path === STATE.lightboxImgPath);\n  if (it) sendItemToChatTab(it);\n}\n\nfunction renderLightboxPixaiMeta(meta) {\n  const box = document.getElementById(\"lbPixaiInspector\");\n  if (!box || !meta) return;\n  box.style.display = \"block\";\n  const chars = (meta.character_tags || []).join(\", \") || \"-\";\n  const ips = (meta.ip_tags || []).join(\", \") || \"-\";\n  const gens = (meta.general_tags || []).slice(0, 25).join(\", \") || \"-\";\n  box.innerHTML =\n    \"<div><b style='color:#c4b5fd;'>Character:</b> \" + chars + \"</div>\" +\n    \"<div style='margin-top:3px;'><b style='color:#38bdf8;'>Series / IP:</b> \" + ips + \"</div>\" +\n    \"<div style='margin-top:3px;'><b style='color:#94a3b8;'>Top General:</b> \" + gens + \"</div>\";\n}\n\ndocument.addEventListener(\"keydown\", (e) => {\n  if (e.key === \"Escape\" && STATE.lightboxImgPath) closeLightbox();\n});\n\n/* ==================== TAB 2: MULTIMODAL CHAT ==================== */\nfunction onChatImagePicked(ev) {\n  const file = ev.target.files[0];\n  if (!file) return;\n  const reader = new FileReader();\n  reader.onload = () => {\n    STATE.chatImageDataUri = reader.result;\n    document.getElementById(\"chatImgPreview\").src = reader.result;\n    document.getElementById(\"chatImgName\").textContent = file.name;\n    document.getElementById(\"chatImgPreviewBar\").style.display = \"flex\";\n  };\n  reader.readAsDataURL(file);\n}\n\nfunction clearChatImage() {\n  STATE.chatImageDataUri = null;\n  document.getElementById(\"chatImgInput\").value = \"\";\n  document.getElementById(\"chatImgPreviewBar\").style.display = \"none\";\n}\n\nasync function sendChatMessage() {\n  const inp = document.getElementById(\"chatInput\");\n  const text = inp.value.trim();\n  if (!text && !STATE.chatImageDataUri) return;\n\n  const box = document.getElementById(\"chatMessages\");\n  const sentImg = STATE.chatImageDataUri;\n\n  const uDiv = document.createElement(\"div\");\n  uDiv.className = \"msg msg-user\";\n  if (sentImg) {\n    const thumb = document.createElement(\"img\");\n    thumb.src = sentImg;\n    thumb.style.cssText = \"max-height:140px; border-radius:7px; display:block; margin-bottom:6px;\";\n    uDiv.appendChild(thumb);\n  }\n  const uTxt = document.createElement(\"div\");\n  uTxt.textContent = text || \"Describe this image in detail.\";\n  uDiv.appendChild(uTxt);\n  box.appendChild(uDiv);\n\n  const aiDiv = document.createElement(\"div\");\n  aiDiv.className = \"msg msg-ai\";\n  aiDiv.textContent = \"⏳ Sedang menganalisis & menyusun respons...\";\n  box.appendChild(aiDiv);\n  box.scrollTop = box.scrollHeight;\n\n  inp.value = \"\";\n  clearChatImage();\n\n  const res = await fetch(\"./studio_api/chat\", {\n    method: \"POST\",\n    headers: {\"Content-Type\": \"application/json\"},\n    body: JSON.stringify({\n      model_key: document.getElementById(\"modelSelect\").value,\n      sys_prompt: document.getElementById(\"chatSysPrompt\").value,\n      history: STATE.chatHistory,\n      message: text,\n      image_data_uri: sentImg,\n      temperature: parseFloat(document.getElementById(\"chatTemp\").value || \"0.7\"),\n      max_tokens: parseInt(document.getElementById(\"chatMaxTok\").value || \"768\", 10),\n      enable_thinking: document.getElementById(\"chatThinkingChk\").checked\n    })\n  });\n  const data = await res.json();\n  aiDiv.innerHTML = \"\";\n\n  if (data.thinking) {\n    const det = document.createElement(\"details\");\n    det.style.cssText = \"margin-bottom:8px; font-size:11.5px; color:#94a3b8; background:#0c0f17; padding:6px 10px; border-radius:6px;\";\n    const sum = document.createElement(\"summary\");\n    sum.style.cursor = \"pointer\";\n    sum.textContent = \"🧠 Lihat Proses Penalaran (Thinking Trace)\";\n    const pre = document.createElement(\"div\");\n    pre.style.marginTop = \"6px\";\n    pre.textContent = data.thinking;\n    det.appendChild(sum);\n    det.appendChild(pre);\n    aiDiv.appendChild(det);\n  }\n\n  const bodyEl = document.createElement(\"div\");\n  bodyEl.textContent = data.reply || \"\";\n  aiDiv.appendChild(bodyEl);\n\n  const tb = document.createElement(\"div\");\n  tb.className = \"msg-toolbar\";\n  const copyBtn = document.createElement(\"button\");\n  copyBtn.type = \"button\";\n  copyBtn.className = \"btn btn-xs\";\n  copyBtn.textContent = \"📋 Copy Prompt / Jawaban\";\n  copyBtn.onclick = () => {\n    navigator.clipboard.writeText(data.reply || \"\");\n    copyBtn.textContent = \"✓ Copied!\";\n    setTimeout(() => { copyBtn.textContent = \"📋 Copy Prompt / Jawaban\"; }, 1500);\n  };\n  tb.appendChild(copyBtn);\n  aiDiv.appendChild(tb);\n\n  // Store image_data_uri inside chatHistory so follow-up questions continue to see the image!\n  STATE.chatHistory.push({\n    role: \"user\",\n    content: text || \"Describe this image.\",\n    image_data_uri: sentImg || null\n  });\n  STATE.chatHistory.push({\n    role: \"assistant\",\n    content: data.reply || \"\"\n  });\n  box.scrollTop = box.scrollHeight;\n  refreshTelemetry();\n}\n\nfunction clearChatHistory() {\n  STATE.chatHistory = [];\n  document.getElementById(\"chatMessages\").innerHTML =\n    '<div class=\"msg msg-ai\">Riwayat chat dibersihkan. Silakan unggah gambar baru atau kirim gambar dari Tab 1.</div>';\n}\n\n/* ==================== TAB 3: SMART FILE MANAGER & DRIVE PRESETS ==================== */\nfunction onManagerActionChange() {\n  const act = document.getElementById(\"mgrAction\").value;\n  if (act === \"Export to Google Drive\") {\n    document.getElementById(\"mgrSrc\").value = STATE.folder;\n    document.getElementById(\"mgrDst\").value = \"/content/drive/MyDrive/vlm_dataset_backup\";\n  } else if (act === \"Import from Google Drive\") {\n    document.getElementById(\"mgrSrc\").value = \"/content/drive/MyDrive/vlm_dataset_backup\";\n    document.getElementById(\"mgrDst\").value = STATE.folder;\n  }\n}\n\nfunction quickDrivePreset(mode) {\n  const sel = document.getElementById(\"mgrAction\");\n  if (mode === \"export\") {\n    sel.value = \"Export to Google Drive\";\n  } else {\n    sel.value = \"Import from Google Drive\";\n  }\n  onManagerActionChange();\n  runSmartManager();\n}\n\nasync function runSmartManager() {\n  const logEl = document.getElementById(\"mgrLog\");\n  logEl.textContent = \"⏳ Menjalankan aksi...\";\n  const res = await fetch(\"./studio_api/manager\", {\n    method: \"POST\",\n    headers: {\"Content-Type\": \"application/json\"},\n    body: JSON.stringify({\n      action: document.getElementById(\"mgrAction\").value,\n      source_path: document.getElementById(\"mgrSrc\").value,\n      destination_path: document.getElementById(\"mgrDst\").value,\n      rename_prefix: document.getElementById(\"mgrPrefix\").value,\n      active_folder: STATE.folder\n    })\n  });\n  const data = await res.json();\n  logEl.textContent = data.log;\n  if (data.items) {\n    STATE.items = data.items;\n    renderRows();\n  }\n}\n\ninitSelectors();\nloadDataset();\nsetInterval(refreshTelemetry, 12000);\n</script>\n</body>\n</html>\n"
}
    for rel_path, content in MODULAR_FILES.items():
        target_p = PROJECT_DIR / rel_path
        target_p.parent.mkdir(parents=True, exist_ok=True)
        target_p.write_text(content, encoding="utf-8")
    print(f"[✓] {len(MODULAR_FILES)} modul produksi berhasil dipasang di {PROJECT_DIR}")

# 2. Pasang dependensi dari requirements.txt
print("[+] Memasang paket Python (Gradio, FastAPI, timm, HuggingFace Hub, CUDA 12 Runtime)...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed",
     "-r", str(PROJECT_DIR / "requirements.txt")],
    check=True
)

# 3. Jalankan bootstrap_vlm.py untuk mengunduh & memverifikasi llama-server & llama-mtmd-cli
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from bootstrap_vlm import bootstrap_llama_cpp
bin_info = bootstrap_llama_cpp()
print("\n✅ SETUP SELESAI! Silakan jalankan SEL 2 di bawah untuk membuka Studio.")


In [ ]:
# @title 🚀 SEL 2: Launch Vision AI Labeling, PixAI Booru Tagger & Multimodal Chat Studio
# @markdown Pilih model awal dan folder dataset, lalu jalankan sel ini untuk membuka antarmuka web penuh.

import os, sys, re, time, socket, signal, subprocess
from pathlib import Path
from IPython.display import display, HTML

PROJECT_DIR = Path("/content/vlm-caption-studio")
DEFAULT_MODEL = "Qwen3.5-9B [Uncensored Aggressive - HauhauCS]" # @param ["Qwen3.5-9B [Uncensored Aggressive - HauhauCS]", "Qwen3-VL-8B [Uncensored Abliterated - wangkanai]", "JoyCaption Beta One [Llama-3.1-8B Uncensored - fancyfeast]", "Gemma-3-12B-IT [Uncensored Abliterated - mlabonne]", "Qwen3-VL-8B [Official - Qwen]", "Qwen3.5-9B [Official - Unsloth]", "Gemma-3-12B-IT [Official - Unsloth]", "PixAI Tagger v0.9 [Fast Anime & Character/IP Booru Tagger]"]
DATASET_FOLDER = "/content/dataset" # @param {type:"string"}
PREWARM_MODEL_ON_START = True # @param {type:"boolean"}
HF_TOKEN_OPTIONAL = "" # @param {type:"string"}

os.makedirs(DATASET_FOLDER, exist_ok=True)
APP_LOG = Path("/content/vlm_studio_gradio.log")
APP_PID = Path("/content/vlm_studio_gradio.pid")

if APP_PID.exists():
    try:
        os.kill(int(APP_PID.read_text().strip()), signal.SIGTERM)
        time.sleep(1)
    except Exception:
        pass

ui_port = 7865
for p in range(7865, 7880):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        if s.connect_ex(("127.0.0.1", p)) != 0:
            ui_port = p
            break

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from bootstrap_vlm import bootstrap_llama_cpp
bin_info = bootstrap_llama_cpp()

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["VLM_LLAMA_SERVER"] = bin_info["llama_server"]
env["VLM_LLAMA_MTMD_CLI"] = bin_info["llama_mtmd_cli"]
env["VLM_LLAMA_CLI"] = bin_info["llama_cli"]
env["VLM_DEFAULT_MODEL"] = DEFAULT_MODEL
env["VLM_DATASET_DIR"] = DATASET_FOLDER
env["VLM_UI_PORT"] = str(ui_port)
env["VLM_PREWARM_MODEL"] = "1" if PREWARM_MODEL_ON_START else "0"
if HF_TOKEN_OPTIONAL.strip():
    env["HF_TOKEN"] = HF_TOKEN_OPTIONAL.strip()

print(f"[+] Meluncurkan Vision AI Studio di port {ui_port} dengan model '{DEFAULT_MODEL}'...")
with open(APP_LOG, "w", encoding="utf-8", errors="replace") as lf:
    proc = subprocess.Popen(
        [sys.executable, "-u", str(PROJECT_DIR / "run_vlm_studio.py")],
        env=env, stdout=lf, stderr=subprocess.STDOUT, text=True
    )
APP_PID.write_text(str(proc.pid))

public_url = None
for _ in range(360):
    if proc.poll() is not None:
        break
    time.sleep(1)
    if APP_LOG.exists():
        txt = APP_LOG.read_text(errors="ignore")
        m = re.search(r"VLM_SHARE_URL=(https://[A-Za-z0-9.-]+\.gradio\.live)", txt)
        if m:
            public_url = m.group(1)
            break

if public_url:
    print("\n✅ VISION AI LABELING, PIXAI TAGGER & CHAT STUDIO SIAP DIGUNAKAN!")
    display(HTML(
        f'<div style="padding:14px 18px;background:#11151f;border:1px solid #38bdf8;border-radius:10px;margin-top:8px;">'
        f'<a href="{public_url}" target="_blank" style="font-size:18px;font-weight:700;color:#38bdf8;text-decoration:none;">'
        f'🚀 Buka Vision AI Studio ({public_url})</a>'
        f'<p style="color:#94a3b8;font-size:12.5px;margin-top:4px;">Folder Aktif: <code>{DATASET_FOLDER}</code> · Model: <code>{DEFAULT_MODEL}</code></p></div>'
    ))
else:
    tail = "\n".join(APP_LOG.read_text(errors="ignore").splitlines()[-100:]) if APP_LOG.exists() else "(no log)"
    raise RuntimeError("Gagal memulai Gradio Studio. Log terakhir:\n" + tail)
